# 21 — 31B LoRA (the student) on the SFT mix → adapter check → submission

Offline (Internet off), GPU L4 × 4. Built by `scripts/planB/notebook_builder/make_nb21.py` from notebook 19 (setup, resume,
deployed agent, submission writer) with notebook 20's scripts (its 31B training fixes).

**Resumable, one 11 h 30 min session at a time** (attach the previous version's output to continue; each session ends with a
submittable `submission.zip`):
1. **This session's slice** of the attached `graphcomp-sft-mix` (`train_fit.jsonl`: test-verified trajectories ≤ 32k tokens,
   duplicates removed): trajectories not trained in an earlier session, **new problems first** (the shortest trajectory of
   each never-trained problem), **shortest first**, sized to the tokens the measured training speed fits before the deadline
   (× 1.5; the trainer stops on time anyway), **mixed over length quartiles** rather than strictly shortest-first (SWE-Gym's
   easy-task bias); from session 2, **~8% replay** of earlier sessions' trajectories spread through the slice (SWE-Lego's
   cumulative curriculum; the replay result of arXiv 2609.33559). **Competition trajectories go last, every session** (our own
   verified 31B runs + external ones on the four competition repos), **rotated**: the least-visited half each session, at most
   3 visits each, so every session ends on competition data while no text is drilled (arXiv 2609.33559).
2. **31B LoRA** (`train_lora.py`, QLoRA, 4-bit model split over the 4 GPUs): **continues the previous session's adapter**;
   1 pass in that order (shuffled only within blocks of 32 windows); LR decays to 30% of the peak, not 0, so the competition
   tail still moves the model; stops at a step boundary before the training deadline (session start + 11.5 h − 0.85 h), and
   checkpoints the adapter every 30 min (atomically), so even a killed run leaves an adapter. The training window starts at
   32,768 tokens and steps down on out-of-memory; a resume that fails to load falls back to a fresh LoRA once.
3. **Adapter check**: vLLM serves the adapter as the scorer does; a no-op adapter is reported (the output stays submittable).
4. **Submission** = the LoRA with the deployment budget (8 min / 30 tool calls per task, user Oct 19; if all ~120 hidden tasks hit
   the cap that is ~17.6 h, so it relies on most tasks finishing early: past 31B runs averaged ~6.3 min at concurrency 4), plus the
   state for the next session (trained trajectory ids, problems, measured tokens/s).

**Inputs:** as notebook 19 (competition data, wheelhouse, `gemma-4-31b-it-qat-w4a16-ct` + `gemma-4-31b-it-qat-q4_0-unquantized`,
notebook 01's output, `graphcomp-planb`, `graphcomp-train-wheels`) **plus `graphcomp-sft-mix`**; for a resumed run, this
notebook's previous version output.


## Training choices (literature check, Oct 15)
- **Assistant-only loss with the whole conversation as context; split by task.** The tool-calling SFT baseline of SFT or RL for
  Tool-Calling Agents? (arXiv 2609.17848) puts the loss on assistant turns only, with LoRA rank 32, alpha 64, lr 1e-4, 2 epochs,
  all linear layers (read from the alphaXiv summary). Turns are packed into windows so one forward pass trains several turns.
- **Step-level error masking** (SWE-Lego, arXiv 2601.01426, alphaXiv summary): turns whose tool call failed keep their place in the context but get
  no loss; test and reproduction failures are not masked (here: run_command results are never masked).
- **Same prompt and serialization in training and deployment** (2609.17848 used the same system prompts and serialization
  in training and evaluation): `--system-prompt` and `--deploy-budget`.
- **Quantity over filtering at small scale** (Systematic Evaluation of Trajectory Data Curation for LoRA Fine-Tuning of Code
  Agents, arXiv 2607.17205, alphaXiv summary: going from 500 to 1,000 trajectories lowered held-out loss about 18x more than
  quality filtering did; resolved-only gating was not better than random at 500): up to 3 resolved trajectories per task instead of 2, every seed kept.
- **Vanilla LoRA, tune the learning rate** (Learning Rate Matters: Vanilla LoRA May Suffice, arXiv 2602.04998, alphaXiv summary:
  LoRA variants reach vanilla LoRA's peak once each one's learning rate is tuned): plain LoRA, lr 1e-4, cosine with warmup.
- **Teacher stronger than student** (SWE-Gym, arXiv 2412.21139, p. 5: fine-tuning on the policy's own trajectories
  lowered the resolve rate from 15.3% to 8.7%); the 31B LoRA is
  close to self-distillation, which is why BASE31 is in the comparison and wins ties.

## Data choices (Oct 15)
- **Verified only, teachers stronger than the student**: every external trajectory passed its task's tests and comes from
  Qwen3-Coder-480B (the user's rule: only models stronger than Gemma 4 31B).
- **Filters from the same teacher's distillation papers** (applied in `convert_trajs.py`): no git history beyond the base
  commit (SWE-Lego arXiv 2601.01426, SWE-HERO arXiv 2604.01496), no test-file edits, < 3 editor errors, a real source change.
- **Train = deploy**: trajectories longer than the served 32k context could not happen at deployment, so they are not used;
  installs (`pip install`, downloads) are dropped because deployment is offline.
- **Diversity first**: one trajectory per task before any task gets a second (different routes for one task are kept,
  duplicates are removed upstream).


In [ ]:
# first cell: proves the kernel is running, and prints "CELL END" after every cell from here on (also after %%writefile cells)
import datetime as _dt, sys as _sys
print(f"NOTEBOOK STARTED {_dt.datetime.now(_dt.timezone.utc):%Y-%m-%d %H:%M:%S} UTC | Python {_sys.version.split()[0]}", flush=True)
_cell_n = [0]
def _cell_end_marker(result=None):
    _cell_n[0] += 1
    failed = bool(result is not None and (result.error_in_exec or result.error_before_exec))
    print(f"CELL END {_cell_n[0]}{' (FAILED)' if failed else ''} | {_dt.datetime.now(_dt.timezone.utc):%H:%M:%S} UTC", flush=True)
_ip = get_ipython()
for _f in [f for f in _ip.events.callbacks["post_run_cell"] if getattr(f, "__name__", "") == "_cell_end_marker"]:
    _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _cell_end_marker)


In [ ]:
# STEP 0/7 - setup: config and progress logging
# ============================== CONFIG (edit here) ==============================
from pathlib import Path

VARIANT_TAG = "NB21"             # names the run directory: runs/NB21 (parts end in _s<seed>)
ARM = "B2"                    # tools: "B2" = all nine; "B3" = the three graph tools removed
SEEDS = []                    # no teacher runs in this notebook
SEED = 1
RECHECK_VALIDITY = False      # notebook 14: 24/24 agree across builds; the rebuilt wheel only split the pytest timeout
SESSION_BUDGET_H = 11.75       # Kaggle stops GPU sessions at 12 h; output is saved only if the notebook ends
TRAIN31_MAX_H, TRAIN_STUDENT_MAX_H, EVAL_ARM_H = 10.0, 0, 0.5   # EVAL_ARM_H = the adapter check (31B load + 1 request)
TRAIN_MAX_LENS = [24576, 20480, 16384]   # training window (tokens); the next one after an OOM restart (session 2: 25.9k did not fit)
MAX_TARGET_FOR = {24576: 2048, 20480: 1536, 16384: 1024}   # target tokens per window (session 1: 8000 -> the 262k-vocab logits ran GPU 3 out of memory)
LAST_GPU_LAYERS = 6           # decoder layers on GPU 3, which also holds the tied embeddings/output layer + logits (18 on each other GPU)
MAX_TRAIN_TRAJ = 100000         # cap per session (the token budget below is the real limit)
POST_TRAIN_H = 0.85             # after training: adapter check (31B load + 1 request), submission, zip, margin
LOAD_H = 0.6                  # 31B load inside the trainer before the first step (notebook 20: ~35 min)
TOK_PER_S_GUESS = 150         # training tokens / s until a session has measured it (train_meta.json)
DATA_OVERSHOOT = 1.5          # select this many times the tokens expected to fit (the trainer stops at the deadline)
FIRST_OVERSHOOT = 4.0         # the same before any session has measured the speed (an unused slice costs only rendering)
REPLAY_FRAC = 0.08            # tokens of earlier sessions' outside trajectories per token of new ones (session 2+)
COMP_MAX_VISITS = 3           # a competition trajectory is trained in at most this many sessions (rotation below)
COMP_PER_SESSION = 0.5        # share of the competition trajectories in each session's tail (least-visited first)
LENGTH_MIX = True             # new data round-robin over length quartiles, not strictly shortest-first
SMOKE = False                  # True: the single shortest trajectory, 1 epoch, 1 window per step, no adapter check
VAL_FRAC = 0.02               # of tasks, for the validation loss (2% of thousands is enough and keeps eval short)
# ---- the deployed agent (also used for the teacher runs and the holdout arms)
N_TASKS_ASSUMED, SERVE_START_MIN, MARGIN_MIN, SETUP_MIN = 120, 15, 20, 0.5
DEPLOY_TIME_MIN = 8.0          # user, Oct 19 (the cap-for-every-task formula gives 5.0)
DEPLOY_BUDGET = dict(max_tool_calls=30, max_time_minutes=DEPLOY_TIME_MIN, max_turns=80, timeout_seconds=240)
_worst_h = (N_TASKS_ASSUMED * (DEPLOY_BUDGET['max_time_minutes'] + SETUP_MIN) + SERVE_START_MIN) / 60
print(f"deployment budget {DEPLOY_BUDGET}: if all {N_TASKS_ASSUMED} tasks hit the cap the run takes {_worst_h:.1f} h of 12 "
      f"({'FITS' if _worst_h * 60 + MARGIN_MIN <= 720 else 'DOES NOT FIT: relies on most tasks finishing before the cap'})")
CONC_SLOWDOWN = 1.6           # estimate: a task at concurrency 4 runs about this much slower than alone
# ---- student: 12B (stronger than E4B on Google's model card: LiveCodeBench v6 72.0% vs 52.0%, Tau2 tool use 69.0% vs 42.2%).
# Trained on all 4 GPUs (4-bit 12B ~9 GB per L4, smaller windows than E4B). vLLM 0.19.1 in the competition wheelhouse has no
# native Gemma4Unified, so serving 12B is unresolved (notebook 11); the holdout run is off (EVAL_STUDENTS) and the adapter is
# saved for later. Fallback to E4B: replace the entry with
#   'STUDENT_E4B': dict(key='e4b', slug='gemma-4-e4b-it-qat-w4a16-ct', train_gpus=[0, 1, 2, 3], master_port=29510, eval_gpu=0, port=8001, max_target=2500, max_len=18432)
STUDENTS = {}                 # no students in this notebook
EVAL_STUDENTS = False

TASK_SCOPE = "list"           # "all" = dev slice, then non-slice tasks, then holdout; "dev" = dev slice only; "list" = TASK_LIST
TASK_LIST = ['fastapi_13207', 'fastapi_13537', 'fastapi_13713', 'fastapi_13786', 'fastapi_13920', 'fastapi_14246', 'fastapi_14262', 'fastapi_14301', 'fastapi_14306', 'fastapi_14349', 'fastapi_14371', 'fastapi_14419', 'fastapi_14448', 'fastapi_14455', 'fastapi_14458', 'fastapi_14459', 'fastapi_14479', 'fastapi_14487', 'fastapi_14492', 'fastapi_14583', 'fastapi_14605', 'fastapi_14609', 'fastapi_14791', 'fastapi_14964', 'fastapi_14978', 'fastapi_15023', 'fastapi_15030', 'fastapi_15280', 'fastapi_15588', 'fastapi_15661', 'fastapi_5077', 'fastapi_5624', 'fastapi_9555', 'rich_2943', 'rich_3006', 'rich_3043', 'rich_3063', 'rich_3067', 'rich_3105', 'rich_3454', 'rich_3470', 'rich_3471', 'rich_3506', 'rich_3676', 'rich_3718', 'rich_3772', 'rich_3782', 'rich_3894', 'rich_3930', 'rich_3934', 'rich_3942', 'rich_3953', 'rich_4006', 'rich_4070', 'rich_4077', 'rich_4079']   # 56: sub_valid in nb01, in neither official slice
MAX_OUTPUT_TOKENS = 8192        # sample value 16384 (Oct 12 decision); only the output-reserve ablation changes it
THINKING_BUDGET = 4096          # sample value 4096; only the thinking ablation changes it
PROMPT_ADDENDUM = "\n\n## Budget: always finish inside the time limit\nThe task prompt states your time allowance. When it runs out the session stops and the edits in /workspace are graded as\nthey are; an unedited tree always scores zero.\n- `get_status` is free (it does not use a tool call): call it every 4-5 tool calls and read `time_seconds_remaining`.\n- Use at most the first 40% of the time to locate the code. By then make your best edit at the most likely source location,\n  even if you are unsure; you can correct it later.\n- With less than 30% of the time left: no more exploration. Run at most one targeted test, fix only what it shows, then call\n  `submit_patch`.\n- With less than 60 seconds left: call `submit_patch` immediately.\n- Keep outputs short (`| head -40`, `sed -n 'START,ENDp' FILE`): long outputs fill the context window and end the session.\n\n## Graph tools\n`get_code_neighbors` and `get_code_subgraph` resolve fully qualified names only (for example\n`fastapi.routing.APIRoute.get_route_handler`); bare names often resolve to the wrong symbol. Use them only with a qualified name\nyou have already seen in the code.\n"
# B4 only: {instance_id: {brief, problem_sha256}} from scripts/planB/build_b4_briefs.py, prepended to the problem statement
BRIEFS = {}
CONCURRENCY = 4               # fixed for every arm (the 30-min per-task budget makes it part of the condition)
CHUNK = 8                     # tasks per run_arm call; the time guard is checked between chunks
RETRY_INFRA_ERRORS = True     # rerun tasks whose record shows a serving / connection failure, not a model outcome
BUDGET = dict(max_tool_calls=30, max_time_minutes=8, max_turns=80, timeout_seconds=240)   # in-notebook runs (concurrency 4)
COMPACTION_INTERVAL = 15      # as the sample notebook
SERVE = dict(util=0.80, max_model_len=32768)   # session A: served on the first rung; any other value is a different condition
MODEL_VARIANT = "31b"            # "31b" in every Plan B arm; "12b" / "e4b" / "e2b" only in the size sweep (Oct 13 announcement)
MODEL_SPECS = {   # tp: all three smaller variants fit one L4 at 4 bits and are served with tp 1 (12B at tp 4 failed, Oct 14)
    "31b": dict(slug="gemma-4-31b-it-qat-w4a16-ct", tp=4, loras=True),
    "12b": dict(slug="gemma-4-12b-it-qat-w4a16-ct", tp=1, loras=False),   # Oct 14: tp 4 failed (vLLM Transformers fallback, hidden_size 3840 vs 256)
    "e4b": dict(slug="gemma-4-e4b-it-qat-w4a16-ct", tp=1, loras=False),
    "e2b": dict(slug="gemma-4-e2b-it-qat-w4a16-ct", tp=1, loras=False),
}
MODEL_SLUG = MODEL_SPECS[MODEL_VARIANT]["slug"]
MODEL_DIR = f"/kaggle/input/models/google/gemma-4/other/{MODEL_SLUG}/2"   # any attached version of this slug is found below
MOUNT_SAMPLE_LORAS = MODEL_SPECS[MODEL_VARIANT]["loras"]   # the sample LoRAs are 31B adapters; the agent declares none either way
SERVER_STARTUP_TIMEOUT = 1800
RUN_VALIDITY = False          # True only in notebook 01; every other notebook reads validity from 01's attached output
SHADOWED_REPOS = ("requests", "httpx")   # tests import the host copy (B1 import check, Oct 12): not scorable here
TEST_WHEELS = {'dirty_equals-0.11-py3-none-any.whl': ('...'), 'executing-2.3.0-py3-none-any.whl': ('...'), 'inline_snapshot-0.36.0-py3-none-any.whl': ('...')}   # {filename: (sha256, base64)}: inline-snapshot 0.36.0, executing 2.3.0, dirty-equals 0.11
SLICE_SEED = 20261012        # dev / holdout slices (Plan B section 2); fixed
# OFFICIAL slices (fixed Oct 12, scripts/planB/official_slices; the VM runner uses the same files)
OFFICIAL_DEV = ['fastapi_11194', 'fastapi_11355', 'fastapi_12942', 'fastapi_14258', 'fastapi_14266', 'fastapi_14356', 'fastapi_14430', 'fastapi_14485', 'fastapi_14786', 'fastapi_14794', 'fastapi_14953', 'fastapi_14962', 'fastapi_15800', 'fastapi_9425', 'rich_3052', 'rich_3468', 'rich_3480', 'rich_3518', 'rich_3521', 'rich_3535', 'rich_3905', 'rich_3935', 'rich_3944', 'rich_4075']
OFFICIAL_HOLDOUT = ['fastapi_14099', 'fastapi_14297', 'fastapi_14303', 'fastapi_14360', 'fastapi_14361', 'fastapi_14372', 'fastapi_14463', 'fastapi_14482', 'fastapi_14512', 'fastapi_14616', 'fastapi_14873', 'fastapi_14986', 'fastapi_15589', 'fastapi_9753', 'rich_2725', 'rich_3061', 'rich_3180', 'rich_3278', 'rich_3472', 'rich_3675', 'rich_3777', 'rich_3882', 'rich_3938', 'rich_4076']
ORDER_SEED = 20261012        # task order only if validity is unavailable (shuffled once, fixed)
# Session A host packages (B0_sessionA_results.md section 1); installing the test deps must not change any of them.
SESSION_A_HOST = {"pydantic": "2.12.3", "typing-inspection": "0.4.2", "fastapi": "0.136.1", "starlette": "0.52.1",
                  "requests": "2.32.4", "rich": "13.9.4", "swegemma": "0.2.11", "adk-submission": "0.2.13",
                  "adk-eval-core": "0.1.0", "google-adk": "1.36.1"}

OUT = Path("/kaggle/working/graphcomp_arm")
GC = Path("/kaggle/working/gc")
RUN_NAME = VARIANT_TAG
RUN_ROOT = OUT / "runs" / RUN_NAME
PARTS = RUN_ROOT / "parts"
OUT.mkdir(parents=True, exist_ok=True)
GC.mkdir(parents=True, exist_ok=True)
for d in ("provenance", "submissions", "scorecard", "validity"):
    (OUT / d).mkdir(exist_ok=True)
PARTS.mkdir(parents=True, exist_ok=True)
print("output folder:", OUT, "| run:", RUN_NAME)
# ============================== progress logging (every cell from here on) ==============================
# Each code cell starts with a "# STEP n/N - ..." line. A hook prints a timestamped START line when a cell
# begins and DONE / FAILED with its duration when it ends, and appends both to OUT/progress.log, so a
# committed run's log shows exactly where it is or where it stopped.
import time, threading, re as _re
from datetime import datetime, timezone
_T0 = time.time()
PROGRESS_LOG = OUT / "progress.log"
def _el(s):
    s = int(s); return f"{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}"
def plog(msg):
    line = f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')} UTC | +{_el(time.time() - _T0)}] {msg}"
    print(line, flush=True)
    try:
        with open(PROGRESS_LOG, "a") as f:
            f.write(line + "\n")
    except Exception:
        pass
def _cell_label(src):
    for l in src.splitlines()[:6]:
        m = _re.match(r"\s*#\s*(STEP\b.*)", l)
        if m:
            return m.group(1).strip()
    first = next((l.strip() for l in src.splitlines() if l.strip()), "")
    return "cell: " + first[:100]
_cell = {}
def _pre_cell(info=None):
    _cell["label"], _cell["t"] = _cell_label(getattr(info, "raw_cell", "") or ""), time.time()
    plog(f">>> START  {_cell['label']}")
def _post_cell(result=None):
    failed = bool(result is not None and (result.error_in_exec or result.error_before_exec))
    plog(f"{'!!! FAILED' if failed else '<<< DONE '}  {_cell.get('label', '')}  ({_el(time.time() - _cell.get('t', time.time()))})")
try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre_cell), ("post_run_cell", _post_cell)):
        for _old in [f for f in _ip.events.callbacks[_ev] if getattr(f, "__name__", "") == _fn.__name__]:
            _ip.events.unregister(_ev, _old)
        _ip.events.register(_ev, _fn)
except NameError:
    pass  # not in IPython

import json as _json, subprocess as _sp, urllib.request as _ur

def gpu_status():
    try:
        out = _sp.run("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits",
                      shell=True, capture_output=True, text=True, timeout=10).stdout.split("\n")
        rows = [l.split(",") for l in out if l.strip()]
        return "GPU util " + "/".join(r[0].strip() for r in rows) + "% mem " + "/".join(r[1].strip() for r in rows) + " MiB" if rows else "GPU n/a"
    except Exception:
        return "GPU n/a"

def vllm_status():
    try:
        txt = _ur.urlopen("http://127.0.0.1:8000/metrics", timeout=3).read().decode(errors="replace")
        val = lambda k: sum(float(l.rsplit(" ", 1)[1]) for l in txt.splitlines() if l.startswith(k) and not l.startswith("#"))
        return f"vLLM running {val('vllm:num_requests_running'):.0f} waiting {val('vllm:num_requests_waiting'):.0f}"
    except Exception:
        return "vLLM not answering"

class Progress:
    # While a long command runs: print one line per finished task as soon as its record lands in task_results.jsonl
    # (outcome, tool calls, graph calls, how it ended, seconds), and a heartbeat every `every` seconds with counts,
    # elapsed time, GPU utilisation and the vLLM queue.
    GRAPH = ("get_code_neighbors", "search_similar_code", "get_code_subgraph")
    def __init__(self, path, total, label, every=120, poll=15):
        self.path, self.total, self.label, self.every, self.poll = Path(path), total, label, every, poll
        self._stop = threading.Event()
        self._lines = 0
        self.n = self.resolved = 0
    def _new(self):
        try:
            lines = [l for l in open(self.path) if l.strip()]
        except FileNotFoundError:
            return []
        new, self._lines = lines[self._lines:], len(lines)
        out = []
        for l in new:
            try:
                out.append(_json.loads(l))
            except Exception:
                pass
        return out
    def _task_line(self, r):
        self.n += 1
        self.resolved += bool(r.get("resolved"))
        iid = r.get("instance_id")
        extra = ""
        tp = self.path.parent / "traces" / f"trace_{iid}.json"
        if tp.exists():
            try:
                d = _json.load(open(tp))
                calls = [tc for s in d.get("steps", []) for tc in (s.get("tool_calls") or [])]
                extra = f" tool_calls={len(calls)} graph_calls={sum(tc.get('function_name') in self.GRAPH for tc in calls)}"
            except Exception:
                pass
        err = " ".join(str(v) for k, v in r.items() if isinstance(v, str) and re.search(r"error|reason", k, re.I))[:100]
        dur = r.get("duration_seconds")
        plog(f"    [{self.label} {self.n}/{self.total}] {iid} resolved={r.get('resolved')}{extra}"
             f"{' ' + str(round(dur)) + 's' if dur else ''}{' | ' + err if err else ''}")
    def _heartbeat(self):
        plog(f"    ... {self.label}: {self.n}/{self.total} done, {self.resolved} resolved, "
             f"{_el(time.time() - self._t0)} in this step | {gpu_status()} | {vllm_status()}")
    def _run(self):
        last = time.time()
        while not self._stop.wait(self.poll):
            for r in self._new():
                self._task_line(r)
            if time.time() - last >= self.every:
                last = time.time()
                self._heartbeat()
    def __enter__(self):
        self._t0 = time.time()
        plog(f"    {self.label}: started ({self.total} tasks) | {gpu_status()}")
        self._t = threading.Thread(target=self._run, daemon=True)
        self._t.start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
        self._t.join(timeout=5)
        for r in self._new():
            self._task_line(r)
        plog(f"    {self.label}: finished, {self.n}/{self.total} task results, {self.resolved} resolved, "
             f"{_el(time.time() - self._t0)}")

_cell.update(label="STEP 0/7 - setup: config and progress logging", t=_T0)
plog(">>> START  STEP 0/7 - setup: config and progress logging (this cell)")
# fail fast, before the 4-minute install, if the session is not the 4 x L4 / Python 3.12 image
import sys as _sys
_gpus = [l for l in _sp.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.splitlines() if l.strip()]
plog(f"session check: Python {_sys.version.split()[0]}, {len(_gpus)} GPUs: {[g[:40] for g in _gpus]}")
if _sys.version_info[:2] != (3, 12):
    raise SystemExit(f"Python {_sys.version.split()[0]}: the wheelhouse needs 3.12 (pinned Getting Started image).")
if len(_gpus) != 4:
    raise SystemExit(f"{len(_gpus)} GPUs visible; these runs need GPU L4 x 4 (tp=4). Change the accelerator and run again.")

In [ ]:
# STEP 0/7 - setup: helper functions
# ============================== helpers ==============================
import json, os, re, shlex, subprocess, sys, time, glob, shutil, zipfile, csv, gzip, base64, platform
from datetime import datetime, timezone

def now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def sh(cmd, log=None, check=False, env=None):
    """Run a shell command, stream its output here, and tee it to `log`."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, **(env or {})})
    f = open(log, "a") if log else None
    if f:
        f.write(f"\n### {now()} $ {cmd}\n")
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line)
    p.wait()
    if f:
        f.close()
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

def walk(root, maxdepth=6, skip=("snapshots", "wheels", "embeddings", "graphs", "__pycache__", ".git")):
    root = Path(root)
    base = len(root.parts)
    for dp, dns, fns in os.walk(root):
        depth = len(Path(dp).parts) - base
        dns[:] = [d for d in dns if d not in skip] if depth < maxdepth else []
        yield Path(dp), dns, fns

def read_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

In [ ]:
# STEP 0/7 - setup: locate inputs
# ============================== locate inputs ==============================
INPUT = Path("/kaggle/input")
found = {"tasks": [], "sample_submission": [], "wheelhouse": [], "model_configs": [], "validity": [], "old_b1": [],
         "oracle_pack": [], "prior_parts": []}
for dp, dns, fns in walk(INPUT, maxdepth=9, skip=("snapshots", "wheels", "embeddings", "graphs", "__pycache__", ".git", "traces", "logs", "test_outputs", "patches")):
    if "tasks.jsonl" in fns:
        found["tasks"].append(dp / "tasks.jsonl")
    if "agent.yaml" in fns and "sample" in str(dp).lower():
        found["sample_submission"].append(dp)
    if "wheelhouse" in dp.name.lower():
        found["wheelhouse"].append(dp)
    if "config.json" in fns and MODEL_SLUG in str(dp).lower():
        found["model_configs"].append(dp)
    if "validity_env.json" in fns:
        found["validity"].append(dp)          # written by step 1 of this notebook
    if "b1_env.json" in fns:
        found["old_b1"].append(dp)            # the CPU B1 notebook's output: different image, not used
    if "oracle_pack.json.gz" in fns:
        found["oracle_pack"].append(dp / "oracle_pack.json.gz")
    elif "oracle_pack.json" in fns:          # Kaggle may unpack the .gz on dataset upload; re-packed below
        found["oracle_pack"].append(dp / "oracle_pack.json")
    if dp.parent.name == "parts" and dp.parent.parent.name == RUN_NAME and "task_results.jsonl" in fns:
        found["prior_parts"].append(dp)
for k, v in found.items():
    print(f"{k:18s}", len(v), [str(x) for x in v][:4])

if found["old_b1"]:
    plog("NOTE: an old B1 output is attached (b1_env.json). It is ignored: it was measured on the CPU image without the "
         "test dependencies. Validity comes from step 1 of this notebook.")
TASKS_JSONL = found["tasks"][0]
DATA_ROOT = TASKS_JSONL.parent
SNAPSHOTS = DATA_ROOT / "snapshots"
SAMPLE_SUB = found["sample_submission"][0] if found["sample_submission"] else None
if (MODEL_DIR is None or not Path(MODEL_DIR).exists()) and found["model_configs"]:
    MODEL_DIR = str(sorted(found["model_configs"], key=lambda p: len(str(p)))[0])
VALIDITY_ROOT = found["validity"][0] if found["validity"] else None
B1_MODE = VALIDITY_ROOT is not None and (VALIDITY_ROOT / "slices" / "dev_slice.txt").exists()
if B1_MODE:
    DEV = [l.strip() for l in open(VALIDITY_ROOT / "slices/dev_slice.txt") if l.strip()]
    HOLDOUT = [l.strip() for l in open(VALIDITY_ROOT / "slices/holdout_slice.txt") if l.strip()]
    B1_PROFILE = VALIDITY_ROOT / "slices/profile.csv"
    B1_ENV = json.load(open(VALIDITY_ROOT / "validity_env.json"))
else:
    DEV, HOLDOUT, B1_PROFILE, B1_ENV = [], [], None, None
assert TASKS_JSONL.exists() and SNAPSHOTS.exists(), "competition data not attached?"
assert SAMPLE_SUB is not None, "sample submission not found"
assert MODEL_DIR and Path(MODEL_DIR).exists(), f"model not attached (google/gemma-4, {MODEL_SLUG})"
assert MODEL_SLUG in str(MODEL_DIR).lower(), f"MODEL_DIR {MODEL_DIR} is not {MODEL_SLUG}: attach only that model"
assert (not B1_MODE) or (len(DEV) == 24 and len(HOLDOUT) == 24 and not set(DEV) & set(HOLDOUT)), "slice files look wrong"
tasks_by_id = {json.loads(l)["instance_id"]: json.loads(l) for l in open(TASKS_JSONL) if l.strip()}
ALL_IDS = sorted(tasks_by_id)
print(f"\n{len(ALL_IDS)} tasks | validity attached: {B1_MODE} ({VALIDITY_ROOT}) | dev {len(DEV)} | holdout {len(HOLDOUT)}")
print("MODEL_DIR =", MODEL_DIR, "| SAMPLE_SUB =", SAMPLE_SUB)

In [ ]:
# STEP 0 - resume: copy an attached earlier version of this notebook's output
_prior = [dp for dp, dns, fns in walk(INPUT, maxdepth=8) if dp.name == "graphcomp_arm" and (dp / "NB21_STATE.json").exists()]
for _p in _prior:
    shutil.copytree(_p, OUT, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.zip"))
    plog(f"resume: copied earlier output {_p}")
STATE_FILE = OUT / "NB21_STATE.json"
STATE = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
def save_state(**kw):
    STATE.update(kw)
    STATE_FILE.write_text(json.dumps(STATE, indent=1))
save_state(sessions=STATE.get("sessions", 0) + 1)
def left_h():
    return SESSION_BUDGET_H - (time.time() - _T0) / 3600
plog(f"state: {STATE}")


## Install the harness and serving stack
The competition sample notebook's install cell, verbatim (as in sessions A and B1).

In [ ]:
# STEP 0/7 - setup: install the harness wheelhouse (takes a few minutes)
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# --- pre-flight checks (added; not in the sample cell) ---
# 1) The wheelhouse is built for CPython 3.12. A newer Kaggle image (Python 3.13) cannot install it.
print("Python", sys.version.split()[0], "at", sys.executable)
if sys.version_info[:2] != (3, 12):
    raise SystemExit(
        f"This kernel runs Python {sys.version_info[0]}.{sys.version_info[1]}; the competition wheelhouse is cp312.\n"
        "Fix: open the competition's 'Getting Started' notebook and use Copy & Edit, then import these cells into it,\n"
        "or set Settings -> Environment -> 'Pin to original environment' to the sample notebook's environment.")
# 2) Find the wheelhouse wherever it is mounted (dataset not attached = no wheels).
if not any(WHEELHOUSE_DIR.glob('*.whl')):
    cands = sorted({p.parent for p in Path('/kaggle/input').rglob('vllm-*.whl')}) if Path('/kaggle/input').exists() else []
    print("wheelhouse not at the sample path; folders holding a vllm wheel:", [str(c) for c in cands])
    if cands:
        WHEELHOUSE_DIR = cands[0]
    else:
        raise SystemExit("No wheelhouse found. Add Input -> Datasets -> 'metric/gemma-4-developer-agent-wheelhouse', then rerun.\n"
                         "Mounted inputs: " + str([p.name for p in Path('/kaggle/input').iterdir()] if Path('/kaggle/input').exists() else []))
print("WHEELHOUSE_DIR =", WHEELHOUSE_DIR, "|", len(list(WHEELHOUSE_DIR.glob('*.whl'))), "wheels")

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')
(OUT / "provenance/wheelhouse_installed.txt").write_text("\n".join(Path(w).name for w in wheels) + "\n")

In [ ]:
# STEP 0/7 - setup: install the test dependencies (embedded wheels: inline-snapshot 0.36.0, executing 2.3.0, dirty-equals 0.11)
# ============================== test dependencies (embedded wheels; offline) ==============================
# The competition's wheels/ folder does not contain them (session of Oct 12, 13:58 UTC), so the three pure-Python wheels are
# embedded here, pinned to the versions Plan A's per-task fastapi environments used (planA_results/vm-a2/.../env_freeze.txt:
# inline-snapshot 0.36.0 and executing 2.3.0 in 67/67; dirty-equals 0.11 in 24/67, 0.9.0 in 43/67, one version for all).
# Installed with --no-index --no-deps: nothing is downloaded and nothing already installed may change.
import base64 as _b64, hashlib as _hl
def pip_freeze():
    out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    d = {}
    for l in out.splitlines():
        l = l.strip()
        if not l or l.startswith("#"):
            continue
        name, _, v = l.partition("==") if "==" in l else l.partition(" @ ")
        d[re.sub(r"[-_.]+", "-", name).lower()] = v.strip()
    return d
TW = GC / "test_wheels"
TW.mkdir(parents=True, exist_ok=True)
for fn, (sha, b64) in TEST_WHEELS.items():
    data = _b64.b64decode(b64)
    got = _hl.sha256(data).hexdigest()
    if got != sha:
        raise SystemExit(f"embedded wheel {fn}: sha256 {got} != {sha}")
    (TW / fn).write_bytes(data)
plog(f"test wheels written and verified: {sorted(TEST_WHEELS)}")
before = pip_freeze()
rc = sh(f"{sys.executable} -m pip install --no-index --no-deps " + " ".join(str(TW / f) for f in sorted(TEST_WHEELS)),
        log=OUT / "provenance/test_deps_install.log")
after = pip_freeze()
TEST_DEPS_CHANGES = {k: (before.get(k), after.get(k)) for k in sorted(set(before) | set(after)) if before.get(k) != after.get(k)}
plog(f"packages added or changed by the test-dependency install: {TEST_DEPS_CHANGES}")
(OUT / "provenance/test_deps_changes.json").write_text(json.dumps(TEST_DEPS_CHANGES, indent=1))
chk = subprocess.run([sys.executable, "-c", "import inline_snapshot, dirty_equals, executing; from inline_snapshot import snapshot; "
                      "print('test deps import ok:', inline_snapshot.__version__ if hasattr(inline_snapshot, '__version__') else '', "
                      "dirty_equals.__version__, executing.__version__)"], capture_output=True, text=True)
print(chk.stdout or chk.stderr[-1500:])
touched = {k: v for k, v in TEST_DEPS_CHANGES.items() if v[0] is not None}
if rc or chk.returncode or touched:
    raise SystemExit(f"Test-dependency install failed or changed packages that were already installed: {touched or rc or chk.stderr[-500:]}. "
                     "Nothing has run yet; see provenance/test_deps_install.log.")

In [ ]:
# STEP 0/7 - setup: check the serving stack imports
r = subprocess.run([sys.executable, "-c", "import swegemma, adk_submission, adk_eval_core, google.adk, vllm; print('stack ok')"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])

In [ ]:
# STEP 0/7 - setup: provenance and full package list (must match the validity session)
# ============================== provenance: GPUs, versions, full package list ==============================
import importlib.metadata as md
def ver(p):
    try:
        return md.version(p)
    except Exception:
        return None
FREEZE = pip_freeze()
env_info = {
    "time": now(), "python": platform.python_version(), "nproc": os.cpu_count(),
    "nvidia_smi": subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                                 shell=True, capture_output=True, text=True).stdout.strip(),
    "versions": {p: ver(p) for p in list(SESSION_A_HOST) + ["vllm", "torch", "pygments", "ipython", "pytest", "inline-snapshot", "dirty-equals"]},
    "test_deps_changes": TEST_DEPS_CHANGES,
    "config": {"ARM": ARM, "SEED": SEED, "TASK_SCOPE": TASK_SCOPE, "TASK_LIST": TASK_LIST, "N_BRIEFS": len(BRIEFS), "CONCURRENCY": CONCURRENCY, "CHUNK": CHUNK,
               "BUDGET": BUDGET, "COMPACTION_INTERVAL": COMPACTION_INTERVAL, "SERVE": SERVE, "SESSION_BUDGET_H": SESSION_BUDGET_H},
}
N_GPUS = len([l for l in env_info["nvidia_smi"].splitlines() if l.strip()])
TP = min(MODEL_SPECS[MODEL_VARIANT]["tp"], N_GPUS)
env_info["tensor_parallel"] = TP
env_info["model"] = {"variant": MODEL_VARIANT, "slug": MODEL_SLUG, "dir": str(MODEL_DIR), "sample_loras_mounted": MOUNT_SAMPLE_LORAS}
(OUT / f"provenance/env_{RUN_NAME}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M')}.json").write_text(json.dumps(env_info, indent=1))
(OUT / "provenance/pip_freeze.json").write_text(json.dumps(FREEZE, indent=1, sort_keys=True))
print(json.dumps(env_info, indent=1))
if N_GPUS != 4:
    raise SystemExit(f"{N_GPUS} GPUs visible; Plan B Kaggle runs use 4 x L4 (tp=4). Set the accelerator to GPU L4 x4.")
diff_a = {p: (v, env_info["versions"].get(p)) for p, v in SESSION_A_HOST.items() if env_info["versions"].get(p) != v}
print("\nsandbox packages vs session A:", "identical" if not diff_a else f"DIFFERS {diff_a}")
if diff_a:
    raise SystemExit("The sandboxes import these packages; they must equal session A's. Use the same Kaggle image.")
import hashlib as _hlib
STACK_ID = _hlib.sha256("".join(str(FREEZE.get(k)) for k in ("vllm", "swegemma", "adk-submission", "adk-eval-core")).encode()).hexdigest()[:10]
STACK_CHANGED, _rebuilt = False, {}
if B1_MODE:
    ref = B1_ENV["freeze"]
    diff_v = {k: (ref.get(k), FREEZE.get(k)) for k in sorted(set(ref) | set(FREEZE)) if ref.get(k) != FREEZE.get(k)}
    print("full package list vs the validity session:", "identical" if not diff_v else f"DIFFERS {diff_v}")
    # Oct 13: the competition wheelhouse was re-uploaded with the SAME versions but rebuilt files (vllm, swegemma,
    # adk-submission, adk-eval-core), and old dataset versions cannot be selected. Only that kind of difference is allowed:
    # same package, same version, both installed from the wheelhouse. It makes this run a different stack build: it pairs only
    # with a B2 run on the same build (notebook 14), and validity is re-checked on the slice tasks before anything is served.
    def _whl(u):
        m = re.search(r"/([^/]+?)-([^-/]+)-[^/]*\.whl", u or "")
        return (m.group(1).lower().replace("_", "-"), m.group(2)) if m else None
    _rebuilt = {k: v for k, v in diff_v.items() if v[0] and v[1] and "/tmp/wheelhouse/" in v[0] and "/tmp/wheelhouse/" in v[1]
                and _whl(v[0]) is not None and _whl(v[0]) == _whl(v[1])}
    _other = {k: v for k, v in diff_v.items() if k not in _rebuilt}
    if _other:
        raise SystemExit("This session's packages differ from the session that measured validity, so validity would not "
                         f"carry over: {sorted(_other)}. Use the same Kaggle image (same Getting Started copy, same inputs).")
    STACK_CHANGED = bool(_rebuilt)
    if STACK_CHANGED:
        plog(f"!!! stack build {STACK_ID} differs from the validity session (same versions, rebuilt files): {sorted(_rebuilt)}. "
             "Pair this run only with B2 on the same build (notebook 14); validity is re-checked on the slice tasks first.")
(OUT / "provenance/stack.json").write_text(json.dumps({"stack_id": STACK_ID, "changed_vs_validity_session": STACK_CHANGED,
    "rebuilt": {k: list(v) for k, v in _rebuilt.items()}, "wheels": {k: FREEZE.get(k) for k in ("vllm", "swegemma", "adk-submission", "adk-eval-core")}}, indent=1))
plog(f"stack build {STACK_ID}" + (" (rebuilt wheelhouse)" if STACK_CHANGED else " (same as the validity session)" if B1_MODE else ""))

## Scripts
`run_arm.py` (session A, unchanged) and `scorecard.py` 0.3.2 (v7 = TQS port, length check, `end_reason`), the same
files as `scripts/planB/`. The oracle pack is NOT embedded (Kaggle limits notebook source to 1 MB): the `graphcomp-planb`
dataset **must be attached**. The next cell stops the notebook (before anything runs on the GPU) if it is missing or lacks a task.

In [ ]:
%%writefile /kaggle/working/gc/run_arm.py
"""Run one Plan B arm (or the gold / empty-patch passes) through swegemma 0.2.11's own Evaluator.

    python run_arm.py --mode agent --submission SUBDIR --results OUT --task-ids a b c ...
    python run_arm.py --mode empty --results OUT --task-ids ...      # Phase 2 with an empty patch
    python run_arm.py --mode gold  --results OUT --task-ids ...      # Phase 2 with the reference patch

Why not `swegemma eval`: the CLI's EvalConfig leaves events_compaction_config and
context_cache_config at None, while the competition's scripts/inference.py sets them; and the CLI
registry skips adk_submission's litellm reasoning patch. Plan B arms must match the scoring path, so
this runner sets both (values from the harness README section 7.2) and builds the model registry
with VllmServer.create_model_registry, like inference.py. Pass --cli-parity to get the CLI's
behaviour instead. Every choice is written to OUT/run_config.json.
"""
from __future__ import annotations

import argparse
import asyncio
import json
import os
import platform
import sys
import time
from pathlib import Path

MODEL_ALIAS = "gemma-4-31b-it-qat-w4a16-ct"


def build_models(a):
    if a.mode != "agent":
        from swegemma.cli import _build_cli_model_registry
        return _build_cli_model_registry(submission_dir=a.submission)
    if a.cli_parity:
        os.environ.setdefault("LOCAL_INFERENCE_URL", f"http://127.0.0.1:{a.port}/v1")
        from swegemma.cli import _build_cli_model_registry
        return _build_cli_model_registry(submission_dir=a.submission)
    # As the sample notebook: litellm.drop_params, openai/ prefix, aliases = declared model + target name.
    import litellm
    litellm.drop_params = True
    if a.remote_api_base:
        return remote_registry(a)
    from adk_submission import VllmConfig, VllmServer
    try:  # newer swegemma (Kaggle wheelhouse); 0.2.11 has no swegemma.models.discovery
        from swegemma.models.discovery import validate_single_declared_model
        declared = validate_single_declared_model(Path(a.submission))
    except ImportError:
        from adk_submission.discovery import discover_declared_models
        declared = sorted(discover_declared_models(a.submission))[0]
    # Oct 14: register the submission's own adapters (submission/adapters/<name>, agent.yaml "adapter: <name>") so the agent
    # can route to a LoRA served by vLLM under that name. No adapters folder -> None -> exactly the previous behaviour.
    try:
        from adk_submission import discover_adapters
        from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
        manifest = discover_adapters(str(a.submission), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS) \
            if (Path(a.submission) / "adapters").is_dir() else None
    except Exception:
        manifest = None
    srv = VllmServer(VllmConfig(model=a.model_path, port=a.port), adapter_manifest=manifest)
    return srv.create_model_registry(aliases=list(dict.fromkeys([declared, a.model_alias])),
                                     model_prefix="openai/", api_key="EMPTY")


def remote_registry(a):
    """Teacher runs (Oct 15): the deployed agent unchanged, but its model is an OpenAI-compatible endpoint (an open-weight
    teacher at an inference provider) instead of the local vLLM server. The key is read from the environment variable named by
    --remote-key-env, never from the command line."""
    from adk_submission import ModelRegistry
    from adk_submission.resolvers.generation import install_litellm_reasoning_patch
    from google.adk.models.lite_llm import LiteLlm
    try:
        from swegemma.models.discovery import validate_single_declared_model
        declared = validate_single_declared_model(Path(a.submission))
    except ImportError:
        from adk_submission.discovery import discover_declared_models
        declared = sorted(discover_declared_models(a.submission))[0]
    install_litellm_reasoning_patch()
    key = os.environ.get(a.remote_key_env)
    if not key:
        sys.exit(f"--remote-api-base given but ${a.remote_key_env} is empty")
    reg = ModelRegistry()
    for alias in dict.fromkeys([declared, a.model_alias]):
        reg.register(alias, LiteLlm(model="openai/" + a.remote_model, api_base=a.remote_api_base, api_key=key))
    return reg


def src_layout_shim():
    """requests and httpx keep their package in src/. Docker scoring exposes it through the editable install, but the
    subprocess sandbox venv has no pip (ensurepip unavailable), so their tests imported the host's installed copy
    (import_check.py, Oct 12). Put /workspace/src first on PYTHONPATH for every sandbox command: the agent's tools and
    the harness's own verify_task. No effect on repos without src/ (fastapi, rich)."""
    from swegemma.sandbox import SubprocessManager
    orig = SubprocessManager.exec
    # SWE_EXTRA_PYTHONPATH (host env, optional): extra dirs after the workspace, e.g. unpacked test-dependency wheels that
    # the competition wheelhouse lacks (pytest-httpbin for requests). Baked in here: the sandbox env does not pass it on.
    extra = os.environ.get("SWE_EXTRA_PYTHONPATH", "")
    prefix = f'export PYTHONPATH="/workspace/src${{PYTHONPATH:+:$PYTHONPATH}}{":" + extra if extra else ""}"; '

    def exec_(self, sandbox_id, command, *, timeout=None):
        return orig(self, sandbox_id, prefix + command, timeout=timeout)
    SubprocessManager.exec = exec_


def log_compactions(path):
    """Append every ADK context compaction (the summary that replaces older events) to `path`. The ATIF trace does not store
    it, so without this train_lora.py cannot rebuild the prompts after a compaction (nb19: 445 of ~1,080 turns lost).
    ADK 1.36: compaction._summarize_events_with_trace is looked up at call time, so patching the module attribute works;
    contents._process_compaction_events then puts the summary in as a model turn at end_timestamp, replacing every event
    with start_timestamp <= ts <= end_timestamp."""
    try:
        from google.adk.apps import compaction as C
    except ImportError:
        print("[compactions] google.adk.apps.compaction not found: summaries not logged", flush=True)
        return
    orig = C._summarize_events_with_trace

    async def wrapped(*, session, config, events_to_compact, trigger):
        ev = await orig(session=session, config=config, events_to_compact=events_to_compact, trigger=trigger)
        try:
            cc = ev and ev.actions.compaction
            if cc and cc.compacted_content is not None:
                ids = [fc.id for e in events_to_compact for fc in (e.get_function_calls() or []) if fc.id]
                rec = {"session_id": session.id, "trigger": trigger, "created": time.time(),
                       "start": cc.start_timestamp, "end": cc.end_timestamp,
                       "event_ts": [e.timestamp for e in events_to_compact], "call_ids": ids,
                       "summary": "".join(p.text or "" for p in (cc.compacted_content.parts or []))}
                with open(path, "a") as f:
                    f.write(json.dumps(rec) + "\n")
        except Exception as e:      # logging must never break a run
            print(f"[compactions] not logged: {type(e).__name__}: {e}", flush=True)
        return ev

    C._summarize_events_with_trace = wrapped


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["agent", "empty", "gold"], required=True)
    ap.add_argument("--tasks", required=True, help="tasks.jsonl")
    ap.add_argument("--snapshots", required=True)
    ap.add_argument("--submission", required=True)
    ap.add_argument("--results", required=True)
    ap.add_argument("--sandbox", default="subprocess", choices=["subprocess", "docker"])
    ap.add_argument("--task-ids", nargs="+")
    ap.add_argument("--max-tool-calls", type=int, default=50)
    ap.add_argument("--max-time-minutes", type=float, default=30.0)
    ap.add_argument("--max-turns", type=int, default=100)
    ap.add_argument("--timeout-seconds", type=int, default=300)
    ap.add_argument("--concurrency", type=int, default=1)
    ap.add_argument("--shard-index", type=int)
    ap.add_argument("--num-shards", type=int)
    ap.add_argument("--model-path", default="")
    ap.add_argument("--model-alias", default=MODEL_ALIAS)
    ap.add_argument("--port", type=int, default=8000)
    ap.add_argument("--remote-api-base", help="OpenAI-compatible endpoint of a teacher model (instead of the local vLLM server)")
    ap.add_argument("--remote-model", help="the teacher's model id at that endpoint")
    ap.add_argument("--remote-key-env", default="TEACHER_API_KEY", help="environment variable holding the endpoint's API key")
    ap.add_argument("--cli-parity", action="store_true", help="no compaction/cache, CLI registry")
    ap.add_argument("--compaction-interval", type=int, default=15)
    ap.add_argument("--briefs", help="B4 push: JSON {instance_id: {brief, problem_sha256}} prepended to the problem statement")
    a = ap.parse_args()

    from swegemma.evaluate import EvalConfig, Evaluator
    if a.sandbox == "subprocess":
        src_layout_shim()

    cache_cfg = comp_cfg = None
    if not a.cli_parity:
        from google.adk.agents.context_cache_config import ContextCacheConfig
        try:
            from google.adk.apps._configs import EventsCompactionConfig
        except ImportError:
            from google.adk.apps.app import EventsCompactionConfig
        # Values from the competition sample notebook (compaction_interval=15; the README's section 7.2 says 5).
        comp_cfg = EventsCompactionConfig(compaction_interval=a.compaction_interval, overlap_size=2,
                                          token_threshold=14336, event_retention_size=5)
        cache_cfg = ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10)

    results = Path(a.results)
    results.mkdir(parents=True, exist_ok=True)
    if comp_cfg is not None:
        log_compactions(results / "compactions.jsonl")
    # As the sample notebook: explicit limits, generation constraints, adapters, graph/embedding/wheel dirs.
    from swegemma.config import build_submission_limits
    limits, gen_constraints = build_submission_limits()
    data_dir = Path(a.tasks).parent
    try:
        from adk_submission import discover_adapters
        from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
        adapters = discover_adapters(str(a.submission), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    except Exception:
        adapters = None
    extra = dict(limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
                 graph_dir=str(data_dir / "graphs"), embeddings_dir=str(data_dir / "embeddings"))
    if (data_dir / "wheels").is_dir():
        extra["wheels_dir"] = data_dir / "wheels"
    cfg = EvalConfig(
        tasks_path=Path(a.tasks), snapshots_dir=Path(a.snapshots), results_dir=results,
        submission_dir=Path(a.submission), models=build_models(a), sandbox=a.sandbox,
        timeout_seconds=a.timeout_seconds, max_tool_calls=a.max_tool_calls, max_turns=a.max_turns,
        max_time_minutes=a.max_time_minutes, task_ids=a.task_ids, skip_agent_patch=(a.mode == "empty"),
        concurrency=a.concurrency, shard_index=a.shard_index, num_shards=a.num_shards,
        display_mode="quiet", context_cache_config=cache_cfg, events_compaction_config=comp_cfg, **extra,
    )

    if a.mode == "gold":
        from adk_eval_core.tracing import SessionTrace

        class GoldEvaluator(Evaluator):
            async def _run_agent_sandbox(self, task, snapshot_path, task_index, total_tasks, **kw):
                task = self._hydrate_task_from_secret(task)
                return task.patch, None, SessionTrace()

        ev = GoldEvaluator(cfg)
    elif a.briefs:
        # B4 (push): the brief is prepended to the problem statement, so it is part of the first user message
        # (swegemma builds that message from task.problem_statement). Tasks without a brief run exactly as B2.
        import dataclasses
        import hashlib
        briefs = json.loads(Path(a.briefs).read_text())

        class PushEvaluator(Evaluator):
            def _hydrate_task_from_secret(self, task):
                task = super()._hydrate_task_from_secret(task)
                b = briefs.get(task.instance_id)
                if not b or task.problem_statement.startswith(b["brief"]):
                    return task
                if hashlib.sha256(task.problem_statement.encode()).hexdigest() != b["problem_sha256"]:
                    raise SystemExit(f"{task.instance_id}: problem statement differs from the one the brief was built from")
                return dataclasses.replace(task, problem_statement=b["brief"] + "\n## Issue\n" + task.problem_statement)

        ev = PushEvaluator(cfg)
    else:
        ev = Evaluator(cfg)

    import importlib.metadata as md
    vers = {}
    for p in ("swegemma", "adk-submission", "adk-eval-core", "google-adk", "litellm", "vllm", "torch"):
        try:
            vers[p] = md.version(p)
        except Exception:
            vers[p] = None
    run_cfg = {"argv": sys.argv, "mode": a.mode, "sandbox": a.sandbox, "task_ids": a.task_ids,
               "budget": {"max_tool_calls": a.max_tool_calls, "max_time_minutes": a.max_time_minutes,
                          "max_turns": a.max_turns, "timeout_seconds": a.timeout_seconds},
               "events_compaction": None if comp_cfg is None else comp_cfg.model_dump(exclude={"summarizer"}),
               "context_cache": None if cache_cfg is None else cache_cfg.model_dump(),
               "registry": "cli" if (a.cli_parity or a.mode != "agent") else "VllmServer.create_model_registry(openai/, drop_params) as sample notebook",
               "model_path": a.model_path, "wheels_dir": str(extra.get("wheels_dir")), "model_alias": a.model_alias, "python": platform.python_version(),
               "versions": vers, "src_layout_shim": a.sandbox == "subprocess", "extra_pythonpath": os.environ.get("SWE_EXTRA_PYTHONPATH", ""), "graph_dir": cfg.graph_dir, "embeddings_dir": cfg.embeddings_dir,
               "started_unix": time.time(),
               "briefs": a.briefs and {"file": a.briefs, "n_tasks_with_brief": len(json.loads(Path(a.briefs).read_text()))}}
    (results / "run_config.json").write_text(json.dumps(run_cfg, indent=1, default=str))
    t0 = time.time()
    res = asyncio.run(ev.run())
    run_cfg["wall_seconds"] = round(time.time() - t0, 1)
    (results / "run_config.json").write_text(json.dumps(run_cfg, indent=1, default=str))
    print(f"[{a.mode}] {res.resolved}/{res.total} resolved in {run_cfg['wall_seconds']} s -> {results}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/gc/scorecard.py
"""Plan B scorecard: score graph-tool use in swegemma 0.2.11 traces against the Plan A oracle.

Usage
-----
    python scorecard.py --results RUN_DIR [RUN_DIR ...] --oracle-pack oracle_pack.json.gz --out OUT
        [--arm B2] [--no-tools-results RUN_DIR ...] [--brief-dir DIR] [--graphs-dir DIR] [--dump-format]

RUN_DIR is a swegemma --results-dir (traces/trace_<id>.json, task_results.jsonl, logs/<id>.log).
Writes OUT/scorecard_calls.jsonl (one row per graph call), OUT/scorecard_tasks.csv (one row per
run x task with every component and every version's primary), OUT/scorecard_summary.json (pooled
rates with Wilson intervals, per repo and per cell), OUT/parse_report.json.

Trace format (read from adk_eval_core 0.1.0 tracing/atif_converter.py, ATIF-v1.7):
  steps[] with source in {user, agent, system}. An agent step holds tool_calls[] (function_name,
  arguments, extra.usage) and at most ONE observation (the first tool response). Further responses
  of the same step become separate system steps carrying observation (+extra.tool_name). Observations
  carry no tool_call_id, so calls and results are matched FIFO by tool name, then by order.
  Compaction = system step with extra.event_type == "compaction". Harness nudges = system step with
  extra.event_type == "continuation_nudge" (message = the nudge text, which names the cause).
  Raw tool returns: observation.content is the json.dumps of the full parsed tool response.

Component definitions and versions: HANDOFF_planB.md section 4.2/4.3; the concrete choices made
where the handoff leaves room are in VERSIONS_DOC below and are written to OUT/scorecard_versions.md.
"""
from __future__ import annotations

import argparse
import collections
import csv
import gzip
import json
import math
import os
import re
import sys
from pathlib import Path

SCORECARD_VERSION = "0.3.2"

GRAPH_TOOLS = ("get_code_neighbors", "search_similar_code", "get_code_subgraph")
HARNESS_TOOLS = {"run_command", "read_file", "edit_file", "write_file", "get_status", "submit_patch",
                 *GRAPH_TOOLS}
# Gemma 4 tool-call markup that leaked into assistant text = the gemma4 parser did not extract a call.
RAW_TOOLCALL_RE = re.compile(r"<\|?tool_call\|?>|<tool_call>|\bcall:[A-Za-z_]\w*\s*\{")
# Wording in logs/<id>.log or the vLLM log for a parser failure (adjust after B0 session A).
PARSER_ERROR_RE = re.compile(r"(error in extracting tool call|failed to parse tool call|tool[_ ]call.*(pars|decod)\w* (error|fail))", re.I)
MALFORMED_ERR_TYPES = {"ValidationError", "TypeError", "ValueError", "InvalidArguments", "ArgumentError",
                       "MissingArgument", "ToolNotFound", "UnknownTool"}
SHELL_READ_RE = re.compile(r"\b(cat|sed|head|tail|nl|less|more|awk|grep|rg|bat)\b")
PATH_TOKEN_RE = re.compile(r"[\w./-]+\.py\b")

PARAMS_DEFAULT = {"R": "R2", "window": 2, "label": "v1", "hub_pct": 95, "hub_share": 0.5}

VERSIONS_DOC = """# Scorecard versions (scorecard.py {ver})

Pre-registered list from HANDOFF_planB.md 4.3. Add versions; never replace one silently.
Components are task-level rates (NaN when their denominator is 0). `yield_R*` = share of graph calls
whose returned nodes intersect R; `novel_yield_R*` counts only returned R-nodes not returned by an
earlier graph call; `follow_w*` = share of graph calls with returned nodes followed within W tool calls
by a read_file on a returned node's file; `necessity_v*` = 1 if (made a graph call) == (need label);
`waste_p*_s*` = share of graph calls that returned only already-returned nodes, or whose resolved
query node is a hub (provided-graph in-degree above the repo-graph percentile) with test share of
returned nodes at or above s; `halluc_rate` = (unknown tool names + malformed arguments + unresolvable
graph arguments) / tool calls, parser failures excluded; `async_hazard` = share of graph calls naming
a gold/R2 node that is absent from the provided graph.

| Version | Primary | Date | Parameters | Why |
| --- | --- | --- | --- | --- |
| v0 | n_graph_calls | 2026-10-12 | none | strawman |
| v1 | mean of available(yield_R2, follow_w2, necessity_v1, 1 - waste_p95_s50) | 2026-10-12 | R2, W2, label v1, hub 95/50 | Oct 11 proposal design. With no graph call the primary is necessity_v1 alone |
| v2 | v1 with necessity_v2 | 2026-10-12 | label v2 (needs --no-tools-results) | counterfactual need |
| v3 | v1 with yield_R1 | 2026-10-12 | R = gold only | strict yield |
| v4 | novel_yield_R2 on need_v1 = 1 tasks (0 if no call), NaN on need_v1 = 0; 0 if halluc_rate > 0.25 | 2026-10-12 | R2, label v1, veto 0.25 | selection-rule form; novel yield because repeated good calls inflated raw yield in the 0.2.0 fixture |
| v5 | follow_w2 | 2026-10-12 | W2 | delivery only |
| v6 | logistic(components) fitted with --fit-v6 on B2 seed 1 of the 81 non-slice tasks | 2026-10-12 | coefficients file | learned composite, expected to overfit |
| v7 | tqs: port of TQS (Li et al. 2026, arXiv 2610.13152), graph-agnostic trajectory-structure score | 2026-10-12 | TQS's published weights and constants; deviations below | external baseline: a graph-specific version must beat a validated generic trajectory score, not only v0 |

v7 port (TQS section 3.3, weights OEC .30, IAC .25, PSN/TTE/PED/DPI .10, SCP .05, renormalised over defined
components; NaN when fewer than 3 tool calls or fewer than 4 of 7 components defined). Deviations from the
paper, fixed here: steps are tool calls (thinking-only steps are not actions); action alphabet
(read_file, write_file, execute_bash, search, submit, other, think) with read_file -> read_file,
edit_file/write_file -> write_file, run_command -> search when the command starts with grep/rg/find/ls/ag/ack
else execute_bash, the three graph tools -> search, submit_patch -> submit, get_status -> other; F_t = file
paths in the call's arguments (read/edit/write filepath, *.py tokens in run_command); error step = result
status "error", or a non-zero exit code; IAC lexicon (ours; the paper does not publish its lexicon) in
TQS_LEXICON, applied to the last 400 characters of the thought text preceding the call, the category whose
keyword occurs last wins; DPI: early = fewer than 5 tool calls, no_edit = no write_file without error,
lp = share of run_command calls repeating an earlier identical command, er = share of error steps repeating
an earlier error step's (action, files, first 100 characters of the error).

Length-confound rule (pre-registered 2026-10-12, from TQS section 3.4 and appendix G): every version's
|Spearman rho| with n_tool_calls is reported (`length_check.json`); a version with |rho| > 0.25 is flagged
`length_confounded` and its test-1 AUROC must also be reported inside the interquartile band of n_tool_calls.
A version that passes test 1 only outside that band does not advance.

Changelog: 0.3.2 (2026-10-12): v7 (TQS port); length-confound check; per-task `end_reason`
(context_exceeded / time_limit / tool_budget / error / none) from task_results.jsonl, so ContextWindowExceededError
endings under max_output_tokens 16384 are counted separately (decision of Oct 12). No earlier definition changed.
0.3.1 (2026-10-12, B0 session A traces): per-call token usage read from the agent step's `metrics`
when `tool_call.extra.usage` is absent (it was absent for all 128 calls on Kaggle). No component definition changed.

Choices made here where the handoff left room (change = new version): the v1 combination rule (equal
weights over available components); the v4 veto threshold 0.25; resolution precision counts a call
precise when every named argument resolves exactly, by a unique delimiter-suffix match, or by a unique
case-insensitive match (a suffix tier with more than one candidate is a same-name collision);
search_similar_code arguments are replayed through the same resolver (an approximation of the
embedding-key lookup).
"""


# --------------------------------------------------------------------------------------------
# small stats helpers
# --------------------------------------------------------------------------------------------
def wilson(k, n, z=1.959964):
    if not n:
        return [None, None]
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [round(max(0.0, c - h), 4), round(min(1.0, c + h), 4)]


def nanmean(xs):
    v = [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return sum(v) / len(v) if v else float("nan")


def rate(k, n):
    return k / n if n else float("nan")


def isnan(x):
    return x is None or (isinstance(x, float) and math.isnan(x))


# --------------------------------------------------------------------------------------------
# resolver: line-for-line port of swegemma 0.2.11 graph_utils.resolve_node_name, plus the tier
# --------------------------------------------------------------------------------------------
def resolve_with_tier(query, node_list, node_set=None):
    """Return (resolved_node or None, tier, n_candidates_in_tier)."""
    q = str(query or "").strip()
    if not q or not node_list:
        return None, "none", 0
    node_set = node_set if node_set is not None else set(node_list)
    bare = q.split(";", 1)[1] if ";" in q else q
    if q in node_set:
        return q, "exact", 1
    if bare in node_set:
        return bare, "exact", 1
    key = lambda x: (len(x.split(".")), len(x), x)
    cands = [n for n in node_list if n.endswith(f".{bare}") or n.endswith(f";{bare}") or n.endswith(f"/{bare}")]
    if cands:
        return min(cands, key=key), "suffix", len(cands)
    lq, lf = bare.lower(), q.lower()
    ci = [n for n in node_list if n.lower() == lf or n.lower() == lq]
    if ci:
        return min(ci, key=key), "ci_exact", len(ci)
    cis = [n for n in node_list if n.lower().endswith(f".{lq}") or n.lower().endswith(f";{lq}") or n.lower().endswith(f"/{lq}")]
    if cis:
        return min(cis, key=key), "ci_suffix", len(cis)
    if not any(c.isspace() for c in bare):
        sub = [n for n in node_list if bare in n or lq in n.lower()]
        if sub:
            return min(sub, key=key), "substring", len(sub)
    return None, "none", 0


def precise_tier(tier, n):
    return tier == "exact" or (tier in ("suffix", "ci_exact", "ci_suffix") and n == 1)


def parity_check(graph_json_path, args, out=None):
    """Compare this port with swegemma's resolve_node_name on the given arguments (needs swegemma)."""
    from swegemma.graph.graph_utils import load_graph_from_json, resolve_node_name
    g = load_graph_from_json(str(graph_json_path))
    nodes = [str(n) for n in g.nodes()]
    ns = set(nodes)
    bad = []
    for a in args:
        mine = resolve_with_tier(a, nodes, ns)[0]
        theirs = resolve_node_name(a, g)
        if mine != theirs:
            bad.append({"arg": a, "port": mine, "swegemma": theirs})
    res = {"n": len(args), "mismatches": bad}
    if out:
        Path(out).write_text(json.dumps(res, indent=1))
    return res


# --------------------------------------------------------------------------------------------
# oracle
# --------------------------------------------------------------------------------------------
class Oracle:
    def __init__(self, pack_path, profile_csv=None):
        op = gzip.open if str(pack_path).endswith(".gz") else open
        with op(pack_path, "rt") as f:
            self.pack = json.load(f)
        self.tasks = self.pack["tasks"]
        self.graphs = self.pack["graphs"]
        self._nodelists = {}
        self.profile = {}
        if profile_csv and Path(profile_csv).exists():
            self.profile = {r["instance_id"]: r for r in csv.DictReader(open(profile_csv))}

    def task(self, iid):
        return self.tasks.get(iid)

    def graph(self, key):
        return self.graphs.get(key)

    def node_list(self, key):
        if key not in self._nodelists:
            g = self.graphs.get(key) or {"nodes": {}}
            nl = list(g["nodes"].keys())
            self._nodelists[key] = (nl, set(nl))
        return self._nodelists[key]

    def node_file(self, key, node):
        g = self.graphs.get(key)
        if not g:
            return None
        v = g["nodes"].get(node)
        if v is None:
            return None
        f = g["files"][v[0]]
        return f or None

    def node_is_test(self, key, node):
        g = self.graphs.get(key)
        v = g["nodes"].get(node) if g else None
        if v is None:
            return _looks_test_node(node)
        return bool(v[1])

    def indeg(self, key, node):
        g = self.graphs.get(key)
        v = g["nodes"].get(node) if g else None
        return v[2] if v else 0

    def hub_threshold(self, key, pct):
        g = self.graphs.get(key) or {}
        return g.get(f"indeg_p{pct}", 10 ** 9)


def _looks_test_node(n):
    parts = n.split(".")
    return any(p in ("tests", "test", "testing") or p.startswith("test_") for p in parts)


def norm_path(p):
    p = str(p or "").strip().strip("'\"")
    for pre in ("/workspace/", "workspace/", "./"):
        if p.startswith(pre):
            p = p[len(pre):]
    return p.lstrip("/")


def path_match(agent_path, true_path):
    a, t = norm_path(agent_path), norm_path(true_path)
    if not a or not t:
        return False
    return a == t or t.endswith("/" + a) or a.endswith("/" + t)


def strip_repo(n):
    s = str(n)
    return s.split(";", 1)[1] if ";" in s else s


def need_v1(cell):
    if not cell or "|" not in cell:
        return None
    d, v = cell.split("|", 1)
    if d in ("d1", "d2+") or v == "v_low":
        return 1
    if v == "v_undef" and d == "d_undef":
        return None
    return 0


# --------------------------------------------------------------------------------------------
# trace parsing
# --------------------------------------------------------------------------------------------
def _parse_content(c):
    if isinstance(c, dict):
        return c
    if isinstance(c, list):
        txt = "".join(x.get("text", "") if isinstance(x, dict) else str(x) for x in c)
    else:
        txt = str(c or "")
    try:
        v = json.loads(txt)
        return v if isinstance(v, dict) else {"result": v}
    except Exception:
        return {"raw": txt}


def _content_len(c):
    if isinstance(c, str):
        return len(c)
    return len(json.dumps(c, default=str))


def parse_trace(path):
    """Return dict with calls (ordered), events, diagnostics."""
    d = json.load(open(path))
    steps = d.get("steps", [])
    calls, pending = [], []
    diag = collections.Counter()
    events = []  # (kind, call_index_at_time, detail)
    turn = 0
    thought_buf = []  # agent text (thinking / text) since the last tool call, for TQS intent-action consistency
    for s in steps:
        src = s.get("source")
        ex = s.get("extra") or {}
        et = ex.get("event_type")
        if src == "agent" and s.get("message") and et in ("thinking", "text", None):
            thought_buf.append(str(s.get("message")))
        diag[f"step:{src}:{et or ('tool_calls' if s.get('tool_calls') else 'obs' if s.get('observation') else 'other')}"] += 1
        if src == "agent" and (s.get("tool_calls") or et in ("thinking", "text", "final")):
            turn += 1 if (s.get("tool_calls") or et in ("text", "final")) else 0
        if et == "compaction":
            events.append({"kind": "compaction", "at_call": len(calls), "chars": len(str(s.get("message") or ""))})
        elif et == "continuation_nudge":
            msg = str(s.get("message") or "")
            cause = ("truncated_tool_call" if "was cut off" in msg else
                     "length" if "token limit while thinking" in msg else "no_call")
            events.append({"kind": "nudge", "cause": cause, "at_call": len(calls)})
        elif src == "agent" and et in ("text", "final") and RAW_TOOLCALL_RE.search(str(s.get("message") or "")):
            events.append({"kind": "parser_failure_text", "at_call": len(calls),
                           "snippet": str(s.get("message"))[:200]})
        step_metrics = s.get("metrics") if s.get("tool_calls") else None
        for j, tc in enumerate(s.get("tool_calls") or []):
            tx = tc.get("extra") or {}
            # 0.3.1: on Kaggle (swegemma 0.2.11) usage arrives as separate 'usage' events that the ATIF converter
            # merges into the step's metrics, not into tool_call.extra.usage; attach it to the step's first call.
            if not tx.get("usage") and j == 0 and step_metrics:
                tx = {**tx, "usage": step_metrics}
            c = {"idx": len(calls), "step_id": s.get("step_id"), "turn": turn,
                 "name": tc.get("function_name") or "", "args": tc.get("arguments") or {},
                 "author": tx.get("author") or ex.get("author") or "",
                 "usage": tx.get("usage") or None, "elapsed_s": tx.get("elapsed_s"),
                 "thought": "\n".join(thought_buf) if j == 0 else "",
                 "result": None, "result_raw_len": None, "result_text": "", "matched_by": None}
            if j == len(s.get("tool_calls") or []) - 1:
                thought_buf = []
            calls.append(c)
            pending.append(c)
            if tx.get("usage"):
                diag["calls_with_usage"] += 1
        obs = s.get("observation")
        if obs is not None:
            oname = (obs.get("extra") or {}).get("tool_name") or ""
            target = None
            for i, c in enumerate(pending):
                if oname and c["name"] == oname:
                    target = pending.pop(i)
                    target["matched_by"] = "name"
                    break
            if target is None and pending:
                target = pending.pop(0)
                target["matched_by"] = "order"
            if target is None:
                diag["results_unmatched"] += 1
                continue
            target["result"] = _parse_content(obs.get("content"))
            target["result_raw_len"] = _content_len(obs.get("content"))
            oc = obs.get("content")
            target["result_text"] = oc if isinstance(oc, str) else json.dumps(oc, default=str)
    for c in pending:
        c["matched_by"] = "no_result"
    diag["calls"] = len(calls)
    diag["matched_by_name"] = sum(c["matched_by"] == "name" for c in calls)
    diag["matched_by_order"] = sum(c["matched_by"] == "order" for c in calls)
    diag["calls_without_result"] = sum(c["matched_by"] == "no_result" for c in calls)
    diag["compactions"] = sum(e["kind"] == "compaction" for e in events)
    diag["nudges"] = sum(e["kind"] == "nudge" for e in events)
    fm = d.get("final_metrics") or {}
    return {"calls": calls, "events": events, "diag": dict(diag), "final_metrics": fm,
            "schema": d.get("schema_version"), "n_steps": len(steps), "step_keys": sorted({k for s in steps for k in s})}


def returned_nodes(call):
    r = call.get("result") or {}
    name = call["name"]
    out = []
    if name == "get_code_neighbors":
        out = r.get("neighbors") or []
    elif name == "search_similar_code":
        out = [x.get("node_name") if isinstance(x, dict) else x for x in (r.get("results") or [])]
    elif name == "get_code_subgraph":
        out = list(r.get("nodes") or [])
        for e in r.get("edges") or []:
            if isinstance(e, dict):
                out += [e.get("from"), e.get("to")]
    seen, res = set(), []
    for n in out:
        if not n:
            continue
        n = strip_repo(n)
        if n not in seen:
            seen.add(n)
            res.append(n)
    return res


def named_args(call):
    a = call.get("args") or {}
    name = call["name"]
    if name == "get_code_neighbors":
        v = a.get("node")
        return [v] if v not in (None, "") else []
    if name == "search_similar_code":
        v = a.get("query")
        return [v] if v not in (None, "") else []
    if name == "get_code_subgraph":
        v = a.get("nodes")
        if isinstance(v, str):
            try:
                v = json.loads(v)
            except Exception:
                v = [x.strip() for x in v.strip("[]").split(",") if x.strip()]
        return [str(x) for x in (v or [])]
    return []


def call_files_read(call):
    """Files this call reads: (path, via) list."""
    a = call.get("args") or {}
    if call["name"] == "read_file":
        return [(norm_path(a.get("filepath") or a.get("path") or ""), "read_file")]
    if call["name"] == "run_command":
        cmd = str(a.get("command") or "")
        if SHELL_READ_RE.search(cmd):
            return [(norm_path(p), "shell") for p in PATH_TOKEN_RE.findall(cmd)]
    return []


def call_error(call):
    r = call.get("result") or {}
    if r.get("status") == "error":
        return r.get("error_type") or "error", str(r.get("error_message") or "")[:200]
    if "error" in r and isinstance(r.get("error"), str):
        return "error", r["error"][:200]
    return None, None


# --------------------------------------------------------------------------------------------
# scoring one trace
# --------------------------------------------------------------------------------------------
def score_trace(iid, parsed, oracle, run_meta, brief_files=None, extra_known_tools=()):
    t = oracle.task(iid) or {}
    key = t.get("graph_key")
    nodes, nodeset = oracle.node_list(key) if key else ([], set())
    calls = parsed["calls"]
    R = {k: set(t.get(k, [])) for k in ("R1", "R2", "R3")}
    gold_files = list(t.get("gold_files", []))
    missing = set(t.get("R2_missing_from_provided", []))
    known = HARNESS_TOOLS | set(extra_known_tools)

    # reads in order
    reads = []  # (call_idx, path, via)
    for c in calls:
        for p, via in call_files_read(c):
            if p:
                reads.append((c["idx"], p, via))

    call_rows = []
    seen_nodes = set()
    seen_R = {k: set() for k in R}
    n_unknown = n_malformed = n_unresolvable = 0
    for c in calls:
        etype, emsg = call_error(c)
        if c["name"] not in known:
            n_unknown += 1
        elif etype and (etype in MALFORMED_ERR_TYPES or "argument" in (emsg or "").lower()):
            n_malformed += 1
        if c["name"] not in GRAPH_TOOLS:
            continue
        args = named_args(c)
        res = [resolve_with_tier(a, nodes, nodeset) for a in args]
        precise = bool(args) and all(precise_tier(tier, n) for _, tier, n in res)
        unresolved = [a for a, (r, _, _) in zip(args, res) if r is None]
        if unresolved:
            n_unresolvable += 1
        ret = returned_nodes(c)
        ret_set = set(ret)
        ret_nontest = [n for n in ret if not oracle.node_is_test(key, n)]
        row = {
            "run": run_meta["run"], "arm": run_meta["arm"], "seed": run_meta["seed"], "instance_id": iid,
            "call_idx": c["idx"], "turn": c["turn"], "tool": c["name"], "args": c["args"], "author": c["author"],
            "status": (c.get("result") or {}).get("status"), "error_type": etype,
            "matched_by": c["matched_by"], "result_raw_len": c["result_raw_len"],
            "resolved": [r for r, _, _ in res], "resolve_tier": [tier for _, tier, _ in res],
            "resolve_n_candidates": [n for _, _, n in res], "precise": precise,
            "n_returned": len(ret), "n_returned_test": len(ret) - len(ret_nontest),
        }
        for k in R:
            hit = ret_set & R[k]
            row[f"yield_{k}"] = bool(hit)
            row[f"novel_yield_{k}"] = bool(hit - seen_R[k])
            seen_R[k] |= hit
        row["returned_all_seen"] = bool(ret) and ret_set <= seen_nodes
        seen_nodes |= ret_set
        # hub
        qnode = next((r for r, _, _ in res if r), None)
        row["query_indeg"] = oracle.indeg(key, qnode) if qnode else 0
        row["test_share"] = (len(ret) - len(ret_nontest)) / len(ret) if ret else 0.0
        for pct in (90, 95):
            thr = oracle.hub_threshold(key, pct)
            for sh in (0.5, 0.75):
                hub = bool(qnode) and row["query_indeg"] > thr and bool(ret) and row["test_share"] >= sh
                row[f"waste_p{pct}_s{int(sh*100)}"] = row["returned_all_seen"] or hub
        # link-following
        ret_files = {oracle.node_file(key, n) for n in ret_nontest}
        ret_files.discard(None)
        if brief_files:
            ret_files |= set(brief_files)
        row["returned_files"] = sorted(ret_files)
        for w in (1, 2, 3):
            nxt = [r for r in reads if c["idx"] < r[0] <= c["idx"] + w and r[2] == "read_file"]
            row[f"follow_w{w}"] = any(path_match(p, f) for _, p, _ in nxt for f in ret_files) if ret_files else None
        # async hazard: argument names a gold/R2 node absent from the provided graph
        haz = []
        for a in args:
            b = strip_repo(a)
            for m in missing:
                if m == b or m.endswith("." + b):
                    haz.append(m)
        row["async_hazard"] = bool(haz)
        row["hazard_nodes"] = sorted(set(haz))
        call_rows.append(row)

    G = len(call_rows)
    with_ret = [r for r in call_rows if r["n_returned"]]
    task = {
        "run": run_meta["run"], "arm": run_meta["arm"], "seed": run_meta["seed"], "instance_id": iid,
        "repo": t.get("repo") or iid.split("_")[0], "cell": t.get("cell"), "graph_key": key,
        "in_oracle": bool(t), "n_tool_calls": len(calls), "n_graph_calls": G,
        "n_graph_calls_with_return": len(with_ret),
        "resolution_precision": rate(sum(r["precise"] for r in call_rows), sum(bool(named_args({"name": r["tool"], "args": r["args"]})) for r in call_rows)),
    }
    for k in R:
        task[f"yield_{k}"] = rate(sum(r[f"yield_{k}"] for r in call_rows), G)
        task[f"novel_yield_{k}"] = rate(sum(r[f"novel_yield_{k}"] for r in call_rows), G)
    for w in (1, 2, 3):
        fr = [r for r in call_rows if r[f"follow_w{w}"] is not None]
        task[f"follow_w{w}"] = rate(sum(bool(r[f"follow_w{w}"]) for r in fr), len(fr))
    for pct in (90, 95):
        for sh in (50, 75):
            task[f"waste_p{pct}_s{sh}"] = rate(sum(r[f"waste_p{pct}_s{sh}"] for r in call_rows), G)
    task["n_unknown_tool"] = n_unknown
    task["n_malformed_args"] = n_malformed
    task["n_unresolvable_graph_args"] = n_unresolvable
    task["halluc_rate"] = rate(n_unknown + n_malformed + n_unresolvable, len(calls))
    task["n_async_hazard"] = sum(r["async_hazard"] for r in call_rows)
    task["async_hazard"] = rate(task["n_async_hazard"], G)
    task["n_parser_failure_text"] = sum(e["kind"] == "parser_failure_text" for e in parsed["events"])
    task["n_nudge_truncated_tool_call"] = sum(e.get("cause") == "truncated_tool_call" for e in parsed["events"])
    task["n_nudge_length"] = sum(e.get("cause") == "length" for e in parsed["events"])
    task["n_nudge_no_call"] = sum(e.get("cause") == "no_call" for e in parsed["events"])
    task["n_compactions"] = sum(e["kind"] == "compaction" for e in parsed["events"])
    task["first_compaction_at_call"] = next((e["at_call"] for e in parsed["events"] if e["kind"] == "compaction"), None)
    # necessity
    nv1 = need_v1(t.get("cell"))
    task["need_v1"] = nv1
    task["made_graph_call"] = int(G > 0)
    task["necessity_v1"] = float(task["made_graph_call"] == nv1) if nv1 is not None else float("nan")
    task["need_v2"] = None
    task["necessity_v2"] = float("nan")
    # localization outcomes
    gold_read_at, read_gold = None, set()
    for ci, p, via in reads:
        for gf in gold_files:
            if path_match(p, gf):
                read_gold.add(gf)
                if gold_read_at is None:
                    gold_read_at = ci + 1
    task["n_gold_files"] = len(gold_files)
    task["steps_to_first_gold_read"] = gold_read_at
    task["gold_file_coverage"] = rate(len(read_gold), len(gold_files))
    task["second_file_discovery"] = (float(len(read_gold) >= 2) if len(gold_files) >= 2 else float("nan"))
    task["gold_file_read"] = int(bool(read_gold))
    # tokens
    fm = parsed.get("final_metrics") or {}
    task["total_prompt_tokens"] = fm.get("total_prompt_tokens")
    task["total_completion_tokens"] = fm.get("total_completion_tokens")
    task["resolved"] = run_meta.get("resolved", {}).get(iid)
    task["end_reason"] = end_reason(run_meta.get("records", {}).get(iid), len(calls))
    task["ended_context_exceeded"] = int(task["end_reason"] == "context_exceeded")
    task.update(tqs_score(calls))
    return task, call_rows


CONTEXT_RE = re.compile(r"ContextWindowExceeded|context.?window|maximum context|context length", re.I)
TIME_RE = re.compile(r"time.?limit|max_time|timed? ?out|TimeoutError", re.I)
BUDGET_RE = re.compile(r"max_tool_calls|tool.?call (budget|limit)|max_turns|turn limit", re.I)


def end_reason(rec, n_calls, max_tool_calls=50):
    """How the agent run ended, from the task_results.jsonl record (error-like string fields only)."""
    if not rec:
        return "unknown"
    txt = " ".join(str(v) for k, v in rec.items()
                   if isinstance(v, str) and re.search(r"error|reason|message|status|exception|stop", k, re.I))
    if CONTEXT_RE.search(txt):
        return "context_exceeded"
    if TIME_RE.search(txt):
        return "time_limit"
    if BUDGET_RE.search(txt) or n_calls >= max_tool_calls:
        return "tool_budget"
    if txt.strip() and re.search(r"error|exception|fail", txt, re.I):
        return "error"
    return "none"


# --------------------------------------------------------------------------------------------
# v7: TQS port (Li et al. 2026, arXiv 2610.13152, section 3.3). Deviations: see VERSIONS_DOC.
# --------------------------------------------------------------------------------------------
TQS_WEIGHTS = {"oec": 0.30, "iac": 0.25, "psn": 0.10, "tte": 0.10, "ped": 0.10, "dpi": 0.10, "scp": 0.05}
TQS_ALPHABET = ("think", "read_file", "write_file", "execute_bash", "search", "submit", "other")
TQS_LEXICON = {  # coarse intent -> regex (ours; the paper's lexicon is not published)
    "reading": r"\b(read|look at|inspect|open|view|examine|check (the )?(file|code|implementation|source)|understand)\b",
    "editing": r"\b(edit|modify|change|fix|patch|update|replace|rewrite|implement|add (a |the )?(check|line|parameter|argument|condition))\b",
    "execution": r"\b(run|test|execute|reproduce|pytest|verify|try running|install)\b",
    "search": r"\b(search|find|grep|locate|look for|where is|neighbou?rs|callers|callees|subgraph)\b",
}
TQS_ACTION_INTENT = {"read_file": "reading", "write_file": "editing", "execute_bash": "execution", "search": "search"}
SEARCH_CMD_RE = re.compile(r"^\s*(cd [^;&]+[;&]+\s*)?(grep|rg|find|ls|ag|ack|tree)\b")


def tqs_action(c):
    n = c["name"]
    if n == "read_file":
        return "read_file"
    if n in ("edit_file", "write_file"):
        return "write_file"
    if n == "run_command":
        return "search" if SEARCH_CMD_RE.search(str((c.get("args") or {}).get("command") or "")) else "execute_bash"
    if n in GRAPH_TOOLS:
        return "search"
    if n == "submit_patch":
        return "submit"
    return "other"


def tqs_files(c):
    a = c.get("args") or {}
    if c["name"] in ("read_file", "edit_file", "write_file"):
        p = norm_path(a.get("filepath") or a.get("path") or "")
        return {p} if p else set()
    if c["name"] == "run_command":
        return {norm_path(p) for p in PATH_TOKEN_RE.findall(str(a.get("command") or ""))}
    return set()


def tqs_error(c):
    r = c.get("result") or {}
    if r.get("status") == "error":
        return True
    for k in ("exit_code", "returncode", "return_code"):
        v = r.get(k)
        if isinstance(v, (int, float)) and v != 0:
            return True
    return False


def _char_entropy(s):
    if not s:
        return 0.0
    cnt = collections.Counter(s)
    n = len(s)
    return -sum(v / n * math.log2(v / n) for v in cnt.values())


def _ranks(xs):
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    r = [0.0] * len(xs)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and xs[order[j + 1]] == xs[order[i]]:
            j += 1
        for k in range(i, j + 1):
            r[order[k]] = (i + j) / 2 + 1
        i = j + 1
    return r


def kendall_tau_b(x, y):
    n = len(x)
    nc = nd = tx = ty = 0
    for i in range(n):
        for j in range(i + 1, n):
            dx, dy = x[i] - x[j], y[i] - y[j]
            if dx == 0 and dy == 0:
                continue
            if dx == 0:
                tx += 1
            elif dy == 0:
                ty += 1
            elif dx * dy > 0:
                nc += 1
            else:
                nd += 1
    den = math.sqrt((nc + nd + tx) * (nc + nd + ty))
    return (nc - nd) / den if den else 0.0


def tqs_score(calls):
    """Return dict of TQS components (NaN = undefined) and the composite `tqs`."""
    n = len(calls)
    nan = float("nan")
    out = {f"tqs_{k}": nan for k in TQS_WEIGHTS}
    out["tqs"] = nan
    if n < 3:
        return out
    acts = [tqs_action(c) for c in calls]
    files = [tqs_files(c) for c in calls]
    errs = [tqs_error(c) for c in calls]
    # A1 OEC
    H = [_char_entropy(c.get("result_text") or "") for c in calls]
    k0 = min(5, n)
    h_open = sum(H[:k0]) / k0
    if h_open > 0:
        sm = [sum(H[max(0, t - 2):t + 1]) / len(H[max(0, t - 2):t + 1]) for t in range(n)]
        out["tqs_oec"] = min(1.0, max(0.0, 1 - min(x / h_open for x in sm)))
    # A2 PSN: distinct files in a 5-step window, Kendall tau vs step index over the second half
    scope = [len(set().union(*files[max(0, t - 4):t + 1])) for t in range(n)]
    half = list(range(n // 2, n))
    if len(half) >= 3:
        tau = kendall_tau_b([float(t) for t in half], [float(scope[t]) for t in half])
        out["tqs_psn"] = min(1.0, max(0.0, (1 - tau) / 2))
    # B1 IAC
    hits = tot = 0
    for c, a in zip(calls, acts):
        if a not in TQS_ACTION_INTENT:
            continue
        txt = (c.get("thought") or "")[-400:].lower()
        best, pos = None, -1
        for intent, rx in TQS_LEXICON.items():
            for m in re.finditer(rx, txt):
                if m.start() > pos:
                    best, pos = intent, m.start()
        if best is None:
            continue
        tot += 1
        hits += best == TQS_ACTION_INTENT[a]
    if tot:
        out["tqs_iac"] = hits / tot
    # C1 TTE
    pairs = collections.Counter(zip(acts, acts[1:]))
    m = sum(pairs.values())
    if m:
        out["tqs_tte"] = -sum(v / m * math.log2(v / m) for v in pairs.values()) / math.log2(len(TQS_ALPHABET) ** 2)
    # C2 SCP
    first_edit = next((t for t in range(n) if acts[t] == "write_file" and not errs[t]), None)
    if first_edit is None:
        out["tqs_scp"] = 0.0
    else:
        u = first_edit / n
        d = max(0.0, 0.2 - u, u - 0.5)
        out["tqs_scp"] = math.exp(-d * d / (2 * 0.2 ** 2))
    # C3 PED
    E = [t for t in range(n - 1) if errs[t]]
    if E:
        ds = []
        for t in E:
            a, b = files[t], files[t + 1]
            jd = (1 - len(a & b) / len(a | b)) if (a | b) else 0.0
            ds.append(0.5 * ((acts[t + 1] != acts[t]) + jd))
        out["tqs_ped"] = sum(ds) / len(ds)
    # D1 DPI
    cmds = [str((c.get("args") or {}).get("command") or "") for c in calls if c["name"] == "run_command"]
    lp = (sum(1 for i, x in enumerate(cmds) if x in cmds[:i]) / len(cmds)) if cmds else 0.0
    ekeys = [(acts[t], tuple(sorted(files[t])), (calls[t].get("result_text") or "")[:100]) for t in range(n) if errs[t]]
    er = (sum(1 for i, x in enumerate(ekeys) if x in ekeys[:i]) / len(ekeys)) if ekeys else 0.0
    p = 0.4 * (n < 5) + 0.4 * (first_edit is None) + min(0.3, 3 * lp) + min(0.3, 0.6 * er)
    out["tqs_dpi"] = 1 - min(1.0, p)
    defined = {k: out[f"tqs_{k}"] for k in TQS_WEIGHTS if not isnan(out[f"tqs_{k}"])}
    if len(defined) >= 4:
        out["tqs"] = sum(TQS_WEIGHTS[k] * v for k, v in defined.items()) / sum(TQS_WEIGHTS[k] for k in defined)
    return out


# --------------------------------------------------------------------------------------------
# length confound and AUROC (pre-registered rule: VERSIONS_DOC)
# --------------------------------------------------------------------------------------------
def spearman(x, y):
    pts = [(a, b) for a, b in zip(x, y) if not isnan(a) and not isnan(b)]
    if len(pts) < 3:
        return float("nan")
    rx, ry = _ranks([a for a, _ in pts]), _ranks([b for _, b in pts])
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    sx = math.sqrt(sum((a - mx) ** 2 for a in rx))
    sy = math.sqrt(sum((b - my) ** 2 for b in ry))
    return sum((a - mx) * (b - my) for a, b in zip(rx, ry)) / (sx * sy) if sx and sy else float("nan")


def auroc(scores, labels):
    pts = [(s, l) for s, l in zip(scores, labels) if not isnan(s) and l is not None]
    pos = [s for s, l in pts if l]
    neg = [s for s, l in pts if not l]
    if not pos or not neg:
        return float("nan")
    r = _ranks([s for s, _ in pts])
    rp = sum(ri for ri, (_, l) in zip(r, pts) if l)
    return (rp - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


VERSION_KEYS = ("v0", "v1", "v2", "v3", "v4", "v5", "v6", "v7")


def length_check(task_rows):
    L = [float(t["n_tool_calls"]) for t in task_rows]
    cov = [t["gold_file_coverage"] for t in task_rows if not isnan(t.get("gold_file_coverage"))]
    med = sorted(cov)[len(cov) // 2] if cov else None
    res_lab = [t["resolved"] if t.get("resolved") is not None else None for t in task_rows]
    cov_lab = [None if (med is None or isnan(t.get("gold_file_coverage"))) else t["gold_file_coverage"] > med for t in task_rows]
    sl = sorted(L)
    q1, q3 = (sl[len(sl) // 4], sl[(3 * len(sl)) // 4]) if sl else (None, None)
    band = [q1 is not None and q1 <= x <= q3 for x in L]
    out = {"n_tasks": len(task_rows), "length_field": "n_tool_calls", "iqr_band": [q1, q3],
           "n_in_band": sum(band), "gold_coverage_median": med, "versions": {}}
    for v in VERSION_KEYS:
        s = [float(t[v]) if not isnan(t.get(v)) else float("nan") for t in task_rows]
        rho = spearman(s, L)
        rnd = lambda x: None if isnan(x) else round(x, 4)
        out["versions"][v] = {
            "spearman_rho_length": rnd(rho),
            "length_confounded": None if isnan(rho) else abs(rho) > 0.25,
            "auroc_resolved": rnd(auroc(s, res_lab)),
            "auroc_resolved_in_band": rnd(auroc([x for x, b in zip(s, band) if b], [l for l, b in zip(res_lab, band) if b])),
            "auroc_gold_cov_hi": rnd(auroc(s, cov_lab)),
            "auroc_gold_cov_hi_in_band": rnd(auroc([x for x, b in zip(s, band) if b], [l for l, b in zip(cov_lab, band) if b])),
        }
    return out


# --------------------------------------------------------------------------------------------
# versions
# --------------------------------------------------------------------------------------------
def _neg(x):
    return float("nan") if isnan(x) else 1.0 - x


def v1_like(t, yield_key="yield_R2", nec_key="necessity_v1"):
    if t["n_graph_calls"] == 0:
        return t[nec_key]
    return nanmean([t[yield_key], t["follow_w2"], t[nec_key], _neg(t["waste_p95_s50"])])


def v4(t, veto=0.25):
    if t["need_v1"] != 1:
        return float("nan")
    if not isnan(t["halluc_rate"]) and t["halluc_rate"] > veto:
        return 0.0
    return 0.0 if t["n_graph_calls"] == 0 else t["novel_yield_R2"]


V6_FEATURES = ["resolution_precision", "yield_R2", "follow_w2", "necessity_v1", "waste_p95_s50",
               "halluc_rate", "async_hazard"]


def v6(t, coef):
    if not coef:
        return float("nan")
    z = coef["intercept"]
    for f in V6_FEATURES:
        x = t.get(f)
        x = coef["fill"][f] if isnan(x) else x
        z += coef["w"][f] * (x - coef["mean"][f]) / (coef["sd"][f] or 1.0)
    return 1 / (1 + math.exp(-z))


def fit_v6(task_rows, target="gold_file_coverage", l2=1.0, iters=3000, lr=0.1):
    rows = [t for t in task_rows if not isnan(t.get(target))]
    if len(rows) < 20:
        return None
    med = sorted(t[target] for t in rows)[len(rows) // 2]
    y = [1.0 if t[target] > med else 0.0 for t in rows]
    fill, mean, sd = {}, {}, {}
    for f in V6_FEATURES:
        vals = [t[f] for t in rows if not isnan(t.get(f))]
        fill[f] = sum(vals) / len(vals) if vals else 0.0
        xs = [fill[f] if isnan(t.get(f)) else t[f] for t in rows]
        mean[f] = sum(xs) / len(xs)
        sd[f] = math.sqrt(sum((x - mean[f]) ** 2 for x in xs) / len(xs)) or 1.0
    X = [[((fill[f] if isnan(t.get(f)) else t[f]) - mean[f]) / sd[f] for f in V6_FEATURES] for t in rows]
    w, b = [0.0] * len(V6_FEATURES), 0.0
    n = len(rows)
    for _ in range(iters):
        gw, gb = [l2 * wi / n for wi in w], 0.0
        for xi, yi in zip(X, y):
            p = 1 / (1 + math.exp(-(b + sum(a * c for a, c in zip(w, xi)))))
            e = p - yi
            gb += e / n
            for j in range(len(w)):
                gw[j] += e * xi[j] / n
        w = [wi - lr * g for wi, g in zip(w, gw)]
        b -= lr * gb
    return {"target": target, "median": med, "n": n, "intercept": b, "w": dict(zip(V6_FEATURES, w)),
            "fill": fill, "mean": mean, "sd": sd}


def add_versions(t, coef=None):
    t["v0"] = t["n_graph_calls"]
    t["v1"] = v1_like(t)
    t["v2"] = v1_like(t, nec_key="necessity_v2") if not isnan(t["necessity_v2"]) else float("nan")
    t["v3"] = v1_like(t, yield_key="yield_R1")
    t["v4"] = v4(t)
    t["v5"] = t["follow_w2"]
    t["v6"] = v6(t, coef)
    t["v7"] = t.get("tqs", float("nan"))
    return t


# --------------------------------------------------------------------------------------------
# run directories
# --------------------------------------------------------------------------------------------
def run_meta_for(run_dir, arm=None):
    run_dir = Path(run_dir)
    name = run_dir.name
    m = re.search(r"_?s(?:eed)?(\d+)$", name)
    seed = int(m.group(1)) if m else None
    resolved, records = {}, {}
    tr = run_dir / "task_results.jsonl"
    if tr.exists():
        for line in open(tr):
            try:
                r = json.loads(line)
            except Exception:
                continue
            resolved[r.get("instance_id")] = bool(r.get("resolved"))
            records[r.get("instance_id")] = r
    return {"run": str(run_dir), "arm": arm or re.sub(r"_?s(?:eed)?\d+$", "", name), "seed": seed,
            "resolved": resolved, "records": records}


def iter_traces(run_dir):
    for p in sorted(Path(run_dir, "traces").glob("trace_*.json")):
        yield p.stem[len("trace_"):], p


def log_parser_hits(run_dir, iid):
    p = Path(run_dir, "logs", f"{iid}.log")
    if not p.exists():
        return None
    txt = p.read_text(errors="replace")
    return len(PARSER_ERROR_RE.findall(txt))


def gold_found_by_task(task_rows):
    by = collections.defaultdict(list)
    for t in task_rows:
        by[t["instance_id"]].append(t["gold_file_read"])
    return {k: (sum(v) / len(v)) >= 0.5 for k, v in by.items()}


# --------------------------------------------------------------------------------------------
# summary
# --------------------------------------------------------------------------------------------
COMPONENT_EVENTS = {
    # component: (numerator field on call rows or task rows, denominator)
    "resolution_precision": ("calls", lambda r: r["precise"], lambda r: True),
    "yield_R1": ("calls", lambda r: r["yield_R1"], lambda r: True),
    "yield_R2": ("calls", lambda r: r["yield_R2"], lambda r: True),
    "yield_R3": ("calls", lambda r: r["yield_R3"], lambda r: True),
    "novel_yield_R2": ("calls", lambda r: r["novel_yield_R2"], lambda r: True),
    "follow_w1": ("calls", lambda r: bool(r["follow_w1"]), lambda r: r["follow_w1"] is not None),
    "follow_w2": ("calls", lambda r: bool(r["follow_w2"]), lambda r: r["follow_w2"] is not None),
    "follow_w3": ("calls", lambda r: bool(r["follow_w3"]), lambda r: r["follow_w3"] is not None),
    "waste_p95_s50": ("calls", lambda r: r["waste_p95_s50"], lambda r: True),
    "waste_p90_s75": ("calls", lambda r: r["waste_p90_s75"], lambda r: True),
    "async_hazard": ("calls", lambda r: r["async_hazard"], lambda r: True),
    "necessity_v1": ("tasks", lambda t: t["necessity_v1"] == 1.0, lambda t: not isnan(t["necessity_v1"])),
    "necessity_v2": ("tasks", lambda t: t["necessity_v2"] == 1.0, lambda t: not isnan(t["necessity_v2"])),
    "made_graph_call": ("tasks", lambda t: t["made_graph_call"] == 1, lambda t: True),
    "gold_file_read": ("tasks", lambda t: t["gold_file_read"] == 1, lambda t: t["n_gold_files"] > 0),
    "resolved": ("tasks", lambda t: t["resolved"] is True, lambda t: t["resolved"] is not None),
}


def summarize(task_rows, call_rows):
    def block(trows, crows):
        out = {"n_tasks": len(trows), "n_graph_calls": len(crows),
               "n_tool_calls": sum(t["n_tool_calls"] for t in trows)}
        for comp, (lvl, num, den) in COMPONENT_EVENTS.items():
            rows = crows if lvl == "calls" else trows
            d = [r for r in rows if den(r)]
            k = sum(bool(num(r)) for r in d)
            out[comp] = {"k": k, "n": len(d), "rate": round(k / len(d), 4) if d else None, "wilson95": wilson(k, len(d))}
        hal = sum(t["n_unknown_tool"] + t["n_malformed_args"] + t["n_unresolvable_graph_args"] for t in trows)
        out["hallucination_channel"] = {"k": hal, "n": out["n_tool_calls"], "rate": round(hal / out["n_tool_calls"], 4) if out["n_tool_calls"] else None,
                                        "wilson95": wilson(hal, out["n_tool_calls"])}
        out["parser_failures_text"] = sum(t["n_parser_failure_text"] for t in trows)
        out["end_reason"] = dict(collections.Counter(t["end_reason"] for t in trows))
        ce = sum(t["ended_context_exceeded"] for t in trows)
        out["ended_context_exceeded"] = {"k": ce, "n": len(trows), "wilson95": wilson(ce, len(trows))}
        for v in VERSION_KEYS:
            out[f"mean_{v}"] = (lambda m: None if isnan(m) else round(m, 4))(nanmean([t[v] for t in trows]))
        return out

    s = {"scorecard_version": SCORECARD_VERSION, "pooled": block(task_rows, call_rows), "by_arm": {}, "by_repo": {}, "by_cell": {}}
    for field, dest in (("arm", "by_arm"), ("repo", "by_repo"), ("cell", "by_cell")):
        keys = sorted({str(t[field]) for t in task_rows})
        for k in keys:
            tr = [t for t in task_rows if str(t[field]) == k]
            ids = {(t["run"], t["instance_id"]) for t in tr}
            cr = [c for c in call_rows if (c["run"], c["instance_id"]) in ids]
            s[dest][k] = block(tr, cr)
    return s


# --------------------------------------------------------------------------------------------
# main
# --------------------------------------------------------------------------------------------
def dump_format(run_dirs, out=sys.stdout):
    for rd in run_dirs:
        for iid, p in iter_traces(rd):
            parsed = parse_trace(p)
            print(f"== {p}  schema={parsed['schema']} steps={parsed['n_steps']}", file=out)
            print("   step keys:", parsed["step_keys"], file=out)
            print("   diag:", json.dumps(parsed["diag"], sort_keys=True), file=out)
            print("   events:", collections.Counter(e["kind"] + (":" + e["cause"] if "cause" in e else "") for e in parsed["events"]), file=out)
            print("   tool names:", collections.Counter(c["name"] for c in parsed["calls"]), file=out)
            for c in parsed["calls"][:8]:
                r = c["result"] or {}
                print(f"   #{c['idx']:>2} turn={c['turn']} {c['name']}({json.dumps(c['args'])[:120]}) -> "
                      f"status={r.get('status')} raw_len={c['result_raw_len']} matched={c['matched_by']} usage={c['usage']}", file=out)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--results", nargs="+", required=True)
    ap.add_argument("--oracle-pack", required=True)
    ap.add_argument("--profile")
    ap.add_argument("--out", required=True)
    ap.add_argument("--arm")
    ap.add_argument("--no-tools-results", nargs="*", default=[])
    ap.add_argument("--brief-dir")
    ap.add_argument("--extra-tools", nargs="*", default=[])
    ap.add_argument("--fit-v6", help="task csv to fit v6 on (B2 seed 1, non-slice tasks)")
    ap.add_argument("--v6-coef", help="json with v6 coefficients")
    ap.add_argument("--dump-format", action="store_true")
    a = ap.parse_args(argv)

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    if a.dump_format:
        with open(out / "dump_format.txt", "w") as f:
            dump_format(a.results, out=f)
        print((out / "dump_format.txt").read_text())

    oracle = Oracle(a.oracle_pack, a.profile)
    coef = json.load(open(a.v6_coef)) if a.v6_coef else None
    task_rows, call_rows, parse_report = [], [], {}
    for rd in a.results:
        meta = run_meta_for(rd, a.arm)
        for iid, p in iter_traces(rd):
            parsed = parse_trace(p)
            brief = None
            if a.brief_dir and Path(a.brief_dir, f"{iid}.md").exists():
                brief = sorted(set(PATH_TOKEN_RE.findall(Path(a.brief_dir, f"{iid}.md").read_text())))
            t, cr = score_trace(iid, parsed, oracle, meta, brief, a.extra_tools)
            t["log_parser_error_hits"] = log_parser_hits(rd, iid)
            task_rows.append(t)
            call_rows.extend(cr)
            parse_report[f"{meta['run']}::{iid}"] = parsed["diag"]

    if a.no_tools_results:
        nt_rows = []
        for rd in a.no_tools_results:
            meta = run_meta_for(rd, "no_tools")
            for iid, p in iter_traces(rd):
                t, _ = score_trace(iid, parse_trace(p), oracle, meta)
                nt_rows.append(t)
        found_tools, found_nt = gold_found_by_task(task_rows), gold_found_by_task(nt_rows)
        for t in task_rows:
            if t["instance_id"] in found_nt:
                need = int(found_tools.get(t["instance_id"], False) and not found_nt[t["instance_id"]])
                t["need_v2"] = need
                t["necessity_v2"] = float(t["made_graph_call"] == need)

    if a.fit_v6:
        rows = list(csv.DictReader(open(a.fit_v6)))
        for r in rows:
            for k, v in r.items():
                try:
                    r[k] = float(v) if v not in ("", "None") else float("nan")
                except ValueError:
                    pass
        coef = fit_v6(rows)
        (out / "v6_coef.json").write_text(json.dumps(coef, indent=1))

    for t in task_rows:
        add_versions(t, coef)

    with open(out / "scorecard_calls.jsonl", "w") as f:
        for r in call_rows:
            f.write(json.dumps(r, default=str) + "\n")
    if task_rows:
        cols = list(task_rows[0].keys())
        with open(out / "scorecard_tasks.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for t in task_rows:
                w.writerow({k: ("" if v is None or (isinstance(v, float) and math.isnan(v)) else v) for k, v in t.items()})
    summ = summarize(task_rows, call_rows)
    lc = length_check(task_rows)
    summ["length_check"] = lc
    (out / "length_check.json").write_text(json.dumps(lc, indent=1))
    summ["inputs"] = {"results": a.results, "oracle_pack": a.oracle_pack, "oracle_pack_version": oracle.pack.get("version"),
                      "no_tools_results": a.no_tools_results, "params_default": PARAMS_DEFAULT}
    (out / "scorecard_summary.json").write_text(json.dumps(summ, indent=1, default=str))
    (out / "parse_report.json").write_text(json.dumps(parse_report, indent=1))
    (out / "scorecard_versions.md").write_text(VERSIONS_DOC.format(ver=SCORECARD_VERSION))
    print(f"scorecard {SCORECARD_VERSION}: {len(task_rows)} task rows, {len(call_rows)} graph-call rows -> {out}")
    return task_rows, call_rows, summ


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/gc/make_slices.py
"""B1: merge the subprocess-sandbox gold/empty pass into the task profile and write the two 24-task slices.

    python make_slices.py --profile profile.csv \
        --empty RUN_DIR [RUN_DIR ...] --gold RUN_DIR [RUN_DIR ...] \
        [--import-check import_check_subprocess.jsonl] --out OUT_DIR [--seed 20261012] [--size 24]

RUN_DIR is a run_arm.py results directory (task_results.jsonl). Pass several empty / gold runs to
detect flaky tests: a task is `sub_valid` only if EVERY empty run fails and EVERY gold run passes,
and (when the import check is given) pytest imported the task's own workspace code.

Eligible for the slices (HANDOFF_planB.md section 2): a2_usable AND sub_valid.

Stratified allocation (deterministic given --seed): eligible tasks are sorted by
(repo, cell, gold_has_async, gold_is_entry_point, random tiebreak from the seed), then dealt one by
one to dev / holdout / rest, each time to the group furthest below its proportional target
(size/N, size/N, rest/N). Every stratum, and every prefix of the sort order, is therefore split in
proportion to within one task, and the totals are exactly size / size / N - 2*size.

Writes OUT/profile.csv (input profile + sub_* columns), OUT/dev_slice.txt, OUT/holdout_slice.txt,
OUT/slices_meta.json (seed, rule, counts, balance tables). The slice files are never rewritten by
hand afterwards (Plan B section 2).
"""
from __future__ import annotations

import argparse
import collections
import csv
import json
import random
from pathlib import Path

STRATA = ("repo", "cell", "gold_has_async", "gold_is_entry_point")


def read_results(dirs):
    """{instance_id: [resolved per run]} over several run dirs (missing task in a run -> None)."""
    runs = []
    for d in dirs:
        p = Path(d) / "task_results.jsonl"
        rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []
        runs.append({r["instance_id"]: r for r in rows})
    ids = set().union(*[set(r) for r in runs]) if runs else set()
    return {i: [r.get(i) for r in runs] for i in ids}, len(runs)


def fmt(rs):
    return ";".join("NA" if r is None else str(bool(r.get("resolved"))) for r in rs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", required=True)
    ap.add_argument("--empty", nargs="+", required=True)
    ap.add_argument("--gold", nargs="+", required=True)
    ap.add_argument("--import-check")
    ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=20261012)
    ap.add_argument("--size", type=int, default=24)
    a = ap.parse_args()

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    prof = list(csv.DictReader(open(a.profile)))
    E, n_e = read_results(a.empty)
    G, n_g = read_results(a.gold)
    IC = {}
    if a.import_check and Path(a.import_check).exists():
        for l in Path(a.import_check).read_text().splitlines():
            if l.strip():
                r = json.loads(l)
                IC[r["instance_id"]] = r

    for row in prof:
        i = row["instance_id"]
        e, g = E.get(i, [None] * n_e), G.get(i, [None] * n_g)
        ic = IC.get(i, {})
        row["sub_import_verdict"] = ic.get("verdict", "" if not IC else "missing")
        row["sub_pytest_file_is_workspace"] = "" if not IC else str(ic.get("verdict") == "workspace tested")
        row["sub_empty_resolved"] = fmt(e)
        row["sub_gold_resolved"] = fmt(g)
        row["sub_gold_exit"] = ";".join("NA" if r is None else str(r.get("test_exit_code")) for r in g)
        # Evaluated in this order (Oct 12): a task whose tests import a host copy is out without running anything;
        # a gold failure settles a task without its empty pass (the empty pass is only run on gold-passing tasks).
        ws_ok = (not IC) or ic.get("verdict") == "workspace tested"
        g_have = [r for r in g if r is not None]
        e_have = [r for r in e if r is not None]
        gold_flaky = len({bool(r.get("resolved")) for r in g_have}) > 1
        empty_flaky = len({bool(r.get("resolved")) for r in e_have}) > 1
        gold_passes = bool(g_have) and len(g_have) == n_g and all(bool(r.get("resolved")) for r in g_have)
        empty_fails = bool(e_have) and len(e_have) == n_e and all(not r.get("resolved") for r in e_have)
        if not ws_ok:
            reason = "released copy tested"
        elif not g_have:
            reason = "incomplete"
        elif gold_flaky:
            reason = "flaky"
        elif not all(bool(r.get("resolved")) for r in g_have):
            reason = "gold fails"
        elif len(g_have) < n_g or len(e_have) < n_e:
            reason = "incomplete"
        elif empty_flaky:
            reason = "flaky"
        elif not empty_fails:
            reason = "empty passes"
        else:
            reason = "ok"
        row["sub_flaky"] = str(gold_flaky or empty_flaky)
        row["sub_valid"] = str(reason == "ok")
        row["sub_reason"] = reason

    fields = list(prof[0].keys())
    with open(out / "profile.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(prof)

    elig = [r for r in prof if r["a2_usable"] == "True" and r["sub_valid"] == "True"]
    n = len(elig)
    if n < 2 * a.size:
        raise SystemExit(f"only {n} eligible tasks (a2_usable and sub_valid); need {2 * a.size}. Nothing written for slices.")
    rng = random.Random(a.seed)
    tie = {r["instance_id"]: rng.random() for r in sorted(elig, key=lambda r: r["instance_id"])}
    order = sorted(elig, key=lambda r: tuple(r[k] for k in STRATA) + (tie[r["instance_id"]],))
    target = {"dev": a.size / n, "holdout": a.size / n, "rest": (n - 2 * a.size) / n}
    got = {"dev": [], "holdout": [], "rest": []}
    for k, r in enumerate(order, 1):
        # deficit after placing this task; ties broken dev > holdout > rest, alternating dev/holdout by parity
        pref = ["dev", "holdout", "rest"] if k % 2 else ["holdout", "dev", "rest"]
        grp = max(pref, key=lambda g: (target[g] * k - len(got[g]), -pref.index(g)))
        got[grp].append(r)
    assert len(got["dev"]) == len(got["holdout"]) == a.size, {g: len(v) for g, v in got.items()}

    for g in ("dev", "holdout"):
        (out / f"{g}_slice.txt").write_text("\n".join(sorted(r["instance_id"] for r in got[g])) + "\n")

    def bal(key):
        t = collections.defaultdict(lambda: {"dev": 0, "holdout": 0, "rest": 0})
        for g, rs in got.items():
            for r in rs:
                t[r[key]][g] += 1
        return dict(sorted(t.items()))

    reasons = collections.Counter((r["repo"], r["sub_reason"]) for r in prof)
    meta = {
        "seed": a.seed, "size": a.size, "n_eligible": n,
        "rule": "eligible = a2_usable and sub_valid (every empty run fails, every gold run passes, pytest imports the workspace); "
                "sorted by " + ", ".join(STRATA) + ", seeded tiebreak; dealt to the group furthest below its proportional target",
        "n_empty_runs": n_e, "n_gold_runs": n_g, "import_check_used": bool(IC),
        "sub_reason_by_repo": {f"{k[0]}|{k[1]}": v for k, v in sorted(reasons.items())},
        "n_sub_valid": sum(r["sub_valid"] == "True" for r in prof),
        "n_flaky": sum(r["sub_flaky"] == "True" for r in prof),
        "balance": {k: bal(k) for k in STRATA + ("multi_file",)},
        "dev": sorted(r["instance_id"] for r in got["dev"]),
        "holdout": sorted(r["instance_id"] for r in got["holdout"]),
    }
    (out / "slices_meta.json").write_text(json.dumps(meta, indent=1))
    print(f"eligible {n}; dev {a.size}, holdout {a.size}, rest {n - 2 * a.size}; seed {a.seed}")
    for k in STRATA:
        print(f"\n{k}:")
        for v, c in meta["balance"][k].items():
            print(f"  {v:18s} dev {c['dev']:2d}  holdout {c['holdout']:2d}  rest {c['rest']:2d}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/gc/profile_input.csv
instance_id,repo,cell,multi_file,a2_usable,gold_has_async,gold_has_async_absent,gold_is_entry_point,entry_hops,n_anchors,a2_gold_passes_post,b0_docker_empty_resolved,b0_docker_gold_resolved,b0_docker_valid
fastapi_11194,fastapi,d_undef|v_low,False,True,True,True,False,none,2,True,False,False,False
fastapi_11355,fastapi,d_undef|v_low,False,True,False,False,False,2,0,True,False,False,False
fastapi_12942,fastapi,d2+|v_low,False,True,False,False,False,5,9,True,False,False,False
fastapi_13207,fastapi,d_undef|v_low,False,False,False,False,False,5,1,False,False,False,False
fastapi_13537,fastapi,d_undef|v_low,False,True,True,True,False,none,4,True,False,False,False
fastapi_13713,fastapi,d_undef|v_high,True,False,False,False,True,0,0,False,False,False,False
fastapi_13786,fastapi,d2+|v_high,True,False,True,True,True,0,15,False,False,False,False
fastapi_13920,fastapi,d2+|v_low,False,True,False,False,False,2,1,True,False,False,False
fastapi_14077,fastapi,d_undef|v_low,True,False,True,True,False,none,0,False,False,False,False
fastapi_14099,fastapi,d_undef|v_high,True,True,True,True,False,none,3,True,False,False,False
fastapi_14186,fastapi,d_undef|v_high,True,False,False,False,False,none,0,False,False,False,False
fastapi_14246,fastapi,d_undef|v_low,False,True,False,False,False,7,0,True,False,False,False
fastapi_14258,fastapi,d_undef|v_high,False,True,False,False,True,0,0,True,False,False,False
fastapi_14262,fastapi,d_undef|v_high,True,False,True,True,False,none,14,True,False,False,False
fastapi_14266,fastapi,d_undef|v_low,False,True,False,False,False,none,0,True,False,False,False
fastapi_14297,fastapi,d_undef|v_low,False,True,False,False,True,0,9,True,False,False,False
fastapi_14301,fastapi,d0|v_high,False,True,False,False,False,2,9,True,False,False,False
fastapi_14303,fastapi,d_undef|v_low,False,True,True,True,False,none,15,True,False,False,False
fastapi_14306,fastapi,d2+|v_high,True,True,True,True,True,0,5,True,False,False,False
fastapi_14349,fastapi,d_undef|v_high,False,True,False,False,False,none,0,True,False,False,False
fastapi_14356,fastapi,d2+|v_low,False,True,False,False,True,0,11,True,False,False,False
fastapi_14360,fastapi,d2+|v_low,False,True,False,False,True,0,5,True,False,False,False
fastapi_14361,fastapi,d_undef|v_high,False,True,False,False,False,none,0,True,False,False,False
fastapi_14371,fastapi,d_undef|v_low,True,False,True,True,True,0,0,False,False,False,False
fastapi_14372,fastapi,d_undef|v_low,True,True,False,False,False,none,2,True,False,False,False
fastapi_14419,fastapi,d_undef|v_high,True,True,True,True,False,2,2,True,False,False,False
fastapi_14430,fastapi,d_undef|v_low,False,True,False,False,True,0,0,True,False,False,False
fastapi_14448,fastapi,d2+|v_low,True,False,True,True,True,0,2,False,False,False,False
fastapi_14455,fastapi,d_undef|v_low,False,True,False,False,False,3,0,True,False,False,False
fastapi_14458,fastapi,d0|v_high,False,False,False,False,True,0,5,False,False,False,False
fastapi_14459,fastapi,d_undef|v_low,True,True,False,False,False,3,0,True,False,False,False
fastapi_14463,fastapi,d1|v_high,False,True,False,False,False,2,17,True,False,False,False
fastapi_14479,fastapi,d0|v_high,False,False,False,False,False,2,7,False,False,False,False
fastapi_14482,fastapi,d_undef|v_low,False,True,False,False,False,5,0,True,False,False,False
fastapi_14485,fastapi,d_undef|v_low,False,True,False,False,False,3,0,True,False,False,False
fastapi_14487,fastapi,d_undef|v_low,False,False,True,True,False,none,0,False,False,False,False
fastapi_14492,fastapi,d_undef|v_high,False,False,False,False,False,none,1,False,False,False,False
fastapi_14512,fastapi,d2+|v_low,False,True,False,False,False,2,2,True,False,False,False
fastapi_14583,fastapi,d_undef|v_low,True,True,False,False,False,1,0,True,False,False,False
fastapi_14605,fastapi,d_undef|v_high,True,False,False,False,False,none,0,False,False,False,False
fastapi_14609,fastapi,d_undef|v_high,True,False,True,True,False,none,0,True,False,False,False
fastapi_14616,fastapi,d2+|v_high,False,True,False,False,True,0,5,True,False,False,False
fastapi_14786,fastapi,d0|v_low,False,True,False,False,True,0,13,True,False,False,False
fastapi_14791,fastapi,d_undef|v_high,False,False,False,False,False,none,4,False,False,False,False
fastapi_14794,fastapi,d0|v_high,False,True,False,False,False,2,4,True,False,False,False
fastapi_14851,fastapi,d2+|v_high,False,False,True,True,False,none,2,True,False,False,False
fastapi_14873,fastapi,d0|v_high,False,True,False,False,False,2,7,True,False,False,False
fastapi_14953,fastapi,d_undef|v_low,True,True,False,False,False,none,5,True,False,False,False
fastapi_14962,fastapi,d_undef|v_high,True,True,True,True,False,none,1,True,False,False,False
fastapi_14964,fastapi,d0|v_high,False,True,False,False,False,none,3,True,False,False,False
fastapi_14978,fastapi,d_undef|v_low,True,False,True,True,False,none,0,True,False,False,False
fastapi_14986,fastapi,d_undef|v_high,True,True,True,False,True,0,1,True,False,False,False
fastapi_15023,fastapi,d_undef|v_low,False,False,True,True,False,none,1,False,False,False,False
fastapi_15030,fastapi,d_undef|v_high,True,False,True,True,False,none,0,True,False,False,False
fastapi_15280,fastapi,d_undef|v_low,True,True,False,False,True,0,0,True,False,False,False
fastapi_15588,fastapi,d_undef|v_high,False,True,False,False,True,0,3,True,False,False,False
fastapi_15589,fastapi,d_undef|v_low,False,True,False,False,True,0,1,True,False,False,False
fastapi_15661,fastapi,d_undef|v_undef,False,False,False,False,False,none,1,True,False,True,True
fastapi_15745,fastapi,d_undef|v_high,True,False,True,True,False,none,8,False,False,False,False
fastapi_15763,fastapi,d_undef|v_high,False,True,False,False,False,1,1,False,False,False,False
fastapi_15785,fastapi,d_undef|v_high,True,False,False,False,False,none,2,False,False,False,False
fastapi_15800,fastapi,d2+|v_high,True,True,True,True,True,0,2,True,False,False,False
fastapi_5077,fastapi,d1|v_high,False,True,False,False,False,3,11,True,False,False,False
fastapi_5624,fastapi,d_undef|v_low,False,True,False,False,False,none,0,True,False,False,False
fastapi_9425,fastapi,d_undef|v_low,False,True,False,False,False,3,1,True,False,False,False
fastapi_9555,fastapi,d2+|v_low,False,True,False,False,True,0,14,True,False,False,False
fastapi_9753,fastapi,d_undef|v_low,False,True,False,False,False,none,2,True,False,False,False
httpx_3672,httpx,d_undef|v_high,True,False,True,True,True,0,1,False,False,False,False
requests_6589,requests,d0|v_high,False,False,False,False,,none,1,,False,False,False
requests_6592,requests,d_undef|v_high,False,False,False,False,,none,0,,False,False,False
requests_6629,requests,d0|v_low,False,False,False,False,,none,1,,False,False,False
requests_6644,requests,d_undef|v_high,False,False,False,False,,none,0,,False,False,False
requests_6757,requests,d_undef|v_high,True,False,False,False,False,none,0,False,False,False,False
requests_7205,requests,d2+|v_high,False,True,False,False,True,0,2,True,False,False,False
requests_7309,requests,d0|v_high,False,False,False,False,True,0,1,False,True,True,False
requests_7315,requests,d0|v_high,False,True,False,False,True,0,4,True,True,True,False
requests_7328,requests,d2+|v_high,False,True,False,False,False,4,2,True,False,False,False
requests_7427,requests,d2+|v_high,False,True,False,False,True,0,1,True,True,True,False
requests_7433,requests,d0|v_high,False,True,False,False,False,3,5,True,False,False,False
requests_7502,requests,d0|v_high,False,True,False,False,False,6,2,True,False,False,False
requests_7505,requests,d_undef|v_high,True,True,False,False,False,3,4,True,False,False,False
rich_2725,rich,d1|v_high,False,True,False,False,False,none,1,True,False,True,True
rich_2943,rich,d_undef|v_high,False,True,False,False,False,none,0,True,False,True,True
rich_3006,rich,d_undef|v_low,False,False,False,False,False,none,2,True,False,True,True
rich_3043,rich,d_undef|v_low,False,False,False,False,False,none,0,False,False,True,True
rich_3052,rich,d2+|v_high,False,True,False,False,True,0,2,True,False,True,True
rich_3061,rich,d2+|v_high,False,True,False,False,True,0,1,True,False,True,True
rich_3063,rich,d_undef|v_low,False,True,False,False,True,0,0,True,False,True,True
rich_3064,rich,d_undef|v_low,False,True,False,False,False,3,0,False,False,False,False
rich_3067,rich,d_undef|v_low,False,False,False,False,False,none,1,False,False,True,True
rich_3105,rich,d_undef|v_low,False,False,False,False,False,none,0,False,False,True,True
rich_3130,rich,d2+|v_high,False,True,False,False,False,3,3,False,False,False,False
rich_3180,rich,d_undef|v_low,True,True,False,False,True,0,0,True,False,True,True
rich_3278,rich,d_undef|v_high,False,True,False,False,False,none,0,True,False,True,True
rich_3296,rich,d2+|v_high,False,True,False,False,False,1,1,False,False,False,False
rich_3454,rich,d0|v_high,False,False,False,False,False,none,3,False,False,True,True
rich_3468,rich,d_undef|v_low,False,True,False,False,False,none,0,True,True,True,False
rich_3469,rich,d_undef|v_low,False,True,False,False,False,1,0,False,False,False,False
rich_3470,rich,d_undef|v_high,False,True,False,False,False,1,0,True,False,True,True
rich_3471,rich,d0|v_low,False,True,False,False,True,0,1,True,False,True,True
rich_3472,rich,d_undef|v_low,False,True,False,False,False,4,0,False,False,False,False
rich_3480,rich,d_undef|v_low,False,True,False,False,True,0,0,True,False,True,True
rich_3486,rich,d_undef|v_high,True,True,False,False,False,none,1,True,False,False,False
rich_3506,rich,d0|v_high,False,True,False,False,False,2,1,True,False,True,True
rich_3518,rich,d0|v_low,False,True,False,False,True,0,2,True,False,True,True
rich_3521,rich,d2+|v_high,False,True,False,False,False,2,1,True,False,True,True
rich_3535,rich,d_undef|v_high,True,True,False,False,False,none,0,True,False,True,True
rich_3675,rich,d_undef|v_low,True,True,False,False,False,1,0,True,False,True,True
rich_3676,rich,d_undef|v_high,True,True,False,False,False,none,0,True,False,True,True
rich_3718,rich,d1|v_low,False,True,False,False,False,none,2,True,False,True,True
rich_3772,rich,d_undef|v_low,False,True,False,False,False,none,0,True,False,True,True
rich_3777,rich,d2+|v_high,True,True,False,False,False,1,1,True,False,True,True
rich_3782,rich,d_undef|v_high,False,True,False,False,True,0,0,False,False,False,False
rich_3882,rich,d0|v_high,False,True,False,False,False,2,2,True,False,True,True
rich_3894,rich,d2+|v_high,False,True,False,False,False,2,6,True,False,True,True
rich_3905,rich,d1|v_high,False,True,False,False,False,1,6,True,False,True,True
rich_3930,rich,d_undef|v_high,True,False,False,False,False,none,0,True,False,True,True
rich_3934,rich,d_undef|v_low,True,True,False,False,False,1,0,True,False,True,True
rich_3935,rich,d_undef|v_low,False,True,False,False,False,2,0,True,False,True,True
rich_3938,rich,d_undef|v_high,True,True,False,False,True,0,0,True,False,True,True
rich_3942,rich,d_undef|v_high,True,False,False,False,False,3,0,False,False,True,True
rich_3944,rich,d_undef|v_low,False,True,False,False,True,0,0,True,False,True,True
rich_3953,rich,d_undef|v_low,False,True,False,False,True,0,0,True,False,True,True
rich_4006,rich,d0|v_high,False,True,False,False,True,0,1,True,False,True,True
rich_4070,rich,d2+|v_high,True,False,False,False,True,0,31,False,False,False,False
rich_4075,rich,d_undef|v_high,False,True,False,False,True,0,0,True,False,True,True
rich_4076,rich,d_undef|v_low,False,True,False,False,True,0,0,True,False,True,True
rich_4077,rich,d_undef|v_high,False,True,False,False,True,0,0,True,False,True,True
rich_4079,rich,d_undef|v_high,False,False,False,False,False,none,0,False,False,False,False

In [ ]:
# STEP 0/7 - setup: oracle pack and scorecard import
ORACLE_PACK = found["oracle_pack"][0] if found["oracle_pack"] else None
if ORACLE_PACK is not None and ORACLE_PACK.suffix == ".json":   # unpacked by Kaggle: scorecard.py reads the .gz
    _gz = GC / "oracle_pack.json.gz"
    with open(ORACLE_PACK, "rb") as _src, gzip.open(_gz, "wb") as _dst:
        shutil.copyfileobj(_src, _dst)
    plog(f"oracle pack found unpacked at {ORACLE_PACK}; re-packed to {_gz}")
    ORACLE_PACK = _gz
if not ORACLE_PACK:   # required (Oct 13): without it the scorecard is silently skipped after hours of GPU time
    plog("!!! STOP: oracle pack not attached")
    raise SystemExit("STOP: the oracle pack (oracle_pack.json.gz) is not attached. Add Input -> Your Work / Datasets -> "
                     "graphcomp-planb, then Save & Run All again. Nothing has run on the GPU yet.")
pk = json.load(gzip.open(ORACLE_PACK))
_pack_missing = sorted({json.loads(l)["instance_id"] for l in open(TASKS_JSONL) if l.strip()} - set(pk["tasks"]))
if _pack_missing:
    plog(f"!!! STOP: oracle pack lacks {len(_pack_missing)} tasks")
    raise SystemExit(f"STOP: oracle pack {pk['version']} lacks {len(_pack_missing)} competition tasks (e.g. {_pack_missing[:5]}); "
                     "attach the current graphcomp-planb dataset.")
plog(f"oracle pack ok: {ORACLE_PACK} | {pk['version']} | {len(pk['tasks'])} tasks | {len(pk['graphs'])} graphs")
sys.path.insert(0, str(GC))
import scorecard as sc
print("scorecard", sc.SCORECARD_VERSION)

In [ ]:
# STEP 0/7 - setup: write the arm's submission
# ============================== submission for this arm ==============================
def write_submission(dest, tools, seed):
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    (dest / "prompts").mkdir(parents=True)
    (dest / "configs").mkdir()
    prompt = (Path(SAMPLE_SUB) / "prompts/system.md").read_text()
    (dest / "prompts/system.md").write_text(prompt + (PROMPT_ADDENDUM or ""))
    (dest / "agent.yaml").write_text(
        "name: swe_planb_agent\n"
        f"model: {MODEL_SLUG}\n"
        "instruction: !include prompts/system.md\n"
        "tools:\n" + "".join(f"  - {t}\n" for t in tools) +
        "generate_content_config: !include configs/sampling.yaml\n")
    (dest / "configs/sampling.yaml").write_text(
        f"temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: {MAX_OUTPUT_TOKENS}\n"
        f"seed: {seed}\n"
        f"thinking_config:\n  thinking_budget: {THINKING_BUDGET}\n  include_thoughts: true\n")
    (dest / "eval_config.yaml").write_text(
        "evaluation:\n"
        f"  timeout_seconds: {BUDGET['timeout_seconds']}\n  max_tool_calls: {BUDGET['max_tool_calls']}\n"
        f"  max_time_minutes: {BUDGET['max_time_minutes']}\n  max_turns: {BUDGET['max_turns']}\n")
    return dest

BASE_TOOLS = ["run_command", "read_file", "edit_file", "write_file", "get_status", "submit_patch"]
GRAPH_TOOLS = ["get_code_neighbors", "search_similar_code", "get_code_subgraph"]
TOOLS = {"B2": BASE_TOOLS + GRAPH_TOOLS, "B3": BASE_TOOLS}[ARM]
SUB = write_submission(OUT / f"submissions/{RUN_NAME}", TOOLS, SEED)
shutil.copy(SUB / "prompts/system.md", OUT / "provenance/system_prompt.md")
print((SUB / "agent.yaml").read_text())
print((SUB / "configs/sampling.yaml").read_text())
plog(f"submission {RUN_NAME}: tools={TOOLS} max_output_tokens={MAX_OUTPUT_TOKENS} thinking_budget={THINKING_BUDGET} "
     f"prompt addendum={'none' if not PROMPT_ADDENDUM else repr(PROMPT_ADDENDUM[:120]) + '...'}")
# B4: write the briefs for run_arm.py --briefs and scorecard.py --brief-dir; stop before serving if a problem statement
# here differs from the one the brief was built from (the brief would then describe another issue text)
if BRIEFS:
    import hashlib
    _bad = [i for i, b in BRIEFS.items() if hashlib.sha256(tasks_by_id[i]['problem_statement'].encode()).hexdigest() != b['problem_sha256']]
    if _bad:
        raise SystemExit(f'STOP: problem statements differ from the briefs\' source for {_bad}. Nothing has run on the GPU yet.')
    (GC / 'briefs.json').write_text(json.dumps(BRIEFS))
    (OUT / 'provenance/briefs').mkdir(parents=True, exist_ok=True)
    for _i, _b in BRIEFS.items():
        (OUT / 'provenance/briefs' / f'{_i}.md').write_text(_b['brief'])
    plog(f'B4 briefs: {len(BRIEFS)} tasks, problem statements match; written to {GC / "briefs.json"}')


In [ ]:
# STEP 1 - script folders
(GC / 'lora').mkdir(exist_ok=True)


In [ ]:
%%writefile /kaggle/working/gc/lora/build_rft_data.py
#!/usr/bin/env python3
"""Build LoRA training data (RFT / distillation) from swegemma ATIF traces (Plan B/C, Oct 14).

    python build_rft_data.py --runs RUN_DIR [RUN_DIR ...] --profile PROFILE.csv --out OUT_DIR
                             [--max-per-task 2] [--require-graph-call] [--max-est-tokens 16000] [--max-repeat-calls N] [--dedupe-routes 0.8]
                             [--accurate-tools TASKS.jsonl] [--order time] [--exclude IDS.txt]

RUN_DIR = a merged run folder (task_results.jsonl + traces/). PROFILE = the validity profile of the environment the runs
came from (sub_valid). A trajectory is kept only if: resolved; sub_valid; NOT in the official dev or holdout slice
(scripts/planB/official_slices; checked again at the end, the script refuses to write otherwise); no infra error, no
context-exceeded / error ending; every agent step has at most one tool call with its result. Up to --max-per-task
trajectories per task (fewest tool calls first), so easy tasks solved by many seeds do not dominate.

Output (OUT_DIR): train.jsonl, one trajectory per line:
  {"instance_id", "repo", "source", "n_turns", "est_tokens", "n_graph_calls",
   "messages": [system, user, (assistant{reasoning_content, content, tool_calls}, tool{tool_call_id, name, content})..., final assistant]}
The agent step's text is stored as reasoning_content (ATIF event_type "thinking"); the final step's text as content. The trainer
renders these with the model's own chat template and puts the loss on assistant turns only. Also: manifest.json (selection,
exclusions with reasons, per-repo counts, sources) and excluded.csv.
"""
import argparse, csv, difflib, hashlib, json, re, sys
from collections import Counter
from pathlib import Path

HERE = Path(__file__).resolve().parent
SLICES = HERE.parent / "official_slices"
GRAPH_TOOLS = {"get_code_neighbors", "search_similar_code", "get_code_subgraph"}


def read_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]


def longest_repeat(msgs):
    """Longest run of consecutive identical tool calls (same name and arguments): the agent spinning."""
    calls = [json.dumps(m["tool_calls"][0]["function"], sort_keys=True) for m in msgs if m.get("tool_calls")]
    best = run = 0
    for i, c in enumerate(calls):
        run = run + 1 if i and c == calls[i - 1] else 1
        best = max(best, run)
    return best


def tool_accuracy(trace, patch):
    """Accurate tool use, judged against the gold patch with tool_trace.py (TQS arXiv 2610.13152, TRACEPROBE 2607.06184):
    every source edit lands on a gold file or function, and a reproduction or test runs after the last source edit.
    Returns (ok, signals)."""
    sys.path.insert(0, str(HERE.parent))
    import tool_trace
    rows, summ = tool_trace.trace_run(trace, tool_trace.gold_of(patch))
    edits = [r for r in rows if r["purpose"] == "edit-src"]
    last = max((r["i"] for r in edits), default=None)
    verified = last is not None and any(r["purpose"] in ("reproduce", "test") and r["i"] > last for r in rows)
    sig = {"edit_src_calls": len(edits), "edits_on_gold": sum(r["on_gold"] for r in edits), "verified_after_last_edit": verified,
           "edit_then_reread": sum(1 for a, b in zip(rows, rows[1:]) if a["purpose"] == "edit-src" and b["purpose"] == "read-src"),
           "first_gold_call": summ["first_gold_call"]}
    ok = bool(edits) and sig["edits_on_gold"] == len(edits) and verified
    return ok, sig


def route(msgs):
    """Route fingerprint: the sequence of (tool, main target) over the trajectory's tool calls. Target = file path for file
    tools, node or query for graph tools, the first word of the command for run_command. Two resolved trajectories with
    the same route took the same path through the repository; keeping both adds a near-copy rather than a second way to
    solve the task (route diversity: arXiv 2609.33780, 2609.31571)."""
    out = []
    for m in msgs:
        for tc in m.get("tool_calls") or []:
            f, a = tc["function"]["name"], tc["function"].get("arguments") or {}
            tgt = a.get("filepath") or a.get("node") or a.get("query") or (str(a.get("command", "")).split() or [""])[0]
            out.append((f, str(tgt).lstrip("./")))
    return tuple(out)


def route_similarity(m1, m2):
    return difflib.SequenceMatcher(None, route(m1), route(m2)).ratio()


RETRY_NOTE = "## Notes from earlier attempts"     # notebook 20's retry note, prepended to the problem statement
NOTE_ECHO = re.compile(r"(earlier|previous|prior) attempt", re.I)


def strip_retry_note(text):
    """The retry note is sampling-time guidance only: training sees the problem statement as deployed."""
    if isinstance(text, str) and text.startswith(RETRY_NOTE) and "\n## Issue\n" in text:
        return text.split("\n## Issue\n", 1)[1], True
    return text, False


def trace_to_messages(trace):
    steps = trace["steps"]
    msgs, n_graph, problems = [], 0, []
    sys_ = [s for s in steps if s.get("source") == "system"]
    usr = [s for s in steps if s.get("source") == "user"]
    if not sys_ or not usr:
        return None, 0, ["no system or user step"]
    ts = lambda s: (s.get("extra") or {}).get("timestamp")     # = the ADK event's timestamp; compactions are keyed by it
    msgs.append({"role": "system", "content": sys_[0]["message"]})
    user, noted = strip_retry_note(usr[0]["message"])
    msgs.append({"role": "user", "content": user, "_ts": ts(usr[0])})
    agent = [s for s in steps if s.get("source") == "agent"]
    k, pending = 0, None
    for s in agent:
        tcs = s.get("tool_calls") or []
        ev = (s.get("extra") or {}).get("event_type")
        if not tcs:
            if ev == "final" or s is agent[-1]:
                # the text-only "thinking" step just before "final" is the reasoning of the final answer
                m = {"role": "assistant", "content": s.get("message", ""), "_ts": ts(s),
                     "_prompt_tokens": (pending or s).get("metrics", {}).get("prompt_tokens") if (pending or s).get("metrics") else None}
                if pending is not None and pending.get("message") and pending.get("message") != s.get("message"):
                    m["reasoning_content"] = pending["message"]
                msgs.append(m)
            else:
                pending = s
            continue
        pending = None
        if len(tcs) > 1:
            problems.append(f"step {s.get('step_id')}: {len(tcs)} tool calls (only the first result is stored)")
        obs = s.get("observation")
        if not isinstance(obs, dict) or "content" not in obs:
            problems.append(f"step {s.get('step_id')}: no tool result")
        calls = []
        for tc in tcs:
            k += 1
            cid = tc.get("tool_call_id") or f"call_{k}"
            name = tc.get("function_name")
            n_graph += name in GRAPH_TOOLS
            calls.append({"id": cid, "type": "function",
                          # a mapping, not a JSON string: the Gemma 4 chat template raises on string arguments
                          "function": {"name": name, "arguments": tc.get("arguments") or {}}})
        msgs.append({"role": "assistant", "content": "", "reasoning_content": s.get("message", ""), "tool_calls": calls, "_ts": ts(s),
                     # recorded prompt size of this LLM call: train_lora.py --check compares its renderings against it
                     "_prompt_tokens": (s.get("metrics") or {}).get("prompt_tokens")})
        msgs.append({"role": "tool", "tool_call_id": calls[0]["id"], "name": calls[0]["function"]["name"],
                     "content": (obs or {}).get("content", ""), "_ts": ts(s)})
    if msgs[-1]["role"] != "assistant" or msgs[-1].get("tool_calls"):
        problems.append("does not end with a final assistant message")
    if noted and any(NOTE_ECHO.search(str(m.get("reasoning_content", "")) + str(m.get("content", ""))) for m in msgs if m["role"] == "assistant"):
        problems.append("reasoning refers to the retry note (earlier attempts), which training does not show")
    return msgs, n_graph, problems


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--runs", nargs="+", required=True)
    ap.add_argument("--profile", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--max-per-task", type=int, default=2)
    ap.add_argument("--require-graph-call", action="store_true", help="adoption data: keep only trajectories with a graph call")
    ap.add_argument("--max-repeat-calls", type=int, help="drop trajectories with more than N consecutive identical tool calls "
                    "(spinning; off by default)")
    ap.add_argument("--dedupe-routes", type=float, metavar="T",
                    help="per task, drop a trajectory whose route similarity (difflib ratio over (tool, target) calls) to a "
                         "kept one is >= T (1.0 = exact copies only). On 54 same-task pairs from the Kaggle runs (Oct 15) "
                         "the median similarity was 0.45 and one pair reached 0.80")
    ap.add_argument("--accurate-tools", metavar="TASKS_JSONL",
                    help="keep only trajectories with accurate tool use (see tool_accuracy); TASKS_JSONL supplies the gold patches")
    ap.add_argument("--order", choices=["tool_calls", "time"], default="tool_calls",
                    help="which trajectories of a task are kept first: fewest tool calls, or least wall time (duration_seconds)")
    ap.add_argument("--exclude", metavar="IDS_TXT", help="never train on these instance ids (one per line), e.g. extra holdout tasks")
    ap.add_argument("--max-est-tokens", type=int, default=16384, help="drop trajectories whose longest context is above this (16384 = the effective input budget at max_output_tokens 16384)")
    a = ap.parse_args()

    dev = {l.strip() for l in open(SLICES / "dev_slice.txt") if l.strip()}
    hold = {l.strip() for l in open(SLICES / "holdout_slice.txt") if l.strip()}
    prof = {r["instance_id"]: r for r in csv.DictReader(open(a.profile))}
    never = {l.strip() for l in open(a.exclude) if l.strip()} if a.exclude else set()
    gold = {t["instance_id"]: t.get("patch", "") for t in map(json.loads, open(a.accurate_tools))} if a.accurate_tools else {}
    cands, excluded = {}, []
    for rd in map(Path, a.runs):
        comps = read_jsonl(rd / "compactions.jsonl") if (rd / "compactions.jsonl").exists() else []   # run_arm.py logs these (runs before Oct 15 have none)
        for r in read_jsonl(rd / "task_results.jsonl"):
            iid = r["instance_id"]
            why = None
            if iid in dev or iid in hold:
                why = "in an official slice (dev/holdout): never trained on"
            elif iid in never:
                why = "in --exclude: never trained on"
            elif not r.get("resolved"):
                why = "not resolved"
            elif prof.get(iid, {}).get("sub_valid") != "True":
                why = f"not valid here ({prof.get(iid, {}).get('sub_reason')})"
            elif r.get("_infra_error") or r.get("error"):
                why = f"error: {str(r.get('error'))[:60]}"
            tp = rd / "traces" / f"trace_{iid}.json"
            if why is None and not tp.exists():
                why = "trace missing"
            if why:
                excluded.append((str(rd), iid, why))
                continue
            trace = json.load(open(tp))
            msgs, n_graph, problems = trace_to_messages(trace)
            if msgs is None or problems:
                excluded.append((str(rd), iid, "; ".join(problems)[:200]))
                continue
            pt = [((s.get("metrics") or {}).get("prompt_tokens") or 0) for s in trace["steps"] if s.get("source") == "agent"]
            est = max(pt) if pt else 0
            if est > a.max_est_tokens:
                excluded.append((str(rd), iid, f"longest context {est} > {a.max_est_tokens}"))
                continue
            if a.max_repeat_calls and (rep_ := longest_repeat(msgs)) > a.max_repeat_calls:
                excluded.append((str(rd), iid, f"spinning: {rep_} identical tool calls in a row > {a.max_repeat_calls}"))
                continue
            if a.require_graph_call and n_graph == 0:
                excluded.append((str(rd), iid, "no graph call (--require-graph-call)"))
                continue
            sig = {}
            if a.accurate_tools:
                ok, sig = tool_accuracy(trace, gold.get(iid, ""))
                if not ok:
                    excluded.append((str(rd), iid, f"inaccurate tool use: {sig}"))
                    continue
            n_turns = sum(1 for m in msgs if m["role"] == "assistant")
            ids = {c["id"] for m in msgs for c in m.get("tool_calls") or []}
            mine = sorted((c for c in comps if ids & set(c.get("call_ids") or [])), key=lambda c: c["created"])
            cands.setdefault(iid, []).append({"instance_id": iid, "repo": iid.split("_")[0], "source": str(rd),
                                              "n_turns": n_turns, "est_tokens": est, "n_graph_calls": n_graph,
                                              "tool_calls": int(r.get("tool_calls") or 0),
                                              "seconds": float(r.get("duration_seconds") or 0), "tool_accuracy": sig,
                                              "compactions": mine, "messages": msgs})
    rows = []
    for iid, lst in sorted(cands.items()):
        lst.sort(key=lambda x: ((x["seconds"], x["tool_calls"]) if a.order == "time" else (x["tool_calls"], x["seconds"]), x["source"]))
        if a.dedupe_routes is not None:      # in --order, so the fastest / shortest of near-copies is kept
            kept = []
            for x in lst:
                sim = max((route_similarity(x["messages"], k["messages"]) for k in kept), default=0.0)
                if sim >= a.dedupe_routes:
                    excluded.append((x["source"], iid, f"near-duplicate route (similarity {sim:.2f} >= {a.dedupe_routes})"))
                else:
                    kept.append(x)
            lst = kept
        for x in lst[a.max_per_task:]:
            excluded.append((x["source"], iid, f"over --max-per-task {a.max_per_task}"))
        rows += lst[:a.max_per_task]
    leak = sorted({x["instance_id"] for x in rows} & (dev | hold | never))
    if leak:
        sys.exit(f"REFUSING to write: dev/holdout tasks in the training set: {leak}")
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "train.jsonl", "w") as f:
        for x in rows:
            x = {k: v for k, v in x.items() if k not in ("tool_calls",)}
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    with open(out / "excluded.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["source", "instance_id", "reason"])
        w.writerows(excluded)
    reasons = Counter(r for _, _, r in excluded)
    man = {"n_trajectories": len(rows), "n_tasks": len({x["instance_id"] for x in rows}),
           "by_repo": dict(Counter(x["repo"] for x in rows)),
           "assistant_turns": sum(x["n_turns"] for x in rows), "graph_calls": sum(x["n_graph_calls"] for x in rows),
           "est_tokens_max_context": {"median": sorted(x["est_tokens"] for x in rows)[len(rows) // 2] if rows else None,
                                      "max": max((x["est_tokens"] for x in rows), default=None)},
           "excluded_reasons": dict(reasons.most_common()), "runs": a.runs, "profile": a.profile,
           "rules": {"max_per_task": a.max_per_task, "require_graph_call": a.require_graph_call, "max_est_tokens": a.max_est_tokens,
                     "max_repeat_calls": a.max_repeat_calls, "dedupe_routes": a.dedupe_routes,
                     "accurate_tools": bool(a.accurate_tools), "order": a.order, "exclude": sorted(never),
                     "never": "official dev and holdout slices"},
           "train_sha256": hashlib.sha256((out / "train.jsonl").read_bytes()).hexdigest()[:16]}
    (out / "manifest.json").write_text(json.dumps(man, indent=1))
    print(json.dumps({k: man[k] for k in ("n_trajectories", "n_tasks", "by_repo", "assistant_turns", "graph_calls",
                                          "est_tokens_max_context", "excluded_reasons")}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/gc/lora/train_lora.py
#!/usr/bin/env python3
"""Train a LoRA on RFT / distillation trajectories (build_rft_data.py output). Plan C (Oct 14; v2 Oct 15).

    # CPU check: renders every sample with the model's chat template and the harness's own tool schemas, verifies that each
    # target turn continues its prompt, and reports token counts.
    python train_lora.py --data train.jsonl --base <unquantized QAT dir> --out OUT --check
    # E4B, every GPU data-parallel (one QLoRA copy per GPU):
    torchrun --nproc_per_node 4 train_lora.py --data ... --base <e4b dir> --out OUT
    # 31B, one process, the 4-bit model split over every GPU (layer by layer):
    python train_lora.py --data ... --base <31b dir> --out OUT --device-map auto --served-model google/gemma-4-31b-it-qat-w4a16-ct

Base: the UNQUANTIZED QAT checkpoint the served w4a16 model was made from; vLLM serves the adapter on w4a16.

Samples (v2): a trajectory is cut into WINDOWS of consecutive assistant turns whose target tokens total at most --max-target.
A window is one sequence = the conversation rendered up to its last turn; the loss is on the window's assistant turns only,
every earlier turn being context. This is the usual multi-turn SFT objective (loss on assistant tokens, full context) and costs
one forward per window instead of one per turn. A window is used only if every turn in it is an exact substring continuation of
the rendering (checked turn by turn, as v1 did); otherwise its turns fall back to one sample per turn.
Practices built in (sources in the notebook that runs this, section "Training choices"):
- assistant-only loss, full multi-turn context; train/val split by task;
- step-level error masking: an assistant turn whose tool result is an error (except run_command, whose failures are test or
  reproduction feedback) stays in the context but gets no loss (--no-error-mask turns it off);
- train = deploy: --system-prompt replaces the recorded system prompt and --deploy-budget rewrites the harness's budget lines in
  the task prompt, so the adapter is trained on exactly the text it will be served with;
- vanilla LoRA on every language-model projection, rank 32 / alpha 64, lr 1e-4, cosine with warmup, at most 2 epochs;
- 31B (attention_k_eq_v): no LoRA on k_proj of the full-attention layers (vLLM would not apply the delta to V there);
- --time-budget-h: after the first optimizer step the run shrinks its total step count to what fits, so the LR schedule
  still anneals to 0 and the adapter is always saved.
Several GPUs: torchrun -> data-parallel (ranks load one after another; LoRA gradients averaged before each step); rank 0 logs
and saves. More ranks than GPUs (smoke test only) share GPUs over gloo.
"""
import argparse, json, math, os, random, re, struct, sys, time
WORLD, RANK, LOCAL = (int(os.environ.get(k, d)) for k, d in (("WORLD_SIZE", "1"), ("RANK", "0"), ("LOCAL_RANK", "0")))
from pathlib import Path

B2_TOOLS = ["run_command", "read_file", "edit_file", "write_file", "get_status", "submit_patch",
            "get_code_neighbors", "search_similar_code", "get_code_subgraph"]


def harness_tool_schemas(names):
    """The tool list exactly as the harness sends it: swegemma tools -> ADK FunctionTool declarations -> litellm params."""
    from swegemma.tools import create_tools
    from google.adk.tools.function_tool import FunctionTool
    from google.adk.models.lite_llm import _function_declaration_to_tool_param

    class _Ctx:                          # the tool factories only bind ctx; nothing is executed here
        def __getattr__(self, k):
            return None
    tools = create_tools(_Ctx())
    return [_function_declaration_to_tool_param(FunctionTool(tools[n])._get_declaration()) for n in names]


PAST_REASONING = "keep"       # set by --past-reasoning; "drop" removes reasoning from all but the target turn
# ADK's identity processor appends this to the agent instruction on every request (google/adk/flows/llm_flows/identity.py)
SYSTEM_SUFFIX = '\n\nYou are an agent. Your internal name is "swe_planb_agent".'
SYSTEM_PROMPT = None          # --system-prompt
DEPLOY_BUDGET = None          # --deploy-budget
# the harness's budget lines in the task prompt (swegemma/harness/agent_runner.py, build prompt)
BUDGET_RE = {"time_minutes": (r"- Time allowance: [\d.]+ minutes", "- Time allowance: {} minutes"),
             "tool_calls": (r"- Tool calls allowance: \d+ calls", "- Tool calls allowance: {} calls"),
             "turns": (r"- Max loop iterations: \d+ turns", "- Max loop iterations: {} turns"),
             "timeout_seconds": (r"- Single command timeout: \d+ seconds", "- Single command timeout: {} seconds")}


def clean(msgs, target_idx=None):
    out = []
    for i, m in enumerate(msgs):
        m = {k: v for k, v in m.items() if not k.startswith("_")}
        if i == 0 and m["role"] == "system":
            m["content"] = (SYSTEM_PROMPT if SYSTEM_PROMPT is not None else m["content"]) + SYSTEM_SUFFIX
        if m["role"] == "user" and DEPLOY_BUDGET and isinstance(m.get("content"), str):
            for k, v in DEPLOY_BUDGET.items():
                if k in BUDGET_RE:
                    m["content"] = re.sub(BUDGET_RE[k][0], BUDGET_RE[k][1].format(v), m["content"])
        if PAST_REASONING == "drop" and m["role"] == "assistant" and i != target_idx:
            m.pop("reasoning_content", None)
        out.append(m)
    return out


def render(tok, msgs, tools, gen, target_idx=None):
    return tok.apply_chat_template(clean(msgs, target_idx), tools=tools, tokenize=False, add_generation_prompt=gen,
                                   enable_thinking=True)


COMPACTION_TOL = 96   # tokens; set by --fidelity-tol
ERR_RE = re.compile(r'"status"\s*:\s*"error"')


def broken_safetensors(d):
    """Files whose safetensors header does not cover the file exactly. Kaggle's gemma-4-12b-it-qat-q4_0-unquantized v2 failed
    here on Oct 15 ("incomplete metadata, file not fully covered"); the Hugging Face copy is intact. [] = all good."""
    bad = []
    for f in sorted(Path(d).glob("*.safetensors")) if Path(d).is_dir() else []:
        try:
            with open(f, "rb") as fh:
                n = struct.unpack("<Q", fh.read(8))[0]
                hdr = json.loads(fh.read(n))
            end = max(v["data_offsets"][1] for k, v in hdr.items() if k != "__metadata__")
            if f.stat().st_size != 8 + n + end:
                bad.append(f"{f.name}: file {f.stat().st_size} bytes, header covers {8 + n + end}")
        except Exception as e:
            bad.append(f"{f.name}: unreadable header ({type(e).__name__}: {e})")
    return bad


def pick_base(spec):
    """--base may list several candidates (comma-separated, e.g. every attached version); the first intact one is used."""
    for c in [x.strip() for x in spec.split(",") if x.strip()]:
        bad = broken_safetensors(c)
        if not bad:
            return c
        print(f"[base] skipping {c}: broken checkpoint: {bad[:3]}", flush=True)
    sys.exit(f"no intact base checkpoint among {spec!r}: attach another version or the Hugging Face copy")


def turn_errored(msgs, i):
    """True if assistant turn i's tool result is an error from a tool other than run_command (SWE-Lego error masking)."""
    nxt = msgs[i + 1] if i + 1 < len(msgs) else None
    return bool(nxt and nxt["role"] == "tool" and nxt.get("name") != "run_command"
                and ERR_RE.search(str(nxt.get("content", ""))))


def compacted_context(msgs, i, comps):
    """The conversation the model was sent at assistant turn i, as ADK 1.36 builds it (contents._process_compaction_events):
    every compaction made before the turn, minus those subsumed by a wider one, replaces the events in [start, end] with its
    summary as a model turn placed at `end`. The system prompt is not an event and always stays."""
    t = msgs[i].get("_ts")
    live = [c for c in comps if t is not None and c["created"] <= t]
    live = [c for c in live if not any(o is not c and o["start"] <= c["start"] and o["end"] >= c["end"]
                                       and (o["start"] < c["start"] or o["end"] > c["end"]) for o in live)]
    if not live:
        return msgs[:i]
    head = [m for m in msgs[:i] if m["role"] == "system"]
    rest = [(m["_ts"], k, m) for k, m in enumerate(msgs[:i]) if m["role"] != "system"
            and not any(c["start"] <= m["_ts"] <= c["end"] for c in live)]
    rest += [(c["end"], i + j, {"role": "assistant", "content": c["summary"]}) for j, c in enumerate(live)]
    return head + [m for _, _, m in sorted(rest, key=lambda x: (x[0], x[1]))]


def turn_samples(tok, traj, tools, stats):
    """[(prompt_text, target_text, turn_index, masked)] per assistant turn.

    The trace records each LLM call's real prompt size. Our rendering matches it to a constant offset; when the harness
    compacts the context (ADK EventsCompactionConfig: older events replaced by a summary), the prompt is rebuilt from the
    summaries run_arm.py logs (traj["compactions"], attached by build_rft_data.py). Without them (runs before Oct 15), or if
    the rebuilt prompt still drifts from the recorded size, the turn and every later one are not used."""
    msgs, out = traj["messages"], []
    comps = traj.get("compactions") or []
    stats["compactions_applied"] += len(comps)
    offset = None
    for i, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        ctx = compacted_context(msgs, i, comps) if comps else msgs[:i]
        prompt = render(tok, ctx, tools, True)
        rec = m.get("_prompt_tokens")
        if rec:
            d = len(tok(prompt, add_special_tokens=False)["input_ids"]) - rec
            offset = d if offset is None else offset
            if abs(d - offset) > COMPACTION_TOL:
                stats["after_compaction"] += sum(1 for x in msgs[i:] if x["role"] == "assistant")
                stats["compacted_trajectories"] += 1
                break
        full = render(tok, ctx + [m], tools, False, target_idx=len(ctx))
        if rec:                                  # fidelity: our prompt vs what the model was actually sent at this turn
            stats["prompt_tok_pairs"].append((len(tok(prompt, add_special_tokens=False)["input_ids"]), rec))
        if not full.startswith(prompt):
            if not str(m.get("reasoning_content") or "").strip():
                # thinking on: the generation prompt opens the thought channel, but a turn with no reasoning text renders
                # straight to its tool call or answer (converted external steps, e.g. read_file continuation chunks, final
                # answers written without a thought). It cannot be a
                # target without inventing a thought; it stays context only (no loss).
                stats["no_reasoning_context_only"] = stats.get("no_reasoning_context_only", 0) + 1
                continue
            stats["prefix_mismatch"] += 1
            if stats["prefix_mismatch"] <= 2:
                k = next((j for j, (x, y) in enumerate(zip(prompt, full)) if x != y), min(len(prompt), len(full)))
                stats["mismatch_examples"].append({"turn": i, "prompt_tail": prompt[max(0, k - 200):k + 200],
                                                   "full_at_same_place": full[max(0, k - 200):k + 200]})
            continue
        masked = ERROR_MASK and turn_errored(msgs, i)
        stats["masked_turns"] += masked
        out.append((prompt, full[len(prompt):], i, masked))
    return out


ERROR_MASK = True


def ids_and_positions(tok, text, spans):
    """Token ids of `text` and the positions of tokens that start inside one of the character `spans`."""
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    pos = [t for t, (a, b) in enumerate(enc["offset_mapping"]) if b > a and any(s <= a < e for s, e in spans)]
    return enc["input_ids"], pos


def window_samples(tok, traj, tools, stats, max_target, max_len):
    """Pack consecutive turns into windows (see the module docstring); returns [(ids, target_positions)]."""
    turns = turn_samples(tok, traj, tools, stats)
    out, cur = [], []

    def flush(group):
        if not group:
            return
        last_p, last_y = group[-1][0], group[-1][1]
        text = last_p + last_y
        spans, ok = [], True
        for p, y, _, masked in group:
            if not text.startswith(p + y):      # the window's text must contain every earlier turn verbatim
                ok = False
                break
            if not masked:
                spans.append((len(p), len(p) + len(y)))
        if ok and len(group) > 1:
            ids, pos = ids_and_positions(tok, text, spans)
            if pos and len(ids) <= max_len:
                out.append((ids, pos))
                stats["windows"] += 1
            elif len(ids) > max_len:
                stats["too_long"] += 1
            return
        if not ok:
            stats["window_fallback"] += 1
        for p, y, _, masked in group:           # one sample per turn
            if masked:
                continue
            ids, pos = ids_and_positions(tok, p + y, [(len(p), len(p) + len(y))])
            if pos and len(ids) <= max_len:
                out.append((ids, pos))
            elif len(ids) > max_len:
                stats["too_long"] += 1

    tgt = 0
    for t in turns:
        n = len(tok(t[1], add_special_tokens=False)["input_ids"])
        if n > max_target:                       # one turn alone over the budget: drop it (keeps it as context in later windows)
            stats["target_too_long"] += 1
            flush(cur); cur, tgt = [], 0
            continue
        if cur and (tgt + n > max_target or not t[0].startswith(cur[-1][0] + cur[-1][1])):   # budget, or a compaction
            flush(cur); cur, tgt = [], 0
        cur.append(t); tgt += n
    flush(cur)
    return out


def custom_device_map(base, ngpu, last_layers):
    """Gemma 4 (31B) over ngpu GPUs, unbalanced on purpose: the last GPU holds the tied embeddings / output layer, the final
    norm, the logits of every window and the loss, so it gets only `last_layers` decoder layers; the others are spread evenly
    over the first ngpu-1 GPUs (session 2, Oct 19: with an even split the last GPU ran out of memory first, the others had
    ~16 GB free). Every module the model has is placed; check_device_map raises if one is missing."""
    from accelerate import init_empty_weights
    from accelerate.utils import check_device_map
    from transformers import AutoConfig, AutoModelForCausalLM
    cfg = AutoConfig.from_pretrained(base)
    n = cfg.get_text_config().num_hidden_layers
    with init_empty_weights():
        meta = AutoModelForCausalLM.from_config(cfg)
    lm = next(name for name, _ in meta.named_modules() if name.endswith("language_model") and hasattr(_, "layers"))
    last = ngpu - 1
    dm = {}
    if ngpu == 1:
        dm = {"": 0}
    else:
        last_layers = max(1, min(last_layers, n - (ngpu - 1)))
        first = n - last_layers                       # layers 0 .. first-1 over GPUs 0 .. ngpu-2
        for i in range(n):
            dm[f"{lm}.layers.{i}"] = last if i >= first else min(ngpu - 2, i * (ngpu - 1) // first)
        dm[f"{lm}.embed_tokens"] = last              # tied with lm_head: one device for both
        dm[f"{lm}.norm"] = last
        for name, mod in meta.named_children():       # lm_head and anything else at the top level
            if name != "model":
                dm[name] = last
        for name, mod in meta.model.named_children():  # vision tower etc.: not used by text training -> GPU 0
            if f"model.{name}" != lm:
                dm[f"model.{name}"] = 0
        for name, mod in getattr(meta.model, lm.split(".")[-1]).named_children():
            key = f"{lm}.{name}"
            if not any(k == key or k.startswith(key + ".") for k in dm):
                dm[key] = 0                           # e.g. rotary embeddings
    check_device_map(meta, dm)
    return dm


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True)
    ap.add_argument("--base", required=True, help="HF id or local dir of the unquantized QAT checkpoint; several, comma-separated, "
                    "= use the first whose safetensors are intact (broken_safetensors)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--check", action="store_true", help="render and count only (CPU)")
    ap.add_argument("--rank", type=int, default=32)
    ap.add_argument("--alpha", type=int, default=64)
    ap.add_argument("--dropout", type=float, default=0.05)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--epochs", type=float, default=2.0)
    ap.add_argument("--grad-accum", type=int, default=4, help="windows per optimizer step (over all ranks)")
    ap.add_argument("--max-len", type=int, default=16384 + 2048, help="tokens per sample; longer samples are dropped")
    ap.add_argument("--max-target", type=int, default=2500, help="target tokens per window: the logits of a T-token target take "
                    "T x 262k x 2 bytes (bf16) plus a float32 chunk; 2,500 fits beside E4B on one L4, 31B split over 4 GPUs takes 8,000")
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--quant", choices=["4bit", "none"], default="4bit")
    ap.add_argument("--device-map", choices=["single", "auto", "custom"], default="single",
                    help="auto: one process, layers spread over every visible GPU (31B); single: one GPU per process")
    ap.add_argument("--max-memory-gib", type=float, default=19.5, help="per GPU, --device-map auto")
    ap.add_argument("--last-gpu-layers", type=int, default=9, help="--device-map custom: decoder layers on the last GPU, which "
                    "also holds the embeddings / tied output layer and the logits (the rest are spread evenly)")
    ap.add_argument("--leak-gib", type=float, default=1.5, help="after an OOM recovery, more than this above the pre-loop "
                    "allocation on any GPU = memory not given back: save and exit 75 (the notebook restarts at a smaller window)")
    ap.add_argument("--time-budget-h", type=float, default=0, help="0 = no limit")
    ap.add_argument("--no-eval-start", action="store_true", help="skip the val loss before training (saves one val pass)")
    ap.add_argument("--no-error-mask", action="store_true")
    ap.add_argument("--init-adapter", help="resume: continue training this LoRA (a previous session's adapter dir)")
    ap.add_argument("--stop-at", type=float, default=0, help="unix time: stop at a step boundary before it, save, exit 0")
    ap.add_argument("--stop-after-s", type=float, default=0, help="like --stop-at, counted from the first training step")
    ap.add_argument("--save-every-min", type=float, default=30, help="checkpoint the adapter (atomically) this often")
    ap.add_argument("--keep-order", action="store_true", help="train in the data file's order (its priority), shuffled only "
                    "within blocks of --order-block windows, so a run cut short has trained the file's head")
    ap.add_argument("--order-block", type=int, default=32)
    ap.add_argument("--max-oom-events", type=int, default=50, help="in-process out-of-memory recoveries before giving up")
    ap.add_argument("--tail-n", type=int, default=0, help="the file's last N trajectories are always trained, LAST: before the "
                    "deadline the run jumps to them in time (user: the model should see our competition data last)")
    ap.add_argument("--lr-floor", type=float, default=0.0, help="the LR decays to this fraction of the peak, not to 0 (so the "
                    "tail trained at the end still moves the model)")
    ap.add_argument("--system-prompt", help="file: replaces the recorded system prompt (train = deploy)")
    ap.add_argument("--deploy-budget", help='JSON, e.g. {"time_minutes": 5, "tool_calls": 30, "turns": 80, "timeout_seconds": 240}')
    ap.add_argument("--served-model", default="google/gemma-4-E4B-it-qat-w4a16-ct", help="written into adapter_config.json")
    ap.add_argument("--fidelity-tol", type=int, default=96, help="tokens: a turn whose rendered prompt departs from the recorded "
                    "size by more than this (beyond the first turn's offset) is after a compaction; it and later turns are dropped")
    ap.add_argument("--past-reasoning", choices=["keep", "drop"], default="keep")
    a = ap.parse_args()
    out = Path(a.out).expanduser()
    out.mkdir(parents=True, exist_ok=True)
    global PAST_REASONING, COMPACTION_TOL, ERROR_MASK, SYSTEM_PROMPT, DEPLOY_BUDGET
    PAST_REASONING, COMPACTION_TOL, ERROR_MASK = a.past_reasoning, a.fidelity_tol, not a.no_error_mask
    SYSTEM_PROMPT = Path(a.system_prompt).read_text() if a.system_prompt else None
    DEPLOY_BUDGET = json.loads(a.deploy_budget) if a.deploy_budget else None
    from transformers import AutoTokenizer
    a.base = pick_base(a.base)
    print(f"[base] {a.base}", flush=True)
    tok = AutoTokenizer.from_pretrained(a.base)
    tools = harness_tool_schemas(B2_TOOLS)
    if RANK == 0:
        (out / "tools_rendered.json").write_text(json.dumps(tools, indent=1))
    trajs = [json.loads(l) for l in open(a.data) if l.strip()]
    tail_tasks = {t["instance_id"] for t in trajs[len(trajs) - a.tail_n:]} if a.tail_n else set()
    tasks = sorted({t["instance_id"] for t in trajs} - tail_tasks)   # the --tail-n trajectories are never held out
    random.Random(a.seed).shuffle(tasks)
    n_val = max(1, round(len(tasks) * a.val_frac)) if len(tasks) > 3 else 0
    val_tasks = set(tasks[:n_val])
    stats = {"prefix_mismatch": 0, "mismatch_examples": [], "too_long": 0, "prompt_tok_pairs": [], "after_compaction": 0,
             "compacted_trajectories": 0, "compactions_applied": 0, "masked_turns": 0, "windows": 0, "window_fallback": 0, "target_too_long": 0}
    splits = {"train": [], "val": []}
    owner = []                          # trajectory key of each train sample (resume bookkeeping)
    tkey = lambda t, k: t.get("source_traj_id") or f"{t['instance_id']}#{k}"
    for k, t in enumerate(trajs):
        for s in window_samples(tok, t, tools, stats, a.max_target, a.max_len):
            sp = "val" if t["instance_id"] in val_tasks else "train"
            splits[sp].append(s)
            if sp == "train":
                owner.append(tkey(t, k))
    rep = {"trajectories": len(trajs), "tasks": len(tasks), "val_tasks": sorted(val_tasks),
           "samples": {k: len(v) for k, v in splits.items()},
           "target_tokens": {k: sum(len(p) for _, p in v) for k, v in splits.items()},
           "sequence_tokens": {k: sum(len(i) for i, _ in v) for k, v in splits.items()},
           "max_sample_tokens": max((len(i) for v in splits.values() for i, _ in v), default=0),
           **{k: stats[k] for k in ("prefix_mismatch", "too_long", "target_too_long", "masked_turns", "windows", "window_fallback")},
           "no_reasoning_turns_context_only": stats.get("no_reasoning_context_only", 0),
           "base": a.base, "past_reasoning": a.past_reasoning, "system_prompt_replaced": bool(SYSTEM_PROMPT),
           "deploy_budget": DEPLOY_BUDGET,
           "turns_dropped_after_compaction": stats["after_compaction"], "trajectories_compacted": stats["compacted_trajectories"],
           "compaction_summaries_applied": stats["compactions_applied"]}
    pairs = stats["prompt_tok_pairs"]
    if pairs:   # rendered prompt tokens vs the trace's recorded prompt_tokens (same turn)
        d = sorted(r - o for o, r in pairs)
        rep["prompt_tokens_recorded_minus_rendered"] = {"n": len(d), "median": d[len(d) // 2], "p10": d[len(d) // 10],
                                                        "p90": d[9 * len(d) // 10]}
    if RANK == 0:
        (out / "check.json").write_text(json.dumps({**rep, "mismatch_examples": stats["mismatch_examples"]}, indent=1))
        print(json.dumps(rep, indent=1), flush=True)
    if stats["prefix_mismatch"]:
        sys.exit(f"STOP: {stats['prefix_mismatch']} turns do not continue their prompt under this chat template "
                 f"(see {out / 'check.json'}).")
    if a.check:
        print("check ok")
        return
    if not splits["train"]:
        sys.exit("STOP: no training samples")

    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
    import torch
    import torch.nn.functional as F
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    torch.manual_seed(a.seed)
    ngpu = torch.cuda.device_count()
    gpu = LOCAL % ngpu   # more ranks than GPUs only in a smoke test (ranks share a GPU over gloo)
    torch.cuda.set_device(gpu)
    dev = f"cuda:{gpu}"
    if WORLD > 1:
        import torch.distributed as dist
        dist.init_process_group("nccl" if ngpu >= WORLD else "gloo")
    if a.device_map == "auto":
        if WORLD > 1:
            sys.exit("--device-map auto is one process; do not use torchrun")
        # "balanced", not "auto": "auto" fills GPU 0 to max_memory first, and the 31B weights (~19 GB in 4 bits) would all land
        # there, leaving no room for its activations. Balanced spreads the layers evenly; max_memory stays a hard cap.
        kw = dict(dtype=torch.bfloat16, device_map="balanced", attn_implementation="sdpa",
                  max_memory={i: f"{a.max_memory_gib}GiB" for i in range(ngpu)})
        dev = "cuda:0"
    elif a.device_map == "custom":
        if WORLD > 1:
            sys.exit("--device-map custom is one process; do not use torchrun")
        kw = dict(dtype=torch.bfloat16, device_map=custom_device_map(a.base, ngpu, a.last_gpu_layers), attn_implementation="sdpa")
        dev = "cuda:0"
    else:
        kw = dict(dtype=torch.bfloat16, device_map={"": gpu}, attn_implementation="sdpa")
    if a.quant == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    for r_ in range(WORLD):                  # one rank at a time: simultaneous loads could exhaust host RAM
        if r_ == RANK:
            model = AutoModelForCausalLM.from_pretrained(a.base, **kw)
        if WORLD > 1:
            dist.barrier()
    # NOT prepare_model_for_kbit_training: it casts every non-quantized weight to float32 (E4B's per-layer embeddings: +11 GB)
    for prm in model.parameters():
        prm.requires_grad_(False)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.config.use_cache = False
    if RANK == 0:
        print(f"GPU memory after loading the base: {[round(torch.cuda.memory_allocated(i) / 1e9, 1) for i in range(ngpu)]} GB", flush=True)
    # every language-model projection, except k_proj on layers where K doubles as V (attention_k_eq_v, full attention: 31B)
    tc = model.config.get_text_config()
    n_layers, lt = tc.num_hidden_layers, getattr(tc, "layer_types", None) or []
    keq = [i for i in range(n_layers) if getattr(tc, "attention_k_eq_v", False) and i < len(lt) and lt[i] == "full_attention"]
    k_ok = "|".join(str(i) for i in range(n_layers) if i not in keq)
    target = (r".*language_model.*\.(q_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
              + (rf"|.*language_model.*\.layers\.({k_ok})\.self_attn\.k_proj" if k_ok else ""))
    if RANK == 0:
        print(f"k_eq_v layers without a k_proj LoRA: {keq}", flush=True)
    lcfg = LoraConfig(r=a.rank, lora_alpha=a.alpha, lora_dropout=a.dropout, bias="none", task_type="CAUSAL_LM",
                      target_modules=target)
    if a.init_adapter:                  # resume: the previous session's LoRA, trainable
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, a.init_adapter, is_trainable=True, autocast_adapter_dtype=False)
        if RANK == 0:
            print(f"resumed from adapter {a.init_adapter}", flush=True)
    else:
        model = get_peft_model(model, lcfg, autocast_adapter_dtype=False)
    if RANK == 0:
        model.print_trainable_parameters()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=0.0)
    train = splits["train"]
    accum = max(1, round(a.grad_accum / WORLD))     # per rank; effective batch = accum x WORLD
    plan = {"total": math.ceil(int(len(train) * a.epochs) // WORLD / accum)}
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, plan["total"] // 10)) * max(a.lr_floor, 1 - s / max(1, plan["total"])))

    FAKE_OOM_ABOVE = int(os.environ.get("TRAIN_LORA_FAKE_OOM_ABOVE", "0"))   # test only: simulate OOM on longer samples
    CE_CHUNK = 512
    CE_NO_CKPT = os.environ.get("TRAIN_LORA_CE_NO_CKPT") == "1"

    def loss_of(ids, pos):
        if FAKE_OOM_ABOVE and len(ids) > FAKE_OOM_ABOVE:
            raise torch.OutOfMemoryError(f"simulated out of memory ({len(ids)} tokens > {FAKE_OOM_ABOVE})")
        # logits only at the positions that predict a target token (logits_to_keep as an index tensor); CE in float32 chunks
        x = torch.tensor([ids], device=dev)
        # a plain list, not a tensor: accelerate's device_map hook moves tensor arguments to the FIRST GPU, while the output
        # layer sits on the last one (Kaggle nb19, Oct 15: "indices should be either on cpu or on the same device ... cuda:3");
        # the model indexes hidden_states[:, keep, :] with it unchanged
        keep = [p - 1 for p in pos]
        logits = model(input_ids=x, logits_to_keep=keep, use_cache=False).logits[0]
        y = torch.tensor([ids[p] for p in pos], device=logits.device)
        # Gemma's 262k vocabulary: T target tokens give T x 262k logits. The float32 copy of each chunk is recomputed in the
        # backward pass (checkpoint) instead of being kept for all chunks at once (session 1, Oct 19: at --max-target 8000 the
        # stored float32 chunks alone were ~8 GB on GPU 3, which also holds the output layer, and every window ran out of memory)
        from torch.utils.checkpoint import checkpoint as _ckpt
        ce = lambda lg, yy: F.cross_entropy(lg.float(), yy, reduction="sum")
        tot = 0.0
        for c in range(0, len(pos), CE_CHUNK):
            if CE_NO_CKPT:                  # test only: the pre-Oct-19 loss, for a memory comparison
                tot = tot + ce(logits[c:c + CE_CHUNK], y[c:c + CE_CHUNK])
            else:
                tot = tot + _ckpt(ce, logits[c:c + CE_CHUNK], y[c:c + CE_CHUNK], use_reentrant=False)
        return tot / len(pos)

    oom = {"len_cap": float("inf"), "events": 0, "skipped_pos": set(), "val_skipped": 0}   # in-process OOM recovery state

    def evaluate():
        if not splits["val"]:
            return None
        model.eval()
        with torch.no_grad():
            v = []
            for i, p in splits["val"][RANK::WORLD]:
                if len(i) >= oom["len_cap"]:
                    continue
                try:
                    v.append(loss_of(i, p).item())
                except torch.OutOfMemoryError:
                    oom["val_skipped"] += 1
                    torch.cuda.empty_cache()
        model.train()
        t = torch.tensor([sum(v), len(v)], device=dev, dtype=torch.float64)
        if WORLD > 1:
            dist.all_reduce(t)
        return (t[0] / max(t[1], 1)).item()

    log = open(out / "train_log.jsonl", "a") if RANK == 0 else open(os.devnull, "w")
    rng, step, t0 = random.Random(a.seed), 0, time.time()
    order = []
    while len(order) < len(train) * a.epochs:
        o = list(range(len(train)))
        if a.keep_order:                # the file's priority order, shuffled only inside small blocks
            o = [x for b in range(0, len(o), a.order_block) for x in rng.sample(o[b:b + a.order_block], len(o[b:b + a.order_block]))]
        else:
            rng.shuffle(o)
        order += o
    order = order[:int(len(train) * a.epochs)]
    tail_keys = set()
    if a.tail_n:                        # the last N trajectories of the file: always trained, at the very end
        keys_in_order = list(dict.fromkeys(owner))
        tail_keys = set(keys_in_order[-a.tail_n:])
        tail = [j for j in order if owner[j] in tail_keys]
        order = [j for j in order if owner[j] not in tail_keys] + tail
    order = order[RANK::WORLD][:len(order) // WORLD]     # same shuffled order on every rank; equal shard lengths
    tail_start = len(order) - sum(1 for j in order if owner[j] in tail_keys)
    model.train()
    if not a.no_eval_start:
        v0 = evaluate()
        if RANK == 0:
            print(f"val loss before training: {v0}", flush=True)
        log.write(json.dumps({"step": 0, "val_loss": v0}) + "\n")
    import signal
    stop = {"why": None}
    def _on_signal(sig, frm):
        stop["why"] = f"signal {sig}"
    for sg in (signal.SIGTERM, signal.SIGINT):
        signal.signal(sg, _on_signal)
    done_i, seq_tok, last_save = 0, 0, time.time()
    if a.stop_after_s:
        a.stop_at = min(a.stop_at or float("inf"), time.time() + a.stop_after_s)

    def save(dest_name="adapter"):
        """Atomic: write to a temp dir, then rename over the old one, with the trajectories trained so far."""
        if RANK != 0:
            return
        tmp, dst = out / (dest_name + ".tmp"), out / dest_name
        import shutil
        shutil.rmtree(tmp, ignore_errors=True)
        model.save_pretrained(tmp)
        cp = tmp / "adapter_config.json"
        c_ = json.loads(cp.read_text()); c_["base_model_name_or_path"] = a.served_model; cp.write_text(json.dumps(c_, indent=2))
        seen = {}
        for j_ in (x for k__, x in enumerate(order[:done_i]) if k__ not in oom["skipped_pos"]):
            seen[owner[j_]] = seen.get(owner[j_], 0) + 1
        need = {}
        for o_ in owner:
            need[o_] = need.get(o_, 0) + 1
        (tmp / "trained_ids.json").write_text(json.dumps(sorted(k for k, n in seen.items() if n >= need[k])))
        shutil.rmtree(dst, ignore_errors=True)
        tmp.rename(dst)

    run, i = 0.0, 0
    base_alloc = [torch.cuda.memory_allocated(g) for g in range(ngpu)]   # for the leak check after an OOM
    grp = []                                # positions in `order` accumulated into the current optimizer step
    while i < len(order):
        if tail_keys and a.stop_at and i < tail_start and i % accum == 0 and i >= accum:
            per_sample = (time.time() - t0) / i
            if time.time() + 1.3 * per_sample * (len(order) - tail_start + accum) > a.stop_at:
                if RANK == 0:
                    print(f"deadline: skipping {tail_start - i} main samples to train the {len(order) - tail_start} tail samples",
                          flush=True)
                order = order[:i] + order[tail_start:]
                tail_start = i
        j = order[i]
        i += 1
        if len(train[j][0]) >= oom["len_cap"]:          # as long as a sample that ran out of memory: skip
            oom["skipped_pos"].add(i - 1)
        else:
            failed = False
            try:
                loss = loss_of(*train[j]) / accum
                loss.backward()
                run += loss.item()
                seq_tok += len(train[j][0])
                grp.append(i - 1)
            except torch.OutOfMemoryError as e:
                failed, msg = True, str(e).split("\n")[0][:160]
            if failed:
                # in-process recovery (no reload): the group's partial gradients are discarded and its optimizer step skipped
                # (AdamW would still move the weights on zero gradients); this sample and every later one at least as long
                # are skipped; the trajectories involved are not counted as trained
                loss = None
                opt.zero_grad(set_to_none=True)
                import gc; gc.collect(); torch.cuda.empty_cache()
                oom["events"] += 1
                oom["len_cap"] = min(oom["len_cap"], len(train[j][0]))
                oom["skipped_pos"].update(grp + [i - 1]); grp.clear(); oom["group_failed"] = True
                over = [round((torch.cuda.memory_allocated(g) - base_alloc[g]) / 2**30, 2) for g in range(ngpu)]
                if max(over) > a.leak_gib:
                    # the failed backward kept memory alive (session 2, Oct 19: every later sample then ran out of memory,
                    # down to 4.9k tokens): save what is trained and let the notebook restart at a smaller window
                    if RANK == 0:
                        print(f"out of memory: {over} GiB above the pre-loop allocation was not given back; saving and "
                              f"exiting 75 (restart at a smaller window)", flush=True)
                    save()
                    sys.exit(75)
                if RANK == 0:
                    print(f"out of memory on a {len(train[j][0])}-token sample ({msg}); skipping samples of >= {oom['len_cap']} "
                          f"tokens from now on (event {oom['events']})", flush=True)
                if oom["events"] > a.max_oom_events:
                    sys.exit(f"STOP: {oom['events']} out-of-memory events: lower --max-len")
        if i % accum == 0 or i == len(order):
            grp.clear()
            if oom.pop("group_failed", False) or not any(p.grad is not None for p in params):
                run = 0.0
                continue                     # nothing (left) to apply from this group
            if WORLD > 1:                    # average LoRA gradients over ranks (equal shard lengths: every rank steps together)
                for p in params:
                    if p.grad is None:
                        p.grad = torch.zeros_like(p)
                    dist.all_reduce(p.grad)
                    p.grad.div_(WORLD)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1
            done_i = i
            if a.time_budget_h and step == 2:   # fit the run into the budget; same decision on every rank
                per_step = (time.time() - t0) / step
                fit = torch.tensor([int(a.time_budget_h * 3600 * 0.9 / per_step)], device=dev)
                if WORLD > 1:
                    dist.broadcast(fit, 0)
                if fit.item() < plan["total"]:
                    plan["total"] = max(step + 1, fit.item())
                    if tail_keys:               # keep the tail: cut the main part instead
                        keep = max(i, plan["total"] * accum - (len(order) - tail_start))
                        order = order[:min(tail_start, keep)] + order[tail_start:]
                        tail_start = min(tail_start, keep)
                    else:
                        order = order[:plan["total"] * accum]
                    if RANK == 0:
                        print(f"time budget {a.time_budget_h} h at {per_step:.0f} s/step: {plan['total']} steps "
                              f"({plan['total'] * accum * WORLD / len(train):.2f} epochs)", flush=True)
            rec = {"step": step, "of": plan["total"], "train_loss": round(run, 4), "lr": sched.get_last_lr()[0],
                   "elapsed_s": round(time.time() - t0), "seq_tokens": seq_tok,
                   "tok_per_s": round(seq_tok / max(1e-9, time.time() - t0), 1),
                   "gpu_mem_gb": [round(torch.cuda.max_memory_allocated(g) / 1e9, 1) for g in range(ngpu)]}
            if i >= len(order):
                rec["val_loss"] = evaluate()
            if RANK == 0:
                print(json.dumps(rec), flush=True)
            log.write(json.dumps(rec) + "\n"); log.flush()
            run = 0.0
            per = (time.time() - t0) / step
            if a.stop_at and time.time() + 1.5 * per > a.stop_at:      # the next step would end past the deadline
                stop["why"] = stop["why"] or "deadline"
            if stop["why"]:
                if RANK == 0:
                    print(f"stopping early ({stop['why']}) after step {step}: saving", flush=True)
                break
            if a.save_every_min and time.time() - last_save > a.save_every_min * 60:
                save(); last_save = time.time()
    if WORLD > 1:
        dist.barrier()
        if RANK != 0:
            dist.destroy_process_group()
            return
    ad = out / "adapter"
    save()                                               # base_model_name_or_path = the served model, + trained_ids.json
    (out / "train_meta.json").write_text(json.dumps({**rep, "args": vars(a), "steps": step, "k_eq_v_layers": keq,
                                                      "stopped": stop["why"], "seq_tokens": seq_tok,
                                                      "oom_events": oom["events"], "oom_len_cap": None if oom["len_cap"] == float("inf") else oom["len_cap"],
                                                      "oom_skipped_samples": len(oom["skipped_pos"]),
                                                      "tok_per_s": round(seq_tok / max(1e-9, time.time() - t0), 1),
                                                      "seconds": round(time.time() - t0)}, indent=1))
    print(f"adapter saved: {ad}")
    if WORLD > 1:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/gc/tool_trace.py
#!/usr/bin/env python3
"""Follow every tool call: what was it for, did it touch or reveal the code the fix needs, and was its output acted on?

    python tool_trace.py RUN_DIR [RUN_DIR ...] --tasks tasks.jsonl [--valid profile.csv] [--out DIR]

Reference = the task's gold patch (tasks.jsonl `patch`): the files it changes and the functions/classes it touches (hunk
headers and changed def/class lines). Function level on purpose: file choice alone barely separates success from failure,
function selection does (TRACEPROBE, arXiv 2607.06184). Per tool call:
  purpose      normalized action: read-src / read-test, edit-src / edit-test-or-script, test, search, inspect, reproduce,
               vcs, graph:<tool>, status, submit (cf. TRACEPROBE's canonical action taxonomy)
  on_gold      its target (file, node, query, command) names a gold file or gold function
  reveals_gold its output names a gold file or function that nothing earlier in the trajectory had named
  followed     one of the next 3 calls targets a .py file or name that first appeared in this call's output
Per trajectory: which call and purpose first reached the gold code, after how many calls, and whether the gold file was
edited. Outputs (in --out, default the current folder): tool_trace.md (per-purpose table for resolved vs unresolved runs,
who first finds the gold code, the shortest resolved routes, purpose bigrams), tool_trace_calls.csv (one row per call),
tool_trace_runs.csv (one row per trajectory). Heuristic text matching: association, not causation.
"""
import argparse
import csv
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

PY = re.compile(r"[\w./-]+\.py\b")


def gold_of(patch):
    files = list(dict.fromkeys(re.findall(r"^diff --git a/(\S+) b/", patch, re.M) +
                               re.findall(r"^\+\+\+ b/(\S+)", patch, re.M)))      # tasks.jsonl patches have only ---/+++
    funcs = set(re.findall(r"^@@[^@]*@@\s*(?:async\s+)?(?:def|class)\s+(\w+)", patch, re.M))
    funcs |= set(re.findall(r"^[+-]\s*(?:async\s+)?(?:def|class)\s+(\w+)", patch, re.M))
    # library code only: tests and documentation examples (docs_src/: names like `app`, `Item`) are not the fix location;
    # dunders (`__init__` is in 26 gold patches) and names under 4 characters match far too much text to mean anything
    src = [f for f in files if not re.search(r"(^|/)(tests?|testing|docs|docs_src)/|test_[^/]*\.py$|_test\.py$|conftest\.py$", f)]
    return {"files": src or files,
            "funcs": {f for f in funcs if not f.startswith(("test", "__")) and len(f) >= 4} if src else set()}


def purpose(tool, args):
    path = str(args.get("filepath", ""))
    is_test = bool(re.search(r"test|repro|reproduce|scratch|tmp", path, re.I))
    if tool == "read_file":
        return "read-test" if is_test else "read-src"
    if tool in ("edit_file", "write_file"):
        return "edit-test-or-script" if is_test or not path.endswith(".py") else "edit-src"
    if tool == "run_command":
        c = str(args.get("command", ""))
        for name, rx in (("test", r"\bpytest\b|-m\s+pytest|unittest"), ("search", r"\b(grep|rg|find|ls|git\s+grep|ack)\b"),
                         ("vcs", r"\bgit\s+(diff|status|log|show|stash|checkout)"), ("install", r"\bpip\b"),
                         ("reproduce", r"\bpython3?\b\s+(-c\b|[\w./-]+\.py)"), ("inspect", r"\b(cat|sed|head|tail|nl|less|wc)\b")):
            if re.search(rx, c):
                return name
        return "command-other"
    if tool in ("get_code_neighbors", "search_similar_code", "get_code_subgraph"):
        return f"graph:{tool}"
    return {"get_status": "status", "submit_patch": "submit"}.get(tool, tool or "?")


def gold_items(text, gold):
    """The gold files (full path, or basename as a word) and gold functions (as a word) that `text` names."""
    text = str(text)
    files = {f for f in gold["files"] if f in text or re.search(rf"(?<![\w.-]){re.escape(f.split('/')[-1])}\b", text)}
    return files | {fn for fn in gold["funcs"] if re.search(rf"\b{re.escape(fn)}\b", text)}


def names_gold(text, gold):
    return bool(gold_items(text, gold))


def target(args):
    return str(args.get("filepath") or args.get("node") or args.get("query") or args.get("command") or "")


def calls_of(trace):
    out = []
    for s in trace.get("steps", []):
        obs = s.get("observation")
        content = obs.get("content", "") if isinstance(obs, dict) else ""
        for tc in s.get("tool_calls") or []:
            out.append({"tool": tc.get("function_name"), "args": tc.get("arguments") or {}, "output": str(content)})
    return out


def trace_run(trace, gold):
    calls = calls_of(trace)
    seen_text, seen_gold, rows, first = "", set(), [], None
    for i, c in enumerate(calls):
        tgt = target(c["args"])
        p = purpose(c["tool"], c["args"])
        on = names_gold(tgt, gold)
        new_gold = bool(gold_items(c["output"], gold) - seen_gold - gold_items(tgt, gold))
        seen_gold |= gold_items(tgt, gold) | gold_items(c["output"], gold)
        new_names = {m for m in PY.findall(c["output"]) if m not in seen_text}
        followed = any(any(n in target(d["args"]) or n.split("/")[-1] in target(d["args"]) for n in new_names)
                       for d in calls[i + 1:i + 4]) if new_names else False
        if first is None and (on or new_gold):
            first = (i, p)
        rows.append({"i": i, "purpose": p, "tool": c["tool"], "target": tgt[:160], "on_gold": on, "reveals_gold": new_gold,
                     "followed": followed})
        seen_text += " " + tgt + " " + c["output"]
    edited_gold = any(r["purpose"] == "edit-src" and names_gold(r["target"], gold) for r in rows)
    return rows, {"n_calls": len(rows), "first_gold_call": first[0] if first else None,
                  "first_gold_purpose": first[1] if first else None, "edited_gold_file": edited_gold}


def collapse(seq):
    out = []
    for p in seq:
        if out and out[-1][0] == p:
            out[-1][1] += 1
        else:
            out.append([p, 1])
    return " → ".join(p if n == 1 else f"{p}×{n}" for p, n in out)


def analyze(run_dirs, tasks, valid=None):
    calls, runs = [], []
    for rd in map(Path, run_dirs):
        for line in open(rd / "task_results.jsonl"):
            if not line.strip():
                continue
            r = json.loads(line)
            iid = r["instance_id"]
            if (valid is not None and iid not in valid) or iid not in tasks:
                continue
            tp = rd / "traces" / f"trace_{iid}.json"
            if not tp.exists():
                continue
            gold = gold_of(tasks[iid]["patch"])
            rows, summ = trace_run(json.load(open(tp)), gold)
            summ["issue_names_gold_file"] = bool(gold_items(tasks[iid]["problem"], {"files": gold["files"], "funcs": set()}))
            summ["issue_names_gold_func"] = bool(gold_items(tasks[iid]["problem"], {"files": [], "funcs": gold["funcs"]}))
            res = bool(r.get("resolved"))
            for x in rows:
                calls.append({"run": rd.name, "instance_id": iid, "resolved": res, **x})
            runs.append({"run": rd.name, "instance_id": iid, "resolved": res, **summ,
                         "route": collapse([x["purpose"] for x in rows])})
    return calls, runs


def report(calls, runs):
    pct = lambda g, k: f"{sum(x[k] for x in g) / len(g):.0%}" if g else "-"
    L = ["# Tool tracing: what each tool call was for, and whether it reached or revealed the gold code", "",
         f"{len(runs)} trajectories ({sum(r['resolved'] for r in runs)} resolved), {len(calls)} tool calls. "
         "Heuristic text matching against the gold patch; association, not causation.", "",
         "## Per purpose: resolved vs unresolved trajectories", "",
         "| purpose | calls (res / unres) | on gold code (res / unres) | reveals gold code first (res / unres) | output followed within 3 calls (res / unres) |",
         "|---|---|---|---|---|"]
    by = defaultdict(lambda: ([], []))
    for c in calls:
        by[c["purpose"]][0 if c["resolved"] else 1].append(c)
    for p in sorted(by, key=lambda p: -len(by[p][0]) - len(by[p][1])):
        a, b = by[p]
        L.append(f"| {p} | {len(a)} / {len(b)} | {pct(a, 'on_gold')} / {pct(b, 'on_gold')} | {pct(a, 'reveals_gold')} / "
                 f"{pct(b, 'reveals_gold')} | {pct(a, 'followed')} / {pct(b, 'followed')} |")
    L += ["", "## Who first reaches the gold code (first call whose target or new output names it)", "",
          "| first purpose | resolved runs | unresolved runs |", "|---|---|---|"]
    fr = Counter((r["first_gold_purpose"] or "never", r["resolved"]) for r in runs)
    for p in sorted({k[0] for k in fr}, key=lambda p: -(fr[(p, True)] + fr[(p, False)])):
        L.append(f"| {p} | {fr[(p, True)]} | {fr[(p, False)]} |")
    med = lambda xs: sorted(xs)[len(xs) // 2] if xs else "-"
    for lab, g in (("resolved", [r for r in runs if r["resolved"]]), ("unresolved", [r for r in runs if not r["resolved"]])):
        L.append(f"\n{lab}: median calls before the gold code is reached {med([r['first_gold_call'] for r in g if r['first_gold_call'] is not None])}; "
                 f"edited a gold file {sum(r['edited_gold_file'] for r in g)}/{len(g)}; never reached it {sum(r['first_gold_call'] is None for r in g)}/{len(g)}")
    for lab, k in (("a gold file", "issue_names_gold_file"), ("a gold function", "issue_names_gold_func")):
        g1, g0 = [r for r in runs if r[k]], [r for r in runs if not r[k]]
        L.append(f"issue text (problem + hints) names {lab}: {len(g1)}/{len(runs)} trajectories; resolved "
                 f"{sum(r['resolved'] for r in g1)}/{len(g1)} when it does vs {sum(r['resolved'] for r in g0)}/{len(g0)} when not")
    L += ["", "## Shortest resolved routes (the best cases)", ""]
    for r in sorted([r for r in runs if r["resolved"]], key=lambda r: r["n_calls"])[:10]:
        L.append(f"- `{r['instance_id']}` ({r['n_calls']} calls): {r['route']}")
    L += ["", "## Purpose bigrams: share of resolved vs unresolved trajectories containing them (top 15 by difference)", "",
          "| bigram | in resolved | in unresolved |", "|---|---|---|"]
    seqs = defaultdict(list)
    for c in calls:
        seqs[(c["run"], c["instance_id"], c["resolved"])].append(c["purpose"])
    big = lambda s: {(a, b) for a, b in zip(s, s[1:])}
    R = [big(s) for k, s in seqs.items() if k[2]]
    U = [big(s) for k, s in seqs.items() if not k[2]]
    allb = set().union(*R, *U) if (R or U) else set()
    share = lambda g, b: sum(b in x for x in g) / len(g) if g else 0
    for b in sorted(allb, key=lambda b: -abs(share(R, b) - share(U, b)))[:15]:
        L.append(f"| {b[0]} → {b[1]} | {share(R, b):.0%} | {share(U, b):.0%} |")
    return "\n".join(L) + "\n"


def write(out, calls, runs):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    (out / "tool_trace.md").write_text(report(calls, runs))
    for name, rows in (("tool_trace_calls.csv", calls), ("tool_trace_runs.csv", runs)):
        if rows:
            with open(out / name, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(rows[0]))
                w.writeheader()
                w.writerows(rows)


def selftest():
    patch = "diff --git a/pkg/core.py b/pkg/core.py\n@@ -1,3 +1,3 @@ def solve(x):\n-    return 1\n+    return 2\n"
    g = gold_of(patch)
    assert g == {"files": ["pkg/core.py"], "funcs": {"solve"}}, g
    assert gold_of("--- a/x/y.py\n+++ b/x/y.py\n@@ -1 +1 @@ def f(\n-a\n+b\n")["files"] == ["x/y.py"]
    assert purpose("run_command", {"command": "python -m pytest tests -x"}) == "test"
    assert purpose("run_command", {"command": "grep -rn solve pkg"}) == "search"
    assert purpose("run_command", {"command": "python3 reproduce.py"}) == "reproduce"
    assert purpose("edit_file", {"filepath": "pkg/core.py"}) == "edit-src"
    assert purpose("write_file", {"filepath": "reproduce_issue.py"}) == "edit-test-or-script"
    tr = {"steps": [
        {"tool_calls": [{"function_name": "run_command", "arguments": {"command": "grep -rn solve ."}}],
         "observation": {"content": "pkg/core.py:1:def solve(x):"}},
        {"tool_calls": [{"function_name": "read_file", "arguments": {"filepath": "pkg/core.py"}}], "observation": {"content": "..."}},
        {"tool_calls": [{"function_name": "edit_file", "arguments": {"filepath": "pkg/core.py"}}], "observation": {"content": "ok"}}]}
    rows, s = trace_run(tr, g)
    assert rows[0]["reveals_gold"] and rows[0]["followed"] and rows[1]["on_gold"] and not rows[1]["reveals_gold"], rows
    assert s == {"n_calls": 3, "first_gold_call": 0, "first_gold_purpose": "search", "edited_gold_file": True}, s
    assert collapse(["a", "a", "b"]) == "a×2 → b"
    print("selftest ok")


def main():
    if "--selftest" in sys.argv:
        return selftest()
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("runs", nargs="+")
    ap.add_argument("--tasks", required=True)
    ap.add_argument("--valid", help="validity profile.csv: only sub_valid tasks")
    ap.add_argument("--out", default=".")
    a = ap.parse_args()
    tasks = {t["instance_id"]: {"patch": t["patch"], "problem": t.get("problem_statement", "") + "\n" + (t.get("hints_text") or "")}
             for t in map(json.loads, open(a.tasks)) if t.get("patch")}
    valid = {x["instance_id"] for x in csv.DictReader(open(a.valid)) if x.get("sub_valid") == "True"} if a.valid else None
    calls, runs = analyze(a.runs, tasks, valid)
    write(a.out, calls, runs)
    print((Path(a.out) / "tool_trace.md").read_text())


if __name__ == "__main__":
    main()


In [ ]:
# STEP 3 - vLLM helpers: one server per port; a server can be pinned to some GPUs (two students side by side)
import traceback, threading
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
SERVERS = {}          # port -> VllmServer
def kill_gpu_procs():
    subprocess.run("nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9; "
                   "pkill -9 -f 'VLLM::|vllm.entrypoints|EngineCore'", shell=True)
    time.sleep(10)
def stop_servers():
    for port, srv in list(SERVERS.items()):
        try:
            srv.stop()
        except Exception as e:
            plog(f"vLLM :{port} stop failed: {e}")
        SERVERS.pop(port, None)
    kill_gpu_procs()
def start_server(model_dir, tp, adapter_sub=None, port=8000, gpus=None):
    """Serve as the scoring harness does (gemma4 parsers, thinking on); adapters discovered from a submission folder.
    gpus: CUDA device ids for this server only (the variable is set while vLLM's process is spawned)."""
    adapters = discover_adapters(str(adapter_sub), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS) if adapter_sub else None
    cfg = VllmConfig(model=str(model_dir), port=port, tool_call_parser="gemma4", reasoning_parser="gemma4",
                     default_chat_template_kwargs={"enable_thinking": True}, max_model_len=SERVE["max_model_len"],
                     dtype="bfloat16", gpu_memory_utilization=SERVE["util"], enable_auto_tool_choice=True,
                     enable_lora=bool(adapters), max_loras=8, max_lora_rank=128, tensor_parallel_size=tp,
                     startup_timeout=SERVER_STARTUP_TIMEOUT)
    srv = VllmServer(cfg, adapter_manifest=adapters)
    plog(f"starting vLLM :{port} on GPUs {gpus or 'all'}: {shlex.join(srv.build_cmd())[:400]}")
    old = os.environ.get("CUDA_VISIBLE_DEVICES")
    if gpus is not None:
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, gpus))
    t0 = time.time()
    try:
        srv.start()
    finally:
        if gpus is not None:
            if old is None:
                os.environ.pop("CUDA_VISIBLE_DEVICES", None)
            else:
                os.environ["CUDA_VISIBLE_DEVICES"] = old
    SERVERS[port] = srv
    plog(f"vLLM :{port} up after {time.time() - t0:.0f} s")
    return srv
def server_alive(port=8000):
    import urllib.request
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=20).read()
        return True
    except Exception:
        return False
def agent_cmd(sub, ids, results, model_dir, alias, concurrency=CONCURRENCY, timeout_s=None, port=8000):
    return (f"cd {GC} && " + (f"timeout -k 60 {int(timeout_s)} " if timeout_s else "") + f"{sys.executable} run_arm.py "
            f"--mode agent --tasks {TASKS_JSONL} --snapshots {SNAPSHOTS} "
            f"--submission {sub} --results {results} --sandbox subprocess --task-ids {' '.join(ids)} "
            f"--max-tool-calls {BUDGET['max_tool_calls']} --max-time-minutes {BUDGET['max_time_minutes']} "
            f"--max-turns {BUDGET['max_turns']} --timeout-seconds {BUDGET['timeout_seconds']} --concurrency {concurrency} "
            f"--compaction-interval {COMPACTION_INTERVAL} --model-path {model_dir} --model-alias {alias} --port {port}")


In [ ]:
# STEP 5 - training data for THIS session: untrained trajectories, own first, then new problems shortest-first, ...
import random, hashlib
MIX = next((Path(dp) for dp, dns, fns in walk(INPUT, maxdepth=7) if "train_fit.jsonl" in fns and "manifest.json" in fns), None)
plog(f"SFT mix: {MIX or 'NOT FOUND: attach the graphcomp-sft-mix dataset'}")
if MIX:
    plog("mix manifest: " + json.dumps({k: v for k, v in json.loads((MIX / 'manifest.json').read_text()).items() if k != 'inputs'})[:1500])
DEPLOY_PROMPT = GC / "deploy_system.md"
DEPLOY_PROMPT.write_text((Path(SAMPLE_SUB) / "prompts/system.md").read_text() + PROMPT_ADDENDUM)
task_of = lambda r: r["instance_id"].split("@")[0]
def traj_key(r):   # stable across sessions (train_lora.py records the same key in trained_ids.json)
    if not r.get("source_traj_id"):
        calls = [json.dumps(tc["function"], sort_keys=True) for m in r["messages"] for tc in (m.get("tool_calls") or [])]
        r["source_traj_id"] = "own:" + task_of(r) + ":" + hashlib.sha1("\\n".join(calls).encode()).hexdigest()[:12]
    return r["source_traj_id"]
TRAINED = set(STATE.get("trained_ids", []))
TRAINED_TASKS = set(STATE.get("trained_tasks", []))
def _scan(path):
    """Light records (key, task, repo, est_tokens, source, byte offset): the 3.7 GB file is never held in memory."""
    out_ = []
    with open(path, "rb") as fh:
        while True:
            off = fh.tell(); line = fh.readline()
            if not line:
                break
            if not line.strip():
                continue
            r = json.loads(line)
            if r.get("est_tokens", 0) > max(TRAIN_MAX_LENS):
                continue
            out_.append({"source_traj_id": traj_key(r), "instance_id": r["instance_id"], "repo": r.get("repo", ""),
                         "est_tokens": r.get("est_tokens", 0), "mix_source": r.get("mix_source", "rft"),
                         "_file": str(path), "_off": off})
    return out_
_rows = _scan(MIX / "train_fit.jsonl") if MIX else []
# our verified 31B trajectories on the competition tasks (graphcomp-comp-trajs: notebooks 01 + 19, dev/holdout included)
COMP_FILE = next((Path(dp) / "competition_trajs.jsonl" for dp, dns, fns in walk(INPUT, maxdepth=7) if "competition_trajs.jsonl" in fns), None)
plog(f"competition trajectories: {COMP_FILE or 'NOT FOUND: attach graphcomp-comp-trajs (only the mix copies are used)'}")
_known = {r["source_traj_id"] for r in _rows}
for r in (_scan(COMP_FILE) if COMP_FILE else []):
    if r["source_traj_id"] not in _known:          # same actions = the same trajectory (the mix holds 17 of them)
        r["mix_source"] = "rft"; _rows.append(r); _known.add(r["source_traj_id"])
def _load(r):
    with open(r["_file"], "rb") as fh:
        fh.seek(r["_off"]); full = json.loads(fh.readline())
    full["source_traj_id"] = r["source_traj_id"]; full["mix_source"] = r["mix_source"]
    return full
# COMPETITION trajectories (our own on the 129 tasks + external ones on the four competition repos) are trained LAST in every
# session (user, Oct 15: "save our competition dataset for the end, we want it to see that last"), every session again
COMP_REPOS = {"fastapi/fastapi", "tiangolo/fastapi", "textualize/rich", "psf/requests", "encode/httpx",
              "fastapi", "rich", "requests", "httpx"}
is_comp = lambda r: r.get("mix_source") == "rft" or str(r.get("repo", "")).lower() in COMP_REPOS
_comp_all = sorted([r for r in _rows if is_comp(r)], key=lambda r: r.get("est_tokens", 0))
# repeat cap (literature check, Oct 18: re-showing the same texts many times leaves a model fragile, a fresh solution per visit
# does not, arXiv 2609.33559): a competition trajectory is trained in at most COMP_MAX_VISITS sessions. Sessions before this
# rule (no comp_visits in the state) count as one visit for every competition trajectory they trained.
COMP_VISITS = dict(STATE.get("comp_visits") or {k: 1 for k in (r["source_traj_id"] for r in _comp_all) if k in TRAINED})
# rotation: each session's tail is the least-visited COMP_PER_SESSION of them (every session still ENDS on competition data, as
# the user asked, while each text repeats about half as often); only if all are at the cap does the tail go over it (logged)
_n_tail = max(1, round(len(_comp_all) * COMP_PER_SESSION)) if _comp_all else 0
_under = [r for r in _comp_all if COMP_VISITS.get(r["source_traj_id"], 0) < COMP_MAX_VISITS]
_comp = sorted(_under or _comp_all, key=lambda r: (COMP_VISITS.get(r["source_traj_id"], 0), r.get("est_tokens", 0)))[:_n_tail]
_comp.sort(key=lambda r: r.get("est_tokens", 0))
COMP_OVER_CAP = not _under and bool(_comp_all)
_replay_pool = [r for r in _rows if not is_comp(r) and traj_key(r) in TRAINED]
_rows = [r for r in _rows if not is_comp(r) and traj_key(r) not in TRAINED]
def _mix_lengths(rs):
    """Round-robin over length quartiles (short, medium, long, longest, short, ...) instead of strictly shortest-first:
    SWE-Gym (arXiv 2412.21139) warns that successful-trajectory data is biased toward easy tasks; shortest-first would
    sharpen that inside one session."""
    if not LENGTH_MIX or len(rs) < 8:
        return sorted(rs, key=lambda r: r.get("est_tokens", 0))
    rs = sorted(rs, key=lambda r: r.get("est_tokens", 0))
    q = [rs[i * len(rs) // 4:(i + 1) * len(rs) // 4] for i in range(4)]
    out_ = []
    for i in range(max(map(len, q))):
        out_ += [b[i] for b in q if i < len(b)]
    return out_
# the rest: NEW problems (never trained) first, one trajectory each (the shortest of each problem), mixed over lengths; then
# second trajectories etc., mixed the same way
_rows.sort(key=lambda r: r.get("est_tokens", 0))
_first, _seen = [], set(TRAINED_TASKS)
for r in _rows:
    if task_of(r) not in _seen:
        _first.append(r); _seen.add(task_of(r))
_firstset = {id(r) for r in _first}
_queue = _mix_lengths(_first) + _mix_lengths([r for r in _rows if id(r) not in _firstset])
# size this session's slice: tokens the trainer can process before the deadline, times DATA_OVERSHOOT
TRAIN_DEADLINE = _T0 + (SESSION_BUDGET_H - POST_TRAIN_H) * 3600
# a speed only counts if that session took optimizer steps (session 2 measured 36 tok/s over 0 steps of OOM recoveries)
_meas = [x.get("tok_per_s") for x in STATE.get("session_log", []) if x.get("tok_per_s") and (x.get("steps") or 0) > 0]
_tps = _meas[-1] if _meas else TOK_PER_S_GUESS
_over = DATA_OVERSHOOT if _meas else FIRST_OVERSHOOT
_train_s = max(0.0, TRAIN_DEADLINE - time.time() - LOAD_H * 3600)
_comp_tok = sum(r.get("est_tokens", 0) for r in _comp)
_budget_tok = max(0.0, _tps * _train_s * _over - _comp_tok)
# replay (SWE-Lego's cumulative curriculum, arXiv 2601.01426; 2609.33559's replay): REPLAY_FRAC of the new-data tokens are
# trajectories trained in earlier sessions, spread through the slice (not at its end)
_replay_tok = _budget_tok * REPLAY_FRAC / (1 + REPLAY_FRAC) if _replay_pool else 0.0
_new_budget = _budget_tok - _replay_tok
_pick, _tok = [], 0
for r in _queue:
    if len(_pick) >= MAX_TRAIN_TRAJ:
        break
    if _pick and _tok + r.get("est_tokens", 0) > _new_budget:
        continue                      # too long for what is left: a shorter one may still fit
    _pick.append(r); _tok += r.get("est_tokens", 0)
_replay, _rtok = [], 0
for r in random.Random(STATE.get("sessions", 1)).sample(_replay_pool, len(_replay_pool)):
    if _rtok + r.get("est_tokens", 0) <= _replay_tok:
        _replay.append(r); _rtok += r.get("est_tokens", 0)
if _replay:                           # spread evenly through the new data
    step_ = max(1, len(_pick) // (len(_replay) + 1))
    for k_, r in enumerate(_replay):
        _pick.insert(min(len(_pick), (k_ + 1) * step_ + k_), r)
_tok += _rtok
TAIL_N = len(_comp)
COMP_KEYS = [r["source_traj_id"] for r in _comp]
_pick = _pick + _comp                 # the trainer always trains these, at the end (--tail-n)
if SMOKE:
    _pick, TAIL_N = (_comp or _queue)[:1], 0
TRAIN_DATA = OUT / "sft" / f"train_session{STATE.get('sessions', 1)}.jsonl"
TRAIN_DATA.parent.mkdir(parents=True, exist_ok=True)
with open(TRAIN_DATA, "w") as f:
    for r in _pick:
        f.write(json.dumps(_load(r)) + "\n")
n_traj = len(_pick)
_new = len({task_of(r) for r in _pick} - TRAINED_TASKS)
plog(f"session {STATE.get('sessions')}: {len(TRAINED)} trajectories / {len(TRAINED_TASKS)} problems trained before; "
     f"this slice {n_traj} trajectories (~{(_tok + _comp_tok) / 1e6:.1f}M tokens; {_new} new problems; the last {TAIL_N} = "
     f"competition trajectories, trained last; {len(_replay)} replayed from earlier sessions; "
     f"tail = least-visited {len(_comp)} of {len(_comp_all)} competition trajectories"
     f"{' (ALL at the ' + str(COMP_MAX_VISITS) + '-visit cap: repeating anyway so the session ends on competition data)' if COMP_OVER_CAP else ''}) for "
     f"{_train_s / 3600:.1f} h at {_tps:.0f} tok/s x {_over}; {len(_queue)} untrained left -> {TRAIN_DATA}")
del _rows, _queue, _first, _replay_pool


In [ ]:
# STEP 6 - train the 31B LoRA (4-bit model split over all 4 GPUs); training window stepped down on out-of-memory
stop_servers()
_models = [dp for dp, dns, fns in walk(INPUT, maxdepth=9) if "config.json" in fns]
def _find(*keys):
    # "-assistant" checkpoints are speculative-decoding drafters, never the model itself
    return next((dp for dp in _models if all(k in str(dp).lower() for k in keys) and "assistant" not in str(dp).lower()), None)
BASE31_TRAIN = _find("31b", "unquantized")
for _s in STUDENTS.values():
    _s["train_dir"], _s["serve_dir"] = _find(_s["key"], "unquantized"), _find(_s["slug"])
plog(f"models: 31B train {BASE31_TRAIN} | " + " | ".join(f"{k}: train {v['train_dir']}, serve {v['serve_dir']}" for k, v in STUDENTS.items()))
# training libraries: the attached graphcomp-train-wheels (peft 0.21.2, accelerate 1.15.0) into a private folder, so the
# serving / sandbox environment is unchanged; without it the image's own peft / accelerate are used
TRAIN_LIB = Path("/tmp/trainlib")
_wh = sorted(glob.glob(str(INPUT / "**/peft-*.whl"), recursive=True)) + sorted(glob.glob(str(INPUT / "**/accelerate-*.whl"), recursive=True))
if _wh and not TRAIN_LIB.exists():
    sh(f"{sys.executable} -m pip install --no-deps --no-index --target {TRAIN_LIB} {' '.join(_wh)}", log=OUT / "provenance/pip_train.log")
TRAIN_ENV = {"PYTHONPATH": str(TRAIN_LIB)} if TRAIN_LIB.exists() else {}
plog(f"training libraries: {'graphcomp-train-wheels ' + str([Path(w).name for w in _wh]) if TRAIN_ENV else 'image (peft 0.19.1, accelerate 1.13.0)'}")
DEPLOY_ARGS = (f"--system-prompt {DEPLOY_PROMPT} --deploy-budget " + shlex.quote(json.dumps({
    "time_minutes": DEPLOY_BUDGET["max_time_minutes"], "tool_calls": DEPLOY_BUDGET["max_tool_calls"],
    "turns": DEPLOY_BUDGET["max_turns"], "timeout_seconds": DEPLOY_BUDGET["timeout_seconds"]})))
RESERVE_H = (2 if EVAL_STUDENTS else 1) * EVAL_ARM_H + 0.4   # the 31B holdout arms (one server), the students if evaluated, the zip
def _budget(max_h, name):
    b = min(max_h, left_h() - RESERVE_H)
    if not TRAIN_DATA.exists() or n_traj == 0:
        plog(f"{name}: no training data"); return None
    if b < 0.25:
        plog(f"{name}: skipped, {left_h():.1f} h left (reserve {RESERVE_H:.1f} h): attach this output and run again"); return None
    return b
def _report(name, out_dir, rc, t0):
    ok = (out_dir / "adapter" / "adapter_config.json").exists()
    plog(f"{name}: exit {rc} after {(time.time() - t0) / 3600:.2f} h; adapter {'saved' if ok else 'MISSING'}")
    for l_ in read_jsonl(out_dir / "train_log.jsonl")[-2:]:
        print(l_)
    return ok
# --- 31B
OUT31 = OUT / "lora31"
ADAPTER31 = OUT31 / "adapter"
OK31 = (ADAPTER31 / "adapter_config.json").exists()
# resume: the previous session's adapter (copied in by the resume step) is the starting point; it stays the submission
# if this session trains nothing. The trainer stops at TRAIN_DEADLINE at a step boundary and always saves (also every 30
# min, atomically), so a session always leaves an adapter.
PREV = OUT / "lora31_prev"
if ADAPTER31.exists() and STATE.get("adapter_session") != STATE.get("sessions"):
    shutil.rmtree(PREV, ignore_errors=True); shutil.move(str(OUT31), str(PREV))
INIT = PREV / "adapter" if (PREV / "adapter" / "adapter_config.json").exists() else None
OK31, L = False, None
# windows that ran out of memory in an earlier session are skipped, and a session never starts above the window that last
# trained (session 1, Oct 18: 32k ran out of memory on GPU 3, which also holds the 262k-vocabulary output layer)
_oom = set(STATE.get("oom_windows", []))
_cap = STATE.get("lora31_window") or max(TRAIN_MAX_LENS)
_lens = [l for l in TRAIN_MAX_LENS if l <= _cap and l not in _oom] or [min(TRAIN_MAX_LENS)]
plog(f"lora31: training windows to try {_lens} (out of memory before: {sorted(_oom) or 'none'})")
for L in ([18432] if SMOKE else _lens):
    if BASE31_TRAIN is None or not n_traj:
        break
    left_s = TRAIN_DEADLINE - time.time()
    if left_s < (LOAD_H + 0.25) * 3600:
        plog(f"lora31: {left_s / 3600:.2f} h to the training deadline: not enough for the load + training"); break
    for init in ([INIT, None] if INIT else [None]):   # a resume that fails to load falls back to a fresh LoRA once
        shutil.rmtree(OUT31, ignore_errors=True); t0 = time.time()
        _log = OUT / f"provenance/train_lora31_s{STATE.get('sessions')}_len{L}{'_resume' if init else ''}.log"
        b = max(0.25, (TRAIN_DEADLINE - time.time()) / 3600 - LOAD_H)
        rc = sh(f"cd {GC}/lora && timeout -k 60 {int(TRAIN_DEADLINE - time.time() + 900)} {sys.executable} train_lora.py "
                f"--data {TRAIN_DATA} --base {BASE31_TRAIN} --out {OUT31} --device-map custom --last-gpu-layers {LAST_GPU_LAYERS} --max-len {L} --max-target {MAX_TARGET_FOR.get(L, 1024)} "
                f"--val-frac {VAL_FRAC} --no-eval-start --served-model google/{MODEL_SLUG} {DEPLOY_ARGS} --time-budget-h {b:.2f} "
                f"--epochs 1 --keep-order --stop-at {TRAIN_DEADLINE:.0f} --save-every-min 30 --seed {STATE.get('sessions', 1)}"
                + (f" --tail-n {TAIL_N} --lr-floor 0.3" if TAIL_N else "")
                + (f" --init-adapter {init}" if init else "") + (" --grad-accum 1" if SMOKE else ""), log=_log, env=TRAIN_ENV)
        kill_gpu_procs()
        OK31 = _report(f"lora31 (window {L}{', resumed' if init else ''})", OUT31, rc, t0)
        _tail = _log.read_text(errors="replace")[-30000:] if _log.exists() else ""
        if rc == 75 and OK31:           # OOM memory not given back: the trainer saved its progress; continue from it smaller
            _part = OUT / f"lora31_partial_len{L}"
            shutil.rmtree(_part, ignore_errors=True); shutil.move(str(OUT31), str(_part))
            INIT, OK31 = _part / "adapter", False
            save_state(oom_windows=sorted(set(STATE.get("oom_windows", [])) | {L}))
            plog(f"lora31: out-of-memory memory not released at window {L}: continuing from its saved adapter at the next window")
            break
        if OK31 or not init or re.search(r"out of memory|OutOfMemoryError", _tail, re.I):
            break
        plog("!!! lora31: resuming from the previous adapter failed; training a fresh LoRA instead (see " + str(_log) + ")")
    if OK31 or not re.search(r"out of memory|OutOfMemoryError", _tail, re.I):
        break
    save_state(oom_windows=sorted(set(STATE.get("oom_windows", [])) | {L}))
    plog(f"lora31: out of memory with a {L}-token window; trying the next one")
if OK31:
    _ids = json.loads((ADAPTER31 / "trained_ids.json").read_text()) if (ADAPTER31 / "trained_ids.json").exists() else []
    for _pp in sorted(OUT.glob("lora31_partial_len*/adapter/trained_ids.json")):   # attempts that exited 75 this session
        _ids = sorted(set(_ids) | set(json.loads(_pp.read_text())))
    _meta = json.loads((OUT31 / "train_meta.json").read_text()) if (OUT31 / "train_meta.json").exists() else {}
    _byid = {}
    for l_ in open(TRAIN_DATA):
        r_ = json.loads(l_); _byid[r_["source_traj_id"]] = task_of(r_)
    _cv = dict(COMP_VISITS)
    for k_ in set(COMP_KEYS) & set(_ids):
        _cv[k_] = _cv.get(k_, 0) + 1
    save_state(comp_visits=_cv, trained_ids=sorted(TRAINED | set(_ids)), trained_tasks=sorted(TRAINED_TASKS | {_byid[i] for i in _ids if i in _byid}),
               tok_per_s=_meta.get("tok_per_s") or STATE.get("tok_per_s"), lora31_window=L, adapter_session=STATE.get("sessions"),
               adapter_applies=None, session_log=STATE.get("session_log", []) + [{"session": STATE.get("sessions"), "trained": len(_ids),
               "stopped": _meta.get("stopped"), "tok_per_s": _meta.get("tok_per_s"), "steps": _meta.get("steps"), "resumed": bool(INIT)}])
    plog(f"lora31: this session trained {len(_ids)} trajectories at {_meta.get('tok_per_s')} tok/s (stopped: {_meta.get('stopped')}); "
         f"total {len(STATE['trained_ids'])} trajectories over {len(STATE['trained_tasks'])} problems")
elif INIT:                          # nothing new this session: the previous adapter remains the submission
    shutil.rmtree(OUT31, ignore_errors=True); shutil.copytree(PREV, OUT31); OK31 = True
    plog("lora31: no new training this session; the previous session's adapter is kept as the submission")
# --- students in parallel: each its own GPUs, its own torchrun rendezvous port, its own log
procs = {}
for name, s in STUDENTS.items():
    s["out"] = OUT / name.lower()
    s["ok"] = (s["out"] / "adapter" / "adapter_config.json").exists()
    if s["ok"] or s["train_dir"] is None:
        plog(f"{name}: {'adapter already in this output; reused' if s['ok'] else 'training checkpoint not attached; skipped'}")
        continue
    b = _budget(TRAIN_STUDENT_MAX_H, name)
    if not b:
        continue
    shutil.rmtree(s["out"], ignore_errors=True)
    cmd = (f"cd {GC}/lora && CUDA_VISIBLE_DEVICES={','.join(map(str, s['train_gpus']))} timeout -k 60 {int((b + 0.3) * 3600)} "
           f"{sys.executable} -m torch.distributed.run --nproc_per_node {len(s['train_gpus'])} --master_port {s['master_port']} "
           f"train_lora.py --data {TRAIN_DATA} --base {s['train_dir']} --out {s['out']} --max-target {s['max_target']} --max-len {s['max_len']} "
           f"--served-model google/{s['slug']} {DEPLOY_ARGS} --time-budget-h {b:.2f}")
    log_ = open(OUT / f"provenance/train_{name}.log", "w")
    procs[name] = (subprocess.Popen(cmd, shell=True, stdout=log_, stderr=subprocess.STDOUT, env={**os.environ, **TRAIN_ENV}), time.time())
    plog(f"{name}: training started on GPUs {s['train_gpus']}: {cmd[:300]}")
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(120)
    plog("    students: " + " | ".join(f"{n}: " + (json.dumps(read_jsonl(STUDENTS[n]['out'] / 'train_log.jsonl')[-1:])[:160] if p.poll() is None else f"exit {p.returncode}")
                                     for n, (p, _) in procs.items()))
kill_gpu_procs()
for name, (p, t0) in procs.items():
    STUDENTS[name]["ok"] = _report(name, STUDENTS[name]["out"], p.returncode, t0)
    if not STUDENTS[name]["ok"]:
        print(open(OUT / f"provenance/train_{name}.log").read()[-3000:])
save_state(lora31=OK31, **{n.lower(): s["ok"] for n, s in STUDENTS.items()})


In [ ]:
# STEP 7 - adapter check: serve the LoRA as the scorer does; it must change a greedy answer (a no-op adapter is not submitted)
def make_sub(dest, model_slug, adapter_dir=None, budget=None, seed=1):
    """The deployed agent; budget=None -> the in-notebook BUDGET (concurrency 4), else the deployment budget, no seed."""
    d = write_submission(dest, TOOLS, seed)
    y = (d / "agent.yaml").read_text().replace(f"model: {MODEL_SLUG}\n", f"model: {model_slug}\n")
    if adapter_dir is not None:
        shutil.copytree(adapter_dir, d / "adapters" / "agent_lora", dirs_exist_ok=True)
        y = y.replace(f"model: {model_slug}\n", f"model: {model_slug}\nadapter: agent_lora\n")
    (d / "agent.yaml").write_text(y)
    if budget is not None:
        (d / "configs/sampling.yaml").write_text(re.sub(r"seed: .*\n", "", (d / "configs/sampling.yaml").read_text()))
        (d / "eval_config.yaml").write_text("evaluation:\n" + "".join(f"  {k}: {v}\n" for k, v in budget.items()))
    return d

def adapter_changes_output(port, name="agent_lora"):
    """One greedy request to the base model and to the adapter: identical text and logprobs -> the adapter is a no-op."""
    import urllib.request
    def post(path, body=None):
        req = urllib.request.Request(f"http://127.0.0.1:{port}" + path, data=json.dumps(body).encode() if body else None,
                                     headers={"Content-Type": "application/json"})
        return json.loads(urllib.request.urlopen(req, timeout=300).read())
    ids = [m["id"] for m in post("/v1/models")["data"]]
    base = next(i for i in ids if i != name)
    outs = []
    for mdl in (base, name):
        r_ = post("/v1/chat/completions", {"model": mdl, "temperature": 0, "max_tokens": 48, "logprobs": True, "top_logprobs": 1,
             "chat_template_kwargs": {"enable_thinking": False},
             "messages": [{"role": "user", "content": "Fix this bug and reply with the function only: def add(a, b): return a - b"}]})
        ch = r_["choices"][0]
        outs.append((ch["message"].get("content"), [round(t["logprob"], 4) for t in (ch.get("logprobs") or {}).get("content") or []]))
    plog(f"adapter probe :{port}: base {outs[0][0]!r:.80} | adapter {outs[1][0]!r:.80} | identical: {outs[0] == outs[1]}")
    return outs[0] != outs[1]

if not OK31:   # still submittable: the base model, flagged (attach this output and run again to train)
    plog("!!! NO ADAPTER this session (see provenance/train_lora31_*.log): the submission is the BASE 31B, no LoRA")
    save_state(adapter_applies="no adapter")
elif SMOKE:
    save_state(adapter_applies="not checked (smoke)")
elif left_h() < 0.6:                 # the submission must still be written inside the session
    plog(f"adapter check skipped: {left_h():.2f} h left"); save_state(adapter_applies="not checked (time)")
elif STATE.get("adapter_applies") is None:
    _chk = make_sub(OUT / "submissions/check_LORA31", MODEL_SLUG, ADAPTER31)
    stop_servers(); start_server(MODEL_DIR, TP, _chk)
    try:
        save_state(adapter_applies=bool(adapter_changes_output(8000)))
    finally:
        stop_servers()
if STATE["adapter_applies"] is False:
    plog("!!! WARNING: vLLM serves the adapter as a no-op (identical greedy output to the base model); the submission is still "
         "written (it behaves like the base model), check the adapter before relying on it")


In [ ]:
# STEP 8 - write the submission (the LoRA, deployment budget) and the summary
FINAL = Path("/kaggle/working/submission")
shutil.rmtree(FINAL, ignore_errors=True)
shutil.copytree(make_sub(OUT / "submissions/final_LORA31", MODEL_SLUG, ADAPTER31 if OK31 else None, budget=DEPLOY_BUDGET), FINAL)
_zip = Path(shutil.make_archive("/kaggle/working/submission", "zip", FINAL))   # agent.yaml at the zip root
_size = sum(f.stat().st_size for f in FINAL.rglob("*") if f.is_file())
assert _size < 3 * 2**30, f"submission is {_size / 2**30:.2f} GiB (limit 3 GiB)"
worst_h = (N_TASKS_ASSUMED * (DEPLOY_BUDGET["max_time_minutes"] + SETUP_MIN) + SERVE_START_MIN) / 60
lines = [f"# Notebook 21 summary ({now()})", "",
         f"- Training trajectories: {n_traj} (SFT mix {MIX}); 31B training window {STATE.get('lora31_window')}",
         f"- Train log (last): {json.dumps(read_jsonl(OUT31 / 'train_log.jsonl')[-1:])[:400]}",
         f"- Adapter changes the served output: {STATE.get('adapter_applies')}",
         f"- Sessions: {STATE.get('session_log')}; trained so far {len(STATE.get('trained_ids', []))} trajectories over "
         f"{len(STATE.get('trained_tasks', []))} problems; attach this output to the next version to continue",
         "", ("**SUBMISSION: LORA31**" if OK31 else "**SUBMISSION: BASE 31B (no adapter was trained; see the logs)**")
         + " (no holdout comparison: every task is training data)",
         f"- deployment budget {DEPLOY_BUDGET}; worst case if all {N_TASKS_ASSUMED} tasks hit the cap: {worst_h:.1f} h of 12"
         + ("" if worst_h * 60 + MARGIN_MIN <= 720 else " (does not fit: relies on most tasks finishing before the 8-minute cap)"),
         f"- {FINAL} ({_size / 1e6:.1f} MB) and {_zip}"]
(OUT / "SUMMARY.md").write_text("\n".join(lines) + "\n")
save_state(submission="LORA31")
print("\n".join(lines))
print((FINAL / "agent.yaml").read_text()); print((FINAL / "eval_config.yaml").read_text())


In [ ]:
# STEP 9 - zip the run folder (Output tab)
stop_servers()
zp = Path("/kaggle/working/graphcomp_nb21.zip")
if zp.exists():
    zp.unlink()
shutil.make_archive(str(zp.with_suffix("")), "zip", OUT)
plog(f"zip: {zp} {zp.stat().st_size / 1e6:.1f} MB")
